# Week 5 Final global profiles, regional evidence and held-out transfer

**Objective:** complete Objective 1 by documenting the development evidence, the final B4/E8 freeze, fixed-model geographic validation and the reusable respondent assignments.

**Output:** regional comparisons; unchanged ARI 0.80 decision; a sealed, reload-tested model package; held-out inclusion and transfer diagnostics; original-unit interpretations and country limitations; bootstrap intervals; reusable figures/maps; an ID-linked handoff for later objectives.

**Current status:** global B4 and E8 are frozen as complementary descriptions. The 29 held-out countries were opened only after sealing and verifying that package. They were assigned without refitting or retuning. Objective 1 is concluded with **broad aggregate descriptive transfer and material country-specific limitations**, not a universal natural typology or eight uniformly secure E profiles.

This is the canonical Week 5 notebook. At the user’s request, superseded Week 5 notebooks v1–v5 are removed from the active workspace using recoverable deletion (Trash); model caches, result tables and the sealed packages are retained. **Sections 1–17 are the historical development-stage record**, including the earlier unopened-holdout boundary and decisions awaiting approval at that time. Sections 18–24 record the subsequent explicit approval, freeze, validation and completion. Historical “not frozen/unopened” statements describe those earlier stages; they are not the current status.

Re-execution verifies and replays completed evidence. It does not launch clustering fits, revise centres, regenerate the country split, or overwrite the frozen model package. Original model objects, complete-case rules, 21-coordinate geometry and cluster IDs are retained. All 146,910 prepared respondent IDs remain in the handoff; the 9,450 excluded respondents remain unassigned.

**Completed remaining descriptive work:** sections 25–29 report within-profile heterogeneity, information lost by two burden totals, country distances, geographical comparisons, exploratory country clustering, conditional sampling sensitivity, maps and a final completion checklist. These post-validation extensions do not change the frozen models or strengthen the held-out acceptance claim. A pre-extension snapshot is preserved under earlier iterations; there is still only one active Week 5 notebook.

## 1 Fixed candidates and execution controls

**Objective:** Declare the unchanged B/E candidates, Moodle-compatible sampling rules, and execution boundary.

**Output:** Specifications, seeds, and run switches; no held-out data access.

B uses 0/1/2 worry coding, balanced worry/harm allocation, and four clusters. E uses 0/1/3 worry coding, worry allocation 0.75, and eight clusters. Both use complete responses, separate personal/known-person harm, six within-person centred worry values, and the appended worry mean. There is no imputation, z-standardization, or PCA.

Follow the Moodle rule: 50 country samples of floor(0.8×N) countries, without replacement within each sample, for regions with at least five countries. Repeated subsets across samples are allowed and reported. Use leave-one-country-out for two–four countries. A single-country region has no country test. Country sampling is uniform within the existing region, not the global crossed-stratum quota design. LOO summaries are not a 50-fit country-stability claim. Respondent subsamples retain approximately 80% within every regional country. All of these limitations are reported before results are interpreted.

**Development replay rule:** baseline fit switches are off. Saved evidence is loaded and checked, not regenerated. The explicit additional-fit resume switch in section 12 is also off by default.

In [ ]:
# Objective: Declare the unchanged B/E candidates, Moodle-compatible sampling rules, and execution boundary.
from pathlib import Path
from dataclasses import dataclass, asdict
import hashlib
import itertools
import json
import math
import platform
import traceback
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, pairwise_distances_argmin
from scipy.optimize import linear_sum_assignment
from threadpoolctl import threadpool_limits

DOMAINS = ['food', 'water', 'crime', 'severe_weather', 'traffic', 'mental_health']
WORRY = ['worry_' + d for d in DOMAINS]
PERSONAL = ['harm_' + d + '_personal' for d in DOMAINS + ['work']]
KNOWN = ['harm_' + d + '_known_person' for d in DOMAINS + ['work']]
FEATURES = WORRY + PERSONAL + KNOWN
TRANSFORM_COLUMNS = ['centred_' + c for c in WORRY] + ['overall_worry'] + PERSONAL + KNOWN

@dataclass(frozen=True)
class Specification:
    name: str
    encoding: str
    top: float
    balance: str
    alpha: float
    global_k: int

SPECIFICATIONS = {
    'B': Specification('B', '012', 2., 'balanced', .5, 4),
    'E': Specification('E', '013', 3., 'worry_emphasized', .75, 8),
}
REGIONAL_DESIGN = {
    'version': 3, 'seed': 20261011, 'k_values': list(range(2, 11)),
    'n_init': 20, 'max_iter': 300, 'repetitions': 50, 'fraction': .8,
    'respondent_sampling': 'without replacement within each regional country',
    'country_sampling': '50 uniform samples without replacement of floor(0.8*N) countries; repeats across samples allowed and reported',
    'small_region_policy': 'leave-one-country-out for 2 to 4 countries',
    'one_country_policy': 'country stability unavailable',
    'pairwise_ari_threshold': .8,
    'global_structural_benchmark_only': {'k_min': 3, 'k_max': 10, 'coverage_min': .9,
                                        'largest_share_max': .4, 'dominance_max_exclusive': 3},
    'assignment': 'nearest centre in unchanged 21-feature geometry',
    'specifications': {name: asdict(spec) for name, spec in SPECIFICATIONS.items()},
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def digest_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':')).encode()).hexdigest()

def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False))
    temp.replace(path)

def atomic_joblib(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    joblib.dump(value, temp)
    temp.replace(path)

def table_write(table, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    table.to_csv(temp, index=False)
    temp.replace(path)

In [ ]:
# Objective: Declare the unchanged B/E candidates, Moodle-compatible sampling rules, and execution boundary.
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
get_ipython().run_line_magic('matplotlib', 'inline')

ROOT = Path.cwd()
if not (ROOT / 'course_artifacts').exists():
    ROOT = ROOT.parent
assert (ROOT / 'course_artifacts').is_dir(), 'Run from the course root or weekly_notebooks directory.'

REGIONAL_DESIGN_APPROVED = True
RUN_REGIONAL_REFERENCES = False
RUN_REGIONAL_STABILITY = False
IMPLEMENTATION_SHA256 = '15c2c6acc9c426e08ada198aa69b87292071295fd4367f43328f3f257fafe958'

print('Regional design approved:', REGIONAL_DESIGN_APPROVED)
print('Reference fits enabled:', RUN_REGIONAL_REFERENCES)
print('Stability enabled:', RUN_REGIONAL_STABILITY)
print('Historical development-stage replay. The completed freeze and held-out validation are reported in sections 18–24.')
print('Read-only baseline replay; saved regional tests are loaded even though refitting switches are off.')

## 2 Reproduce the Week 4 transformation

**Objective:** Implement the exact Week 4 feature geometry and correct equal-country weights.

**Output:** Transformation and weighting helpers with validation checks; no model fitting.

Let \(q_{ij}\) be respondent i's worry values after top-category recoding and division by 2 or 3. Let \(\bar q_i\) be their mean. Each centred worry value and the appended mean receives multiplier \(\sqrt{\alpha/6}\); each of the 14 harm indicators receives \(\sqrt{(1-\alpha)/14}\).

These are exactly the Week 4 rules, including the deliberately reduced overall-worry contribution. Regional fitting does not learn a new scale. Survey weights are normalized within countries, then every retained regional country receives equal total weight. In every subsample the weights are recalculated on the training respondents; comparison/profile weights describe the full regional cohort.

In [ ]:
# Objective: Implement the exact Week 4 feature geometry and correct equal-country weights.
def transform(frame, spec):
    values = frame[FEATURES].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('This specification requires complete observed worry and harm responses.')
    if not frame[WORRY].isin([0, 1, 2]).all().all():
        raise ValueError('Unexpected original worry code.')
    if not frame[PERSONAL + KNOWN].isin([0, 1]).all().all():
        raise ValueError('Unexpected binary harm code.')
    worry = np.where(values[:, :6] == 2, spec.top, values[:, :6]) / spec.top
    mean = worry.mean(axis=1, keepdims=True)
    multiplier = np.sqrt(spec.alpha / 6)
    return np.column_stack([(worry - mean) * multiplier, mean * multiplier,
                            values[:, 6:] * np.sqrt((1 - spec.alpha) / 14)])

def country_weights(frame):
    if not len(frame) or not np.isfinite(frame.WGT).all() or not frame.WGT.gt(0).all():
        raise ValueError('Survey weights must be finite and positive in a nonempty cohort.')
    result = frame.WGT / frame.groupby('COUNTRY_ISO3').WGT.transform('sum')
    result = result / frame.COUNTRY_ISO3.nunique()
    totals = result.groupby(frame.COUNTRY_ISO3).sum()
    assert np.allclose(totals, 1 / len(totals)) and np.isclose(result.sum(), 1)
    return result.to_numpy()

def prepare_development(root):
    root = Path(root)
    artifacts = root / 'course_artifacts'
    input_path = artifacts / 'week04_search_v4/stage1_eligible_development.parquet'
    split_path = artifacts / 'week03_country_split.csv'
    split = pd.read_csv(split_path)
    assert split.COUNTRY_ISO3.is_unique
    dev_countries = set(split.loc[split.country_split.eq('train'), 'COUNTRY_ISO3'])
    hold_countries = set(split.loc[split.country_split.eq('heldout_country'), 'COUNTRY_ISO3'])
    assert len(dev_countries) == 113 and len(hold_countries) == 29
    assert not dev_countries.intersection(hold_countries)
    raw = pd.read_parquet(input_path)  # Development-only input, never the held-out response file.
    assert raw.country_split.eq('train').all() and raw.WPID_RANDOM.is_unique
    assert set(raw.COUNTRY_ISO3) == dev_countries
    eligible = raw[WORRY].notna().sum(axis=1).ge(4) & raw[PERSONAL].notna().sum(axis=1).ge(4)
    complete = eligible & raw[FEATURES].notna().all(axis=1)
    data = raw.loc[complete].copy().reset_index(drop=True)
    audit = raw.assign(previous_eligible=eligible, complete=complete).groupby('COUNTRY_ISO3').agg(
        available_input=('WPID_RANDOM', 'size'), previous_eligible=('previous_eligible', 'sum'),
        complete=('complete', 'sum'))
    audit['retained_fraction'] = audit.complete / audit.available_input
    audit = split.loc[split.country_split.eq('train')].merge(audit, on='COUNTRY_ISO3', validate='one_to_one')
    audit['retained_of_prepared'] = audit.complete / audit.respondents
    region_rows = audit.groupby('GlobalRegion').agg(countries=('COUNTRY_ISO3', 'size'),
        prepared_respondents=('respondents', 'sum'), input_respondents=('available_input', 'sum'),
        complete_respondents=('complete', 'sum')).reset_index()
    dictionary_path = root / '23_wrp/WRP_2023_data_dictionary.csv'
    dictionary = pd.read_csv(dictionary_path)
    labels_text = dictionary.loc[dictionary.variable.eq('GlobalRegion'), 'value_labels'].iloc[0]
    region_labels = {int(pair.split(' = ', 1)[0]): pair.split(' = ', 1)[1]
                     for pair in labels_text.split(' | ')}
    region_rows['region_name'] = region_rows.GlobalRegion.map(region_labels)
    region_rows['country_diagnostic'] = np.where(region_rows.countries.eq(1),
                                                 'unavailable: one country', 'pending sampling-plan audit')
    region_map = split.set_index('COUNTRY_ISO3').GlobalRegion
    data['GlobalRegion'] = data.COUNTRY_ISO3.map(region_map)
    data['CountryIncomeLevel'] = data.COUNTRY_ISO3.map(split.set_index('COUNTRY_ISO3').CountryIncomeLevel)
    assert len(data) == 109488 and data.COUNTRY_ISO3.nunique() == 113
    for spec in SPECIFICATIONS.values():
        assert transform(data, spec).shape == (len(data), 21)
    input_hash = file_sha256(input_path)
    provenance = {'development_input': str(input_path.relative_to(root)), 'input_sha256': input_hash,
                  'country_split_sha256': file_sha256(split_path),
                  'region_dictionary_sha256': file_sha256(dictionary_path),
                  'python': platform.python_version(), 'sklearn': sklearn.__version__,
                  'numpy': np.__version__, 'pandas': pd.__version__, 'joblib': joblib.__version__}
    references = {}
    reference_dir = artifacts / 'week04_experiment1_complete_overall_worry_v3'
    design_path = reference_dir / 'design.json'
    old_design = json.loads(design_path.read_text())
    # Exact legacy fingerprint rule; preserves the existing cache contract.
    digest = hashlib.sha256()
    with input_path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    digest.update(json.dumps(old_design, sort_keys=True).encode())
    expected_fingerprint = digest.hexdigest()
    for name, spec in SPECIFICATIONS.items():
        path = reference_dir / f'reference_{spec.encoding}_{spec.balance}_k{spec.global_k}.joblib'
        cached = joblib.load(path)  # Trusted local course cache only.
        if cached.get('fingerprint') != expected_fingerprint:
            raise ValueError(f'{name}: Week 4 cache fingerprint does not match data and design.')
        labels = cached['labels']
        if len(labels) != len(data):
            raise ValueError('Reference row order/length changed.')
        matrix = transform(data, spec)
        with threadpool_limits(limits=1):
            predicted = pairwise_distances_argmin(matrix, cached['centres'])
        if not np.array_equal(predicted, labels):
            raise ValueError(f'{name}: reloaded centres do not reproduce the stored development labels.')
        references[name] = {'labels': labels, 'centres': cached['centres'], 'spec': asdict(spec),
                            'source_sha256': file_sha256(path), 'fingerprint': cached['fingerprint']}
    provenance['reference_sources'] = {name: value['source_sha256'] for name, value in references.items()}
    provenance['reference_design_sha256'] = file_sha256(design_path)
    return data, audit, region_rows, references, provenance

def previous_stability(root):
    table = pd.read_csv(Path(root) / 'course_artifacts/week04_experiment1_complete_overall_worry_v3/stability_all_encodings.csv',
                        dtype={'encoding': str})
    rows = []
    for name, spec in SPECIFICATIONS.items():
        selected = table.loc[table.encoding.str.zfill(3).eq(spec.encoding)
                             & table.balance.eq(spec.balance) & table.k.eq(spec.global_k)].copy()
        assert set(selected.experiment) == {'countries', 'respondents'}
        selected.insert(0, 'candidate', name)
        rows.append(selected)
    return pd.concat(rows, ignore_index=True)

## 3 Verify development inputs and saved global assignments

**Objective:** Verify development-only inputs and exact replay of the saved global models.

**Output:** Country/region retention audits, fingerprints, and exact B/E label-reload verification.

Read only the development-only Week 4 prepared input and country metadata. Apply the same eligibility and complete-response rules in the same row order. Verify the legacy Week 4 data/design fingerprint, then reproduce every stored B/E label from its saved centres. A mismatch stops the notebook; it is not repaired by silently refitting.

The audit distinguishes the 116,820 prepared development respondents, the 116,808-row Week 4 input after entirely unobserved profiles were removed, and the 109,488 complete-case cohort. Region labels come from the supplied data dictionary; no regions are invented from response patterns.

In [ ]:
# Objective: Verify development-only inputs and global label replay, then read the preserved baseline.
data, country_audit, regions, global_references, provenance = prepare_development(ROOT)
CONTEXT = digest_json({'provenance': provenance, 'design': REGIONAL_DESIGN,
                       'implementation_sha256': IMPLEMENTATION_SHA256})
OUTPUT = ROOT / 'course_artifacts/week05_regional_validation_v3' / CONTEXT[:16]
baseline_manifest = json.loads((OUTPUT/'run_manifest.json').read_text())
assert baseline_manifest['context'] == CONTEXT
assert baseline_manifest['provenance'] == provenance
assert baseline_manifest['regional_design'] == REGIONAL_DESIGN
assert baseline_manifest['heldout_responses_used'] is False
print('Verified:', len(data), 'complete respondents;', data.COUNTRY_ISO3.nunique(), 'development countries.')
print('Global B/E labels reproduce exactly. No global or regional refit; no baseline table overwrite.')
display(regions)
display(previous_stability(ROOT)[['candidate','experiment','median_pairwise','p10_pairwise']])

## 4 Comparison measures and original-response distributions

**Objective:** Define the membership and original-response measures used for regional comparisons.

**Output:** ARI/overlap, structural, and categorical-profile functions; definitions alone do not generate results.

Unweighted ARI measures membership agreement on the same respondents, independent of cluster IDs. Weighted overlap tables allow many local clusters to correspond to one global cluster and vice versa. Local-to-global purity sums the largest global overlap for each local cluster; reverse purity sums the largest local overlap for each global cluster. A one-to-one matched share is also reported, but is restrictive when k differs.

Original-unit tables retain every worry category and both personal and known-person harm indicators, with cluster-versus-outside contrasts. The global structural screen (k=3–10, coverage≥90%, largest share≤40%, D<3) is reported as a **comparison benchmark**, not a mandatory rule for regional structure. A local two-group solution can be informative without passing that global benchmark. Neither ARI nor overlap is a significance test.

In [ ]:
# Objective: Define the membership and original-response measures used for regional comparisons.
def structure(labels, weights):
    labels = np.asarray(labels)
    groups = np.unique(labels[labels >= 0])
    coverage = float(weights[labels >= 0].sum())
    shares = [float(weights[labels == group].sum() / coverage) for group in groups] if coverage else []
    largest = max(shares) if shares else np.nan
    dominance = len(groups) * largest
    return {'observed_k': len(groups), 'coverage': coverage, 'largest_share': largest,
            'dominance': dominance,
            'global_benchmark_pass': bool(3 <= len(groups) <= 10 and coverage >= .9
                                          and largest <= .4 and dominance < 3)}

def compare_partitions(local, global_labels, weights):
    local_groups, local_index = np.unique(local, return_inverse=True)
    global_groups, global_index = np.unique(global_labels, return_inverse=True)
    overlap = np.zeros((len(local_groups), len(global_groups)))
    np.add.at(overlap, (local_index, global_index), weights)
    row_sum = overlap.sum(axis=1, keepdims=True)
    col_sum = overlap.sum(axis=0, keepdims=True)
    local_purity = overlap.max(axis=1).sum() / overlap.sum()
    global_purity = overlap.max(axis=0).sum() / overlap.sum()
    matched_rows, matched_cols = linear_sum_assignment(-overlap)
    records = []
    for i, lg in enumerate(local_groups):
        for j, gg in enumerate(global_groups):
            records.append({'local_cluster': int(lg), 'global_cluster': int(gg),
                'joint_share': overlap[i, j], 'global_given_local': overlap[i, j] / row_sum[i, 0],
                'local_given_global': overlap[i, j] / col_sum[0, j]})
    return {'ari_unweighted': adjusted_rand_score(local, global_labels),
            'local_to_global_purity': local_purity, 'global_to_local_purity': global_purity,
            'one_to_one_matched_share': overlap[matched_rows, matched_cols].sum() / overlap.sum()}, pd.DataFrame(records)

def original_profiles(frame, labels, weights):
    rows = []
    for group in np.unique(labels):
        inside = labels == group
        for feature in FEATURES:
            for category in ([0, 1, 2] if feature in WORRY else [0, 1]):
                response = frame[feature].to_numpy() == category
                proportion = weights[inside & response].sum() / weights[inside].sum()
                outside = ~inside
                outside_proportion = (weights[outside & response].sum() / weights[outside].sum()
                                      if outside.any() else np.nan)
                rows.append({'cluster': int(group), 'cluster_share': weights[inside].sum(),
                    'feature': feature, 'category': category, 'proportion': proportion,
                    'outside_proportion': outside_proportion,
                    'difference_vs_outside': proportion - outside_proportion})
    return pd.DataFrame(rows)

## 4a Geographical distributions using fixed global profiles

**Objective:** Describe fixed-global B/E profile prevalence at global, regional, and country levels.

**Output:** Counts, respondent shares, weighted shares, and original-response distributions in tables and saved files.

Before regional refitting, retain the existing global B/E assignments. Report counts and percentages globally, within all 15 recorded regions, and within each of the 113 development countries. Survey weights are normalized within countries, and each retained country has equal total weight in its region. Unweighted respondent shares are also saved; they describe a different estimand. These are complete-case development summaries, not estimates of worldwide or regional population-proportional prevalence.

Holding centres fixed makes the profile IDs comparable. It does not imply that every region's members have the same original responses or fit the centre equally well. Save all original worry categories and personal/known-person harm proportions within each region/profile, with differences from the global profile and from outside the profile within that region. No model is refitted, and this is not an external validation result because these regions contributed to model development.

In [ ]:
# Objective: Describe fixed-global B/E profile prevalence at global, regional, and country levels.
WORKING_PROFILE_LABELS = {
    'B': {0: 'Broad harm exposure, especially known-person harm',
          1: 'Crime, severe-weather and traffic worry with lower harm',
          2: 'Food and water worry with substantial crime worry',
          3: 'Lower overall worry and harm'},
    'E': {0: 'Traffic-focused worry', 1: 'Lower worry intensity',
          2: 'Mental-health worry with other non-food/water worries',
          3: 'Crime and traffic worry', 4: 'Crime-focused worry',
          5: 'Severe-weather-focused worry', 6: 'Food and water worry',
          7: 'Broad high worry'},
}

def fixed_global_geographical_results(data, references, regions, output_dir):
    """Describe the saved global partitions; never fit a model or open held-out data."""
    prevalence_rows, profile_tables, semantic_rows = [], [], []
    region_names = regions.set_index('GlobalRegion').region_name.to_dict()
    for name, spec in SPECIFICATIONS.items():
        full_labels = references[name]['labels']
        weights = country_weights(data)
        global_profiles = original_profiles(data, full_labels, weights)
        global_profiles['candidate'] = name
        global_lookup = global_profiles.set_index(['cluster', 'feature', 'category']).proportion
        scopes = [('global', 'global_development', 'Global development', np.ones(len(data), dtype=bool))]
        scopes += [('region', str(int(region)), region_names[region], data.GlobalRegion.eq(region).to_numpy())
                   for region in sorted(data.GlobalRegion.unique())]
        scopes += [('country', country, country, data.COUNTRY_ISO3.eq(country).to_numpy())
                   for country in sorted(data.COUNTRY_ISO3.unique())]
        for scope, key, label, mask in scopes:
            frame = data.loc[mask]
            assigned = full_labels[mask]
            local_weights = country_weights(frame)
            country_count = frame.COUNTRY_ISO3.nunique()
            for cluster in range(spec.global_k):
                inside = assigned == cluster
                prevalence_rows.append({'candidate': name, 'scope': scope, 'area_id': key, 'area_name': label,
                    'cluster': cluster, 'working_description': WORKING_PROFILE_LABELS[name][cluster],
                    'respondents': int(inside.sum()), 'area_respondents': len(frame),
                    'area_countries': country_count, 'respondent_share': float(inside.mean()),
                    'equal_country_weighted_share': float(local_weights[inside].sum())})
            # Full original-unit distributions globally and regionally. These permit examination
            # of profile meaning rather than presuming fixed labels imply identical responses.
            if scope != 'country':
                profiles = original_profiles(frame, assigned, local_weights)
                profiles['candidate'], profiles['scope'] = name, scope
                profiles['area_id'], profiles['area_name'] = key, label
                profiles['global_proportion'] = [global_lookup.loc[(r.cluster, r.feature, r.category)]
                                                  for r in profiles.itertuples()]
                profiles['difference_from_global_pp'] = 100 * (profiles.proportion - profiles.global_proportion)
                profile_tables.append(profiles)
                for cluster in sorted(np.unique(assigned)):
                    group = profiles.loc[profiles.cluster.eq(cluster)]
                    extreme = group.loc[group.difference_from_global_pp.abs().idxmax()]
                    semantic_rows.append({'candidate': name, 'scope': scope, 'area_id': key, 'area_name': label,
                        'cluster': int(cluster), 'respondents': int((assigned == cluster).sum()),
                        'largest_absolute_category_difference_pp': abs(extreme.difference_from_global_pp),
                        'feature': extreme.feature, 'category': int(extreme.category),
                        'regional_proportion': float(extreme.proportion),
                        'global_proportion': float(extreme.global_proportion),
                        'signed_difference_pp': float(extreme.difference_from_global_pp)})
    prevalence = pd.DataFrame(prevalence_rows)
    profiles = pd.concat(profile_tables, ignore_index=True)
    semantics = pd.DataFrame(semantic_rows)
    for (_, _, _), group in prevalence.groupby(['candidate', 'scope', 'area_id']):
        assert np.isclose(group.equal_country_weighted_share.sum(), 1)
        assert np.isclose(group.respondent_share.sum(), 1)
        assert group.respondents.sum() == group.area_respondents.iloc[0]
    # Global shares are the country-count-weighted mixture of regional shares.
    for name in SPECIFICATIONS:
        global_rows = prevalence.loc[prevalence.candidate.eq(name) & prevalence.scope.eq('global')]
        local_rows = prevalence.loc[prevalence.candidate.eq(name) & prevalence.scope.eq('region')]
        for cluster in global_rows.cluster:
            local = local_rows.loc[local_rows.cluster.eq(cluster)]
            reconstructed = np.average(local.equal_country_weighted_share, weights=local.area_countries)
            expected = global_rows.loc[global_rows.cluster.eq(cluster), 'equal_country_weighted_share'].iloc[0]
            assert np.isclose(reconstructed, expected, atol=1e-12)
    table_write(prevalence, Path(output_dir) / 'fixed_global_profile_prevalence.csv')
    table_write(profiles, Path(output_dir) / 'fixed_global_original_profiles_by_region.csv')
    table_write(semantics, Path(output_dir) / 'fixed_global_regional_semantic_differences.csv')
    return prevalence, profiles, semantics

In [ ]:
# Objective: Describe fixed-global B/E profile prevalence at global, regional, and country levels.
fixed_prevalence = pd.read_csv(OUTPUT/'fixed_global_profile_prevalence.csv')
fixed_original_profiles = pd.read_csv(OUTPUT/'fixed_global_original_profiles_by_region.csv')
fixed_semantic_differences = pd.read_csv(OUTPUT/'fixed_global_regional_semantic_differences.csv')
print('Loaded preserved global, regional, and country prevalence, with complete regional original-response distributions.')
display(fixed_prevalence.loc[fixed_prevalence.scope.eq('global'),
    ['candidate','cluster','working_description','respondents','equal_country_weighted_share']])
for name in SPECIFICATIONS:
    selected = fixed_prevalence.loc[fixed_prevalence.candidate.eq(name)
                                   & fixed_prevalence.scope.isin(['global','region'])].copy()
    selected['weighted_percentage'] = 100 * selected.equal_country_weighted_share
    pivot = selected.pivot(index='area_name', columns='cluster', values='weighted_percentage')
    order = ['Global development'] + regions.sort_values('GlobalRegion').region_name.tolist()
    display(Markdown('### '+name+' fixed-global cluster shares by geographical area (%)'))
    display(pivot.reindex(order).round(1))
print('Checks passed: every area sums to 100%; regional country-weighted averages reproduce global shares.')

## 4b Interpretation of the fixed-global regional distributions

**Objective:** Explain substantive geographical contrasts without causal or transfer claims.

**Output:** Interpretation tied to saved cluster IDs, plus semantic, sample-selection, and small-region cautions.

**B describes both worry and harm.** Globally, C0 comprises 18.53% and has broad exposure especially through personally known people; C1 (26.02%) emphasizes crime, weather and traffic worry with less harm; C2 (21.68%) emphasizes food/water worry with substantial crime worry; C3 (33.77%) has lower worry and harm. These are working descriptions tied to the saved cluster IDs, not newly frozen names.

The lower-worry/harm C3 share is 53.4% in Northern/Western Europe and 51.6% in Central Asia, versus 17.7% in Central/Western Africa and 13.1% in Southern Africa. The broad-harm C0 share is about 29.6–29.8% in Central/Western and Eastern Africa, compared with 7.6% in Central Asia and 12.0% in Northern/Western Europe. These differences describe the retained respondents under equal-country weighting; they do not establish causes, universal regional traits, or country-level homogeneity.

**E gives a finer worry description.** Globally, its lower-worry C1 comprises 36.11%. Its share is 59.9% in Northern/Western Europe, 52.3% in Central Asia and 52.2% in Eastern Europe, versus 18.2% in Central/Western Africa and 11.1% in Southern Africa. Broad high-worry C7 is 20.0% in Central/Western Africa and 18.7% in Eastern Africa, compared with 2.5% in Northern/Western Europe. B and E therefore show a similar broad geographical worry gradient, while B additionally separates substantial known-person harm. Their similarly worded low-worry groups are not identical memberships and must not be treated as the same cluster.

**Thematic differences matter too.** Under E, weather-focused C5 is 14.2% in Eastern Asia and 12.5% in Southern Europe; food/water C6 is 14.8% in Northern Africa but 3.5% in Southeastern Asia. These identify contrasts to investigate in the full original-response tables, not proof that every country in those regions shares one pattern.

**E requires a specific caution.** Its traffic-focused C0, crime-focused C4 and weather-focused C5 each have 100% very-worried responses for their defining item in the saved global assignments; C3 has 100% very worried about both crime and traffic. Regional frequencies can therefore partly reflect these response-category combinations. Stable groups and Ward agreement do not by themselves prove discrete natural types or establish added value beyond burden/category summaries.

**Semantic compatibility remains a separate empirical check.** The saved regional original-response tables and descriptive maximum category differences show how members of each fixed profile differ by region. There is no automatic semantic pass/fail threshold. Inspect defining contrasts, harm, and overall intensity together; a stable defining category can coexist with differences in other features. Distance/margin diagnostics will provide an additional representation-quality check. The example regions highlighted above are descriptive contrasts, not a preselected significance test or a complete explanation.

**Coverage and uncertainty limit interpretation.** Northern America has only the United States in development; Australia/New Zealand has only New Zealand. These are single-country summaries, not replicated regional evidence. Complete-case retention varies by country and may select respondents systematically. No confidence intervals or statistical-significance claims are provided by these point estimates. Region differences may conceal stronger differences between countries, so country prevalence is saved alongside the regional tables. Maps are planned to show these fixed-global proportions and the coverage limits, not to replace the tables.

## 4c Fixed-global representation quality

**Objective:** Assess how well fixed centres represent respondents in each area.

**Output:** Distances, margins, provisional development thresholds, exceedances, and original-response differences.

Point 1 is not complete with prevalence alone. Calculate nearest-centre distances and the normalized nearest/second-nearest margin for every development respondent, without changing assignments. Save profile-specific weighted development 95th-percentile distances and report exceedance overall and by region/country. The margin is a geometric diagnostic, not a probability; distances must be interpreted within B or E, not as comparable scales across models. Thresholds are descriptive, provisional development references, not held-out pass/fail criteria or a final model freeze. Original-response semantic differences remain available alongside these diagnostics.

In [ ]:
# Objective: Assess how well fixed centres represent respondents in each area.
def weighted_quantile(values, weights, probability):
    order = np.argsort(values, kind='stable')
    cumulative = np.cumsum(weights[order]) / np.sum(weights)
    return float(np.asarray(values)[order][min(np.searchsorted(cumulative, probability), len(order)-1)])

def fixed_assignment_quality(data, references, regions, output_dir):
    from scipy.spatial.distance import cdist
    regional_names = regions.set_index('GlobalRegion').region_name.to_dict()
    summaries, thresholds, assignment_tables = [], [], []
    for name, spec in SPECIFICATIONS.items():
        weights = country_weights(data)
        with threadpool_limits(limits=1):
            distances = cdist(transform(data, spec), references[name]['centres'], metric='euclidean')
        ordered = np.sort(distances, axis=1)
        nearest, second = ordered[:,0], ordered[:,1]
        margin = np.divide(second-nearest, second, out=np.zeros_like(second), where=second>0)
        labels = references[name]['labels']
        assert np.array_equal(distances.argmin(axis=1), labels)
        cutoff = {}
        for cluster in range(spec.global_k):
            inside = labels == cluster
            cutoff[cluster] = weighted_quantile(nearest[inside], weights[inside], .95)
            thresholds.append({'candidate': name, 'cluster': cluster, 'development_distance_p95': cutoff[cluster]})
        exceed = nearest > np.array([cutoff[int(c)] for c in labels])
        assigned = data[['WPID_RANDOM','COUNTRY_ISO3','GlobalRegion']].copy()
        assigned['candidate'], assigned['cluster'] = name, labels
        assigned['nearest_distance'], assigned['margin'] = nearest, margin
        assigned['beyond_development_profile_p95'] = exceed
        assignment_tables.append(assigned)
        scopes = [('global','global_development','Global development', np.ones(len(data),bool))]
        scopes += [('region',str(int(r)),regional_names[r],data.GlobalRegion.eq(r).to_numpy()) for r in sorted(data.GlobalRegion.unique())]
        scopes += [('country',c,c,data.COUNTRY_ISO3.eq(c).to_numpy()) for c in sorted(data.COUNTRY_ISO3.unique())]
        for scope,key,label,mask in scopes:
            local_weights = country_weights(data.loc[mask])
            for cluster in [None] + list(range(spec.global_k)):
                select = np.ones(mask.sum(),bool) if cluster is None else labels[mask] == cluster
                if not select.any():
                    continue
                ww = local_weights[select]
                summaries.append({'candidate':name, 'scope':scope,'area_id':key,'area_name':label,
                    'cluster': -1 if cluster is None else cluster, 'respondents':int(select.sum()),
                    'distance_median':weighted_quantile(nearest[mask][select],ww,.5),
                    'distance_p90':weighted_quantile(nearest[mask][select],ww,.9),
                    'margin_median':weighted_quantile(margin[mask][select],ww,.5),
                    'margin_p10':weighted_quantile(margin[mask][select],ww,.1),
                    'profile_p95_exceedance':float(np.average(exceed[mask][select],weights=ww))})
    table_write(pd.DataFrame(thresholds),Path(output_dir)/'development_distance_thresholds_provisional.csv')
    table_write(pd.DataFrame(summaries),Path(output_dir)/'fixed_global_assignment_quality.csv')
    table_write(pd.concat(assignment_tables),Path(output_dir)/'development_fixed_assignments_with_quality.csv')
    return pd.DataFrame(summaries), pd.DataFrame(thresholds)

In [ ]:
# Objective: Assess how well fixed centres represent respondents in each area.
quality_summary = pd.read_csv(OUTPUT/'fixed_global_assignment_quality.csv')
provisional_distance_thresholds = pd.read_csv(OUTPUT/'development_distance_thresholds_provisional.csv')
display(quality_summary.loc[quality_summary.scope.eq('region') & quality_summary.cluster.eq(-1),
    ['candidate','area_name','distance_median','distance_p90','margin_median','margin_p10','profile_p95_exceedance']])
display(fixed_semantic_differences.loc[fixed_semantic_differences.scope.eq('region')]
        .sort_values('largest_absolute_category_difference_pp',ascending=False).head(12))

## 4d Country prevalence maps

**Objective:** Visualize fixed-global profile prevalence without conflating missing data with zero.

**Output:** B/E prevalence maps, country-key checks, and cartographic provenance.

Use the saved global assignments, not independently fitted local labels. Every panel uses the same 0–100% colour scale. Held-out countries are hatched and contain no analysed responses; countries outside the development set have no profile value rather than a zero. Tiny countries use coloured dots so they remain visible. These are within-country survey-weighted shares among complete respondents, not country populations or causal risk estimates. Natural Earth cartography is public-domain map geometry only; the country-to-region classification still comes exclusively from the survey dataset. No geographical regrouping is performed.

Source: [Natural Earth](https://www.naturalearthdata.com/about/terms-of-use/), cached 1:50m country boundaries. Country-key joins and the boundary-file SHA256 are saved. A country missing from the map keys stops the map stage rather than being silently omitted.

In [ ]:
# Objective: Visualize fixed-global profile prevalence without conflating missing data with zero.
def prevalence_maps(prevalence, root, output_dir):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon, Patch
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable
    cartography = Path(root)/'course_artifacts/cartography/ne_50m_admin_0_countries.geojson'
    world = json.loads(cartography.read_text())
    registry = pd.read_csv(Path(root)/'course_artifacts/week03_country_split.csv')
    heldout = set(registry.loc[registry.country_split.eq('heldout_country'),'COUNTRY_ISO3'])
    features = world['features']
    mapping = {}
    for feature in features:
        p = feature['properties']
        code = p.get('ADM0_A3')
        if code in ['KOS','SDS']:
            code = {'KOS':'XKX','SDS':'SSD'}[code]
        mapping[code] = feature
    dev = set(registry.loc[registry.country_split.eq('train'),'COUNTRY_ISO3'])
    missing = sorted(dev-set(mapping))
    if missing:
        raise ValueError('Country-map keys missing: '+', '.join(missing))
    cmap = plt.get_cmap('viridis')
    norm = Normalize(0,100)
    plots = Path(output_dir)/'figures'; plots.mkdir(exist_ok=True)
    files = []
    for name,spec in SPECIFICATIONS.items():
        nrows = 2 if name == 'B' else 4
        fig, axes = plt.subplots(nrows,2,figsize=(15,4*nrows),squeeze=False)
        for cluster,ax in enumerate(axes.ravel()):
            selected = prevalence.loc[prevalence.candidate.eq(name)&prevalence.scope.eq('country')&prevalence.cluster.eq(cluster)]
            values = (selected.set_index('area_id').equal_country_weighted_share*100).to_dict()
            for code,feature in mapping.items():
                if code == 'ATA':
                    continue
                geometry = feature['geometry']
                polygons = geometry['coordinates'] if geometry['type']=='MultiPolygon' else [geometry['coordinates']]
                colour = cmap(norm(values[code])) if code in values else ('#c7c7c7' if code in heldout else '#eeeeee')
                for polygon in polygons:
                    points = np.array(polygon[0])
                    ax.add_patch(Polygon(points, closed=True,facecolor=colour, edgecolor='#858585',linewidth=.18,
                                         hatch='///' if code in heldout else None))
                if code in values:
                    p=feature['properties']
                    bounds=np.array(max(polygons,key=lambda polygon: np.ptp(np.array(polygon[0])[:,0])*np.ptp(np.array(polygon[0])[:,1]))[0])
                    if np.ptp(bounds[:,0])*np.ptp(bounds[:,1])<2:
                        ax.scatter(float(p['LABEL_X']),float(p['LABEL_Y']),s=18,color=colour,edgecolor='black',linewidth=.3,zorder=5)
            ax.set_xlim(-180,180);ax.set_ylim(-60,85);ax.set_aspect('equal')
            ax.set_xticks([]);ax.set_yticks([])
            ax.set_title(f'{name} C{cluster}: {WORKING_PROFILE_LABELS[name][cluster]}',fontsize=11)
            ax.set_facecolor('#f4f8fc')
        fig.subplots_adjust(left=.02,right=.90,bottom=.08,top=.94,hspace=.20)
        bar=fig.colorbar(ScalarMappable(norm=norm,cmap=cmap),cax=fig.add_axes([.92,.20,.015,.60]))
        bar.set_label('Within-country survey-weighted profile share (%)')
        fig.legend(handles=[Patch(facecolor='#c7c7c7',hatch='///',label='Held-out: responses unopened'),
                            Patch(facecolor='#eeeeee',label='No development data')],loc='lower center',bbox_to_anchor=(.5,.04),ncol=2)
        fig.suptitle(f'{name}: fixed global profiles in development countries',fontsize=17)
        fig.text(.02,.015,'Complete cases only. Tiny countries use dots. Natural Earth boundaries; de facto depiction is not a political claim.',fontsize=9)
        path=plots/f'{name}_fixed_global_prevalence_maps.png'
        fig.savefig(path,dpi=160,bbox_inches='tight');plt.close(fig);files.append(path)
    atomic_json({'source_url':'https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_50m_admin_0_countries.geojson',
                 'source_sha256':file_sha256(cartography),'cartography_only':True,'development_keys_matched':len(dev),
                 'missing_development_keys':missing},Path(output_dir)/'map_source_audit.json')
    return files

In [ ]:
# Objective: Display preserved fixed-global prevalence maps; no model or geographical redesign.
from IPython.display import Image
for name in SPECIFICATIONS:
    display(Image(filename=str(OUTPUT/'figures'/f'{name}_fixed_global_prevalence_maps.png')))

## 5 Fix and audit regional sampling plans before fitting

**Objective:** Construct and audit regional resampling from country metadata before fitting.

**Output:** Exact respondent/country samples, distinct-subset counts, omissions, and small-region limitations.

Reuse the same regional respondent/country plans across B/E and k candidates. Respondent sampling keeps approximately 80% without replacement within every country. Country sampling uses whole countries, with no response-derived selection. The audit reports the exact test type, the number of distinct fits, and every country's omission count.

With two–four countries, use one fit per omitted country. With five or more countries, use all 50 random samples even when subsets repeat; report the exact number of distinct country subsets and actual retained-country fraction. Rounding down means, for example, a six-country region retains four countries, not five. With only one development country, country stability is unavailable. All pairwise ARIs are dependent; they are descriptive summaries, not independent observations for a statistical test.

In [ ]:
# Objective: Construct and audit regional resampling from country metadata before fitting.
def make_regional_plans(frame, region, design):
    names = sorted(frame.COUNTRY_ISO3.unique())
    indices = {name: np.flatnonzero(frame.COUNTRY_ISO3.to_numpy() == name) for name in names}
    repetitions = design['repetitions']
    seed = design['seed'] + int(region) * 10000
    respondent = []
    for repeat in range(repetitions):
        rng = np.random.default_rng(seed + repeat)
        respondent.append(np.sort(np.concatenate([
            rng.choice(idx, max(1, round(design['fraction'] * len(idx))), replace=False)
            for idx in indices.values()])))
    if len(names) == 1:
        subsets, kind = [], 'unavailable_one_country'
        subset_count = 0
    else:
        selected = min(len(names) - 1, max(1, math.floor(design['fraction'] * len(names))))
        subset_count = math.comb(len(names), selected)
        if len(names) < 5:
            subsets = [tuple(name for name in names if name != omitted) for omitted in names]
            kind = 'leave_one_country_out'
        else:
            rng = np.random.default_rng(seed + 5000)
            subsets = [tuple(sorted(rng.choice(names, selected, replace=False))) for _ in range(repetitions)]
            kind = 'random_country_subsamples'
    country = [np.flatnonzero(frame.COUNTRY_ISO3.isin(subset).to_numpy()) for subset in subsets]
    audit = [{'country': name, 'omitted_count': sum(name not in subset for subset in subsets),
              'country_evaluations': len(subsets), 'diagnostic': kind} for name in names]
    info = {'country_diagnostic': kind, 'country_fit_count': len(subsets),
            'distinct_country_fits': len(set(subsets)), 'possible_80pct_subsets': subset_count,
            'country_stability_claim_available': kind == 'random_country_subsamples'}
    return {'respondents': respondent, 'countries': country}, pd.DataFrame(audit), info

In [ ]:
# Objective: Read preserved country-sampling and omission audits, without rewriting plans.
sampling_audit = pd.read_csv(OUTPUT/'regional_sampling_design_audit.csv')
display(sampling_audit[['region_name','countries','complete_respondents','country_diagnostic_y',
                       'distinct_country_fits','possible_80pct_subsets']].rename(columns={'country_diagnostic_y':'country_test'}))

## 6 Regional reference fits

**Objective:** Recover local structure under fixed representations and a bounded k search.

**Output:** 270 cached reference fits and local/global overlap, ARI, structural, and profile tables.

Fit K-means within each region under the fixed B and E geometries, with k=2–10 and 20 initializations per fit. B4/E8 are the matched-resolution comparisons. This is a bounded k search, not a new encoding/algorithm search. Fixed seeds are recorded; one numerical thread per fit limits resource contention.

Caches depend on input and code fingerprints, software versions, regional design, specification, seed, and training indices. Recorded failures remain failures until explicitly retried; retries preserve the previous failure. Failed, degenerate, and iteration-limit fits are reported rather than assigned substitute results.

In [ ]:
# Objective: Recover local structure under fixed representations and a bounded k search.
def cached_kmeans(frame, spec, k, training_rows, seed, cache_dir, context, tag, force_retry=False, full_matrix=None):
    payload = {'context': context, 'spec': asdict(spec), 'k': int(k), 'seed': int(seed), 'tag': tag,
               'training_rows_hash': hashlib.sha256(np.asarray(training_rows, dtype=np.int64).tobytes()).hexdigest()}
    key = digest_json(payload)
    path = Path(cache_dir) / (key + '.joblib')
    if path.exists():
        saved = joblib.load(path)
        if saved.get('cache_key') != key:
            raise ValueError('Corrupt or stale regional cache.')
        if saved['status'] == 'ok' or not force_retry:
            return saved
        # Explicit retry preserves previous failures.
        history_path = path.with_name(path.stem + '_failure_' + file_sha256(path)[:12] + '.joblib')
        if not history_path.exists():
            atomic_joblib(saved, history_path)
    result = {'cache_key': key, 'payload': payload, 'status': 'failed', 'labels': None}
    try:
        training = frame.iloc[training_rows]
        if len(training) < k:
            raise ValueError('Fewer training respondents than requested clusters.')
        with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
            matrix = transform(frame, spec) if full_matrix is None else full_matrix
            model = KMeans(n_clusters=k, n_init=20, max_iter=300, random_state=seed)
            model.fit(matrix[training_rows], sample_weight=country_weights(training))
            labels = model.predict(matrix).astype('int16')
        # A max-iteration stop is retained as a fitting limitation, not a silently successful fit.
        if model.n_iter_ >= 300:
            raise RuntimeError('K-means reached the declared iteration limit.')
        if len(np.unique(labels)) != k:
            raise RuntimeError('The requested number of nonempty clusters was not recovered.')
        result.update(status='ok', labels=labels, centres=model.cluster_centers_,
                      iterations=int(model.n_iter_), inertia=float(model.inertia_),
                      warnings=[str(w.message) for w in caught])
    except (ValueError, RuntimeError, MemoryError) as error:
        result.update(error=str(error), traceback=traceback.format_exc())
    atomic_joblib(result, path)
    return result

def pairwise_stability(results, reference):
    good = [r['labels'] for r in results if r['status'] == 'ok']
    pairwise = [adjusted_rand_score(a, b) for a, b in itertools.combinations(good, 2)]
    reference_ari = [adjusted_rand_score(a, reference) for a in good]
    return {'requested_fits': len(results), 'successful_fits': len(good),
            'failed_fits': len(results) - len(good), 'pairwise_count': len(pairwise),
            'median_pairwise': float(np.median(pairwise)) if pairwise else np.nan,
            'p10_pairwise': float(np.quantile(pairwise, .1)) if pairwise else np.nan,
            'minimum_pairwise': float(np.min(pairwise)) if pairwise else np.nan,
            'median_reference': float(np.median(reference_ari)) if reference_ari else np.nan,
            'p10_reference': float(np.quantile(reference_ari, .1)) if reference_ari else np.nan}, np.asarray(pairwise), np.asarray(reference_ari)

In [ ]:
# Objective: Recover local structure under fixed representations and a bounded k search.
def regional_references(data, references, design, output_dir, context):
    output_dir = Path(output_dir)
    summary, overlaps, profile_tables = [], [], []
    fitted = {}
    for region in sorted(data.GlobalRegion.unique()):
        mask = data.GlobalRegion.eq(region).to_numpy()
        frame = data.loc[mask].reset_index(drop=True)
        weights = country_weights(frame)
        for name, spec in SPECIFICATIONS.items():
            full_matrix = transform(frame, spec)
            for k in design['k_values']:
                tag = f'region_{region}_{name}_k{k}_reference'
                result = cached_kmeans(frame, spec, k, np.arange(len(frame)), design['seed'] + int(region),
                                       output_dir / 'fits', context, tag, full_matrix=full_matrix)
                fitted[(int(region), name, int(k))] = result
                row = {'region': int(region), 'candidate': name, 'k': k, 'status': result['status'],
                       'matched_global_resolution': k == spec.global_k, 'countries': frame.COUNTRY_ISO3.nunique(),
                       'respondents': len(frame)}
                if result['status'] == 'ok':
                    row.update(structure(result['labels'], weights))
                    comparison, overlap = compare_partitions(result['labels'], references[name]['labels'][mask], weights)
                    row.update(comparison, iterations=result['iterations'], inertia=result['inertia'])
                    for col, value in [('region', int(region)), ('candidate', name), ('k', k)]:
                        overlap[col] = value
                    overlaps.append(overlap)
                    profiles = original_profiles(frame, result['labels'], weights)
                    profiles['region'], profiles['candidate'], profiles['k'] = int(region), name, k
                    profile_tables.append(profiles)
                else:
                    row['error'] = result['error']
                summary.append(row)
                table_write(pd.DataFrame(summary), output_dir / 'regional_reference_summary.csv')
                print(f'Region {region}, {name}, k={k}: {result["status"]}', flush=True)
    table_write(pd.concat(overlaps, ignore_index=True) if overlaps else pd.DataFrame(), output_dir / 'regional_overlaps.csv')
    table_write(pd.concat(profile_tables, ignore_index=True) if profile_tables else pd.DataFrame(), output_dir / 'regional_original_profiles.csv')
    return pd.DataFrame(summary), fitted

def regional_stability(data, fitted, shortlist, design, output_dir, context, n_jobs=2):
    # Every selection is recorded before the stability stage. Matched B4/E8 are the starting set.
    output_dir = Path(output_dir)
    table_write(pd.DataFrame(shortlist, columns=['region', 'candidate', 'k']), output_dir / 'regional_stability_shortlist.csv')
    summaries, plan_rows, pair_rows = [], [], []
    for region in sorted({int(item[0]) for item in shortlist}):
        frame = data.loc[data.GlobalRegion.eq(region)].reset_index(drop=True)
        plans, omission, plan_info = make_regional_plans(frame, region, design)
        omission['region'] = region
        table_write(omission, output_dir / f'region_{region}_country_omissions.csv')
        for kind, samples in plans.items():
            for repeat, rows in enumerate(samples):
                plan_rows.append({'region': region, 'test': kind, 'repeat': repeat, 'respondents': len(rows),
                    'country_list': ' '.join(sorted(frame.iloc[rows].COUNTRY_ISO3.unique())), **plan_info})
        atomic_joblib({'context': context, 'plans': plans, 'info': plan_info}, output_dir / f'region_{region}_sampling_plans.joblib')
        for _, name, k in [item for item in shortlist if int(item[0]) == region]:
            ref = fitted[(region, name, k)]
            if ref['status'] != 'ok':
                summaries.append({'region': region, 'candidate': name, 'k': k, 'status': 'reference_failed'})
                continue
            for kind, samples in plans.items():
                full_matrix = transform(frame, SPECIFICATIONS[name])
                from joblib import Parallel, delayed, parallel_config
                with parallel_config(backend='loky', inner_max_num_threads=1):
                    results = Parallel(n_jobs=n_jobs)(delayed(cached_kmeans)(
                        frame, SPECIFICATIONS[name], k, training,
                        design['seed'] + region * 10000 + 1000 + repeat,
                        output_dir / 'fits', context, f'region_{region}_{name}_k{k}_{kind}_{repeat:02d}',
                        full_matrix=full_matrix) for repeat, training in enumerate(samples))
                print(f'Region {region}, {name}, k={k}, {kind}: {sum(r["status"] == "ok" for r in results)}/{len(samples)} fits succeeded', flush=True)
                stats, pairwise, ref_ari = pairwise_stability(results, ref['labels'])
                numerical_pass = (stats['successful_fits'] == len(samples) and len(samples) >= 2
                                  and stats['median_pairwise'] >= .8 and stats['p10_pairwise'] >= .8)
                formal_pass = numerical_pass and len(samples) == 50
                if kind == 'countries' and plan_info['country_diagnostic'] != 'random_country_subsamples':
                    formal_pass = False  # LOO summaries are diagnostics, not the 50-fit country-stability claim.
                summaries.append({'region': region, 'candidate': name, 'k': k, 'test': kind,
                    'status': 'complete' if samples else 'unavailable', **stats, **plan_info,
                    'numerical_ari_benchmark_met': bool(numerical_pass),
                    'full_50_fit_stability_benchmark_met': bool(formal_pass)})
                for pair, ari in enumerate(pairwise):
                    pair_rows.append({'region': region, 'candidate': name, 'k': k, 'test': kind,
                                      'pair_number': pair, 'ari_unweighted': float(ari)})
                table_write(pd.DataFrame(summaries), output_dir / 'regional_stability_summary.csv')
                failure_rows = [{'region': region, 'candidate': name, 'k': k, 'test': kind,
                                 'repeat': i, 'error': r.get('error', '')}
                                for i,r in enumerate(results) if r['status'] != 'ok']
                if failure_rows:
                    existing_path = output_dir / 'regional_failures.csv'
                    previous = pd.read_csv(existing_path) if existing_path.exists() else pd.DataFrame()
                    table_write(pd.concat([previous, pd.DataFrame(failure_rows)]).drop_duplicates(), existing_path)
    table_write(pd.DataFrame(plan_rows), output_dir / 'regional_sampling_plan_audit.csv')
    table_write(pd.DataFrame(pair_rows), output_dir / 'regional_pairwise_ari.csv')
    return pd.DataFrame(summaries)

In [ ]:
# Objective: Read all 270 successful regional reference diagnostics, not refit them.
regional_summary = pd.read_csv(OUTPUT/'regional_reference_summary.csv')
assert len(regional_summary) == 270 and regional_summary.status.eq('ok').all()
regional_fitted = {}  # Only needed for new fits; selected cached references are loaded in section 12.
display(regional_summary.loc[regional_summary.matched_global_resolution])

## 7 Stability of selected regional fits

**Objective:** Measure respondent and country sensitivity of the matched B4/E8 regional fits.

**Output:** 50-fit or LOO summaries, all failures, and dependent pairwise ARIs; unsupported tests remain unavailable.

Start with matched-resolution B4/E8 in every region. Add other local k values only after documenting their reference-profile interpretation; declare the shortlist before the stability run. We do not run 50+50 refits for all 270 regional candidates automatically.

Each fit predicts the full same regional cohort. For respondent resampling, all 50 fits must succeed; median and lower-decile pairwise ARI≥0.80 remain the benchmark. The same rule applies to a full 50-fit country test. LOO summaries use their actual number of fits and explicitly cannot pass that 50-fit country benchmark. Regional structural benchmarks remain separate from stability; there is no automatic winner or final global model revision.

In [ ]:
# Objective: Read the initial matched B4/E8 regional stability results; section 12 extends this set.
REGIONAL_STABILITY_SHORTLIST = [(int(region), name, spec.global_k)
    for region in sorted(data.GlobalRegion.unique()) for name, spec in SPECIFICATIONS.items()]
stability_summary = pd.read_csv(OUTPUT/'regional_stability_summary.csv')
assert len(stability_summary) == 60
display(stability_summary)

## 8 Inspect local structure and compare it with global profiles

**Objective:** Inspect the substantive patterns behind numerical local/global agreement.

**Output:** Matched-resolution comparison table and a reusable original-unit plotting function.

For every completed regional reference, inspect the saved overlap and original-response tables. Distinguish changed frequency from changed profile meaning. Examine which groups recur, merge, split, or appear only locally. The heatmap below shows original very-worried proportions and observed personal/known-person harm proportions; these are not the transformed model inputs.

Local labels are arbitrary and are not automatically renamed to global profile IDs. Many-to-one overlap and directional purity must guide any proposed correspondence. Record limitations of single-country regions and unsupported fixed-k fits explicitly.

In [ ]:
# Objective: Inspect the substantive patterns behind numerical local/global agreement.
def plot_regional_profile(region, candidate, k):
    path = OUTPUT / 'regional_original_profiles.csv'
    if not path.exists():
        print('No regional profile output yet. Enable the approved reference stage first.')
        return
    profiles = pd.read_csv(path)
    selected = profiles.loc[profiles.region.eq(region) & profiles.candidate.eq(candidate) & profiles.k.eq(k)]
    if selected.empty:
        raise ValueError('No successful fit for that region/specification/k.')
    worry = selected.loc[selected.feature.isin(WORRY) & selected.category.eq(2)]
    harm = selected.loc[selected.feature.isin(PERSONAL + KNOWN) & selected.category.eq(1)]
    matrices = [worry.pivot(index='feature', columns='cluster', values='proportion').reindex(WORRY),
                harm.pivot(index='feature', columns='cluster', values='proportion').reindex(PERSONAL + KNOWN)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 7), gridspec_kw={'width_ratios':[1,1]})
    for ax, matrix, title in zip(axes, matrices, ['Very worried', 'Harm experienced']):
        image = ax.imshow(matrix, vmin=0, vmax=1, aspect='auto', cmap='YlGnBu')
        ax.set_yticks(range(len(matrix)), matrix.index.str.replace('_', ' '))
        ax.set_xticks(range(len(matrix.columns)), ['C'+str(c) for c in matrix.columns])
        ax.set_xlabel('Local cluster ID')
        ax.set_title(title)
        for row in range(len(matrix)):
            for col in range(len(matrix.columns)):
                value = matrix.iloc[row,col]
                ax.text(col, row, f'{100*value:.0f}', ha='center', va='center',
                        color='white' if value > .55 else 'black', fontsize=8)
    region_name = regions.set_index('GlobalRegion').loc[region,'region_name']
    fig.suptitle(f'{region_name}: {candidate} geometry, local k={k}; equal-country weighted percentages')
    fig.tight_layout()
    plt.show()

if len(regional_summary):
    display(regional_summary.loc[regional_summary.matched_global_resolution,
        ['region','candidate','k','status','ari_unweighted','local_to_global_purity',
         'global_to_local_purity','global_benchmark_pass']])
else:
    print('Profile comparison awaits regional reference computation.')
# Example after fitting: plot_regional_profile(13, 'E', 8)

## 8a Regional semantic correspondence and exploratory resolutions

**Objective:** Examine local/global semantic correspondence and delimit the exploratory k search.

**Output:** Overlap-associated profile comparisons, original-unit TV differences, and illustrative regional profiles.

Compare matched-resolution local clusters with global original-response distributions. Associate each local cluster with its largest weighted global overlap; this is an association, not an automatic identity or renaming. Report that dominant global fraction and mean feature-wise total variation across the 20 original categorical features (six worry, fourteen harm indicators). For each feature, total variation is half the sum of absolute category-probability differences. These descriptive differences have no declared semantic pass/fail threshold.

Inspect overlaps, defining features, intensity, harm, and stability together. High overlap can coexist with shifted characteristics; low ARI can reflect local splits/merges under forced global k rather than entirely new meanings. k=2–10 reference fits are exploratory; only matched B4/E8 had been stability-tested at the initial v3 stage; sections 11–16 now extend those tests. No alternative local k is declared a supported winner merely because it matches the global partition or has a low inertia.

In [ ]:
# Objective: Examine local/global semantic correspondence and delimit the exploratory k search.
def matched_regional_semantics(root_profiles, fixed_profiles, overlaps, output_dir):
    rows=[]
    global_table=fixed_profiles.loc[fixed_profiles.scope.eq('global')]
    for name,spec in SPECIFICATIONS.items():
        local=root_profiles.loc[root_profiles.candidate.eq(name)&root_profiles.k.eq(spec.global_k)]
        for (region,cluster),group in local.groupby(['region','cluster']):
            cross=overlaps.loc[overlaps.candidate.eq(name)&overlaps.k.eq(spec.global_k)&overlaps.region.eq(region)&overlaps.local_cluster.eq(cluster)]
            top=cross.loc[cross.global_given_local.idxmax()]
            reference=global_table.loc[global_table.candidate.eq(name)&global_table.cluster.eq(top.global_cluster)]
            aligned=group.merge(reference[['feature','category','proportion']],on=['feature','category'],suffixes=('_local','_global'),validate='one_to_one')
            aligned['absolute_difference']=abs(aligned.proportion_local-aligned.proportion_global)
            feature_tv=aligned.groupby('feature').absolute_difference.sum()/2
            strongest=aligned.loc[aligned.absolute_difference.idxmax()]
            rows.append({'region':int(region),'candidate':name,'k':spec.global_k,'local_cluster':int(cluster),
                'associated_global_cluster':int(top.global_cluster),'dominant_global_fraction':float(top.global_given_local),
                'working_global_description':WORKING_PROFILE_LABELS[name][int(top.global_cluster)],
                'mean_original_feature_TV':float(feature_tv.mean()),
                'max_category_difference_pp':100*float(strongest.absolute_difference),
                'max_difference_feature':strongest.feature,'max_difference_category':int(strongest.category),
                'local_cluster_share':float(group.cluster_share.iloc[0])})
    result=pd.DataFrame(rows)
    table_write(result,Path(output_dir)/'matched_regional_semantic_correspondence.csv')
    return result

In [ ]:
# Objective: Examine local/global semantic correspondence and delimit the exploratory k search.
local_profiles = pd.read_csv(OUTPUT/'regional_original_profiles.csv')
local_overlaps = pd.read_csv(OUTPUT/'regional_overlaps.csv')
semantic_correspondence = pd.read_csv(OUTPUT/'matched_regional_semantic_correspondence.csv')
display(semantic_correspondence.sort_values('dominant_global_fraction').head(12))
display(semantic_correspondence.sort_values('mean_original_feature_TV',ascending=False).head(12))
display(regional_summary.groupby('candidate').agg(successful_references=('status',lambda s:int(s.eq('ok').sum()))))
# Illustrative local profiles: a low-B-agreement region and a high-E-agreement region.
plot_regional_profile(13, 'B', 4)
plot_regional_profile(7, 'E', 8)

## 9 Development decision and freeze preparation

**Objective:** Record readiness for a later final decision without freezing models or opening the holdout.

**Output:** Checklist distinguishing computed evidence from unapproved final scope and held-out access.

The regional analysis may qualify B/E interpretations or justify a documented development-side revision. B and E can be complementary; do not force them into one hierarchy. Review the regional fits and limitations before deciding whether one or both models are retained. Optional native-Ward comparison is not a required completion step.

The table below is a checklist, not a freeze record. Completed regional results do not automatically approve final scope or held-out access. The next coding stage will save the approved centres, profile IDs, transforms, provenance, development distance thresholds, and decision log; only then implement and run the gated held-out transfer diagnostics. The distance thresholds saved here are provisional development references, not a final freeze. Do not use the superseded Week 4 search-v3 handoff as the candidate source.

In [ ]:
# Objective: Record readiness for a later final decision without freezing models or opening the holdout.
def freeze_checklist(data, references):
    # Descriptive preparation; never writes a final model or marks the decision approved.
    rows = []
    for name, spec in SPECIFICATIONS.items():
        weights = country_weights(data)
        rows.append({'candidate': name, 'k': spec.global_k, 'encoding': spec.encoding,
            'alpha': spec.alpha, 'assignment_reload_verified': True,
            **structure(references[name]['labels'], weights),
            'regional_analysis_complete': False, 'final_scope_approved': False,
            'holdout_access_authorized': False})
    return pd.DataFrame(rows)

In [ ]:
# Objective: Read the baseline freeze-preparation checklist, not freeze a model.
checklist = pd.read_csv(OUTPUT/'freeze_preparation_checklist.csv')
display(checklist)
display(Markdown('The matched-resolution stage is complete. Sections 11–16 extend the local search. '
                '**No final freeze and no held-out response access.**'))

## 10 Initial matched-resolution conclusions (v3 evidence, retained for context)

**Objective:** distinguish geographical prevalence, representation quality, local recovery, and local stability using completed evidence.

**Output:** substantive conclusions with numerical support and explicit limits. This is not final model selection, held-out validation, or a country-similarity analysis.

### Computation and scope

All 270 regional reference fits (15 regions × two fixed geometries × nine k values) succeeded. The matched-resolution B4/E8 stability stage completed 1,500 respondent fits and 1,114 country/leave-one-country-out fits: **2,614 successful fits, zero fitting failures**. The saved region/country original-response tables, overlaps, quality summaries, and maps complete point 1; the reference search and matched-resolution stability/semantic analyses complete point 2 within the stated scope. k=2–10 alternatives are exploratory reference fits, not stability-qualified local winners. The global B/E centres and assignments are unchanged. No held-out responses were opened and no final model was frozen.

### Point 1 Fixed global profiles have geographically different frequencies and characteristics

Both models show a broad worry gradient. B's lower-worry/harm C3 is 53.4% in Northern/Western Europe and 51.6% in Central Asia, versus 17.7% in Central/Western Africa and 13.1% in Southern Africa. E's lower-worry C1 is 59.9% in Northern/Western Europe, while broad high-worry C7 is 2.5%; in Central/Western Africa those shares are 18.2% and 20.0%. B additionally distinguishes broad known-person harm exposure. These are descriptive, equal-country-weighted complete-case results, not causal claims, population-proportional regional estimates, or evidence that every country in a region is alike.

Fixed labels do not ensure identical semantics. For example, among global E C7 members, very-worried traffic prevalence is 34.1% in Northern/Western Europe versus 74.6% globally. Among B C0 members it is 18.9% there versus 54.4% globally. These are shifts within a fixed global profile, not changes to the model. The original-category tables show all such contrasts and respondent counts; selected extremes are illustrative, not a multiple-comparison significance test. Sparse profile/area cells require caution: the US-only E C2 cell has 63 respondents, despite a large weighted personal mental-health-harm difference (67.6% versus 15.2% globally).

Representation quality varies as well. In the full development cohort, profile-specific distance-p95 exceedance is 4.99% for B and 4.98% for E. Central/Western Africa has 8.7% for B and 7.8% for E; Eastern Africa has 8.4% and 8.1%. These areas contribute more unusually distant assignments under the fixed models. This is an in-development descriptive warning, not a transfer failure or a declared regional pass/fail rule. Distances are interpreted within each model; margins are geometric separation diagnostics, not posterior probabilities.

The maps retain one 0–100% scale across all B/E panels, mark held-out countries as unopened, and distinguish absent data from zero. They do not replace the numerical tables. Northern America contains only the US in development, and Australia/New Zealand only New Zealand; their regional rows are single-country descriptions. Complete-case selection may differ by country. No survey-design confidence intervals were estimated in this stage.

### Point 2 Regional refits do not generally reproduce the global partitions

For matched resolutions, local-versus-global ARI ranges from **0.135 to 0.733 for B4** and **0.499 to 0.893 for E8**. E is closer to its global partition in 14 of 15 regions in this particular reference run; Latin America/Caribbean is the exception (B 0.733, E 0.729). Central Asia has E ARI 0.893, and Southern Europe 0.822. B's strongest match is Latin America/Caribbean (0.733); Northern/Western Europe is 0.203. These are descriptive agreement values, not retention thresholds or inferential comparisons between the methods. Reference seed and fixed k affect local results.

No matched regional B4 solution passes the median and lower-decile ARI≥0.80 benchmark in both respondent and country tests. E8 passes both full 50-fit tests only in **Northern/Western Europe**: respondent median/lower-decile 0.975/0.958; country 0.971/0.944. Northern Africa E8 passes the respondent benchmark (0.885/0.824) and its four-fit LOO numerical benchmark (0.874/0.826), but four omitted-country diagnostics are not a 50-fit country-stability claim. The US and New Zealand have no country test. Failure to pass here concerns reproducibility of these specific local procedures/resolutions; it does not establish that no regional structure exists.

The lower decile matters. Latin America/Caribbean B4 has country median ARI 0.926 but lower decile 0.375. Central Asia E8 has excellent reference agreement (0.893) and respondent stability (0.949/0.923), yet country lower decile 0.702. A good median or a good single reference match cannot substitute for the full stability evidence. Country omissions and random initializations both vary in these fits; the results do not isolate the effect of geography alone.

### A stable local model can still differ meaningfully from the global model

Northern/Western Europe E8 is stable, yet its local/global ARI is only **0.571**. Two local groups, covering 17.5% and 40.1% of the region, predominantly map to global low-worry C1 (99.6% and 100.0% of their weights, respectively). Known-person mental-health harm is 99.7% in the first and 0% in the second. Regional refitting therefore resolves a harm contrast within the global low-worry profile. It also creates a 4.5% food-very-worried group that spans several global profiles; its largest overlap is only 40.6% with global C1. Local cluster IDs remain arbitrary and these are overlap associations, not automatic global relabeling.

This local E8 fit does **not** pass the global structural benchmark: its largest share is 40.07% and D=3.206. Regional size patterns were explicitly not governed by that global retention rule, so we report it as a stable local partition with a structural caveat rather than silently relaxing the global rule or declaring it a new global finalist.

### Small-country sampling and interpretation limits

Moodle's floor rule makes the actual retained-country fraction lower than 80% in some regions: Southern Asia retains four of six countries (66.7%), and Southeastern Asia five of seven (71.4%). Thus regional/global stability thresholds are numerical benchmarks under different sample sizes and sampling designs, not perfectly equivalent stress tests. In five-country regions, 50 country fits use only five distinct four-country subsets. Repeated subsets and dependent pairwise ARIs are explicitly reported; they are not 50 distinct geographic validations. The two–four-country LOO summaries are diagnostic and the one-country cases are unavailable.

### Implication for the next stage

B and E remain global reference descriptions with complementary content; these results do not yet authorize a final freeze. E shows more recurring thematic structure, but neither model should be presented as an unchanged local partition in every region. We should qualify the geographic scope and distinguish shared profiles with different prevalence from local splits/merges and semantic shifts. Final model scope and held-out transfer remain separate decisions. At this initial stage no alternative local k had been stability-qualified. The new local-resolution evidence in sections 11–16 supersedes that scope limitation; no new geographic grouping or country clustering is introduced.

**v4 update:** the negative matched-resolution results must not be read as absence of local structure. The bounded additional-k tests below support several alternative local resolutions. The cluster-recovery audit also qualifies the earlier description of Northern/Western Europe E8 as stable: the whole partition is stable, but not every smaller reference profile is recovered equally well.

## 11 Why extend the local k search?

**Objective:** separate “does the global resolution recur locally?” from “which local resolutions are supported?”

**Output:** a bounded, reproducible development-only selection rule, independent of local/global ARI.

Forcing B4 and E8 was a controlled comparison, not a local optimization procedure. A region may support fewer or different distinctions than the pooled development data. A failed B4/E8 local stability test therefore does not imply that no repeatable local structure exists.

We keep the algorithm, complete-case cohort, equal-country weights and two Week 4 geometries unchanged. B uses 0/1/2 worry coding and worry budget \(\alpha=0.5\); E uses 0/1/3 and \(\alpha=0.75\). The six normalized worry deviations and the saved overall-worry coordinate use \(\sqrt{\alpha/6}\); the fourteen separate personal/known-person harm indicators use \(\sqrt{(1-\alpha)/14}\). Section 2 gives the exact transformation. No additional imputation, standardization or PCA is introduced. Thus B/E comparisons also change coding and geometry, not just k.

For every region and geometry we already have references for k=2,…,10. We now stability-test at most three additional k: **2, 3, and one near the within-region inertia elbow**, excluding the previously tested matched k. Normalize k to \(x=(k-2)/8\), inertia to \(z=(I_k-I_{10})/(I_2-I_{10})\), and use gap \(g=(1-x)-z\). Among eligible nonmatched k=3,…,9, choose the smallest k whose gap is at least 95% of the maximum gap. This coarsest-near-tie rule avoids claiming a precise elbow. In this run the extra B values are 2/3/5 everywhere; E uses 2/3/4 in Eastern Africa, Northern Africa and Central Asia, and 2/3/5 elsewhere.

Original response distributions and smallest-group sizes are inspected before the new stability stage, but no undocumented semantic pass/fail threshold is used. All selected alternatives are tested, including unpromising ones. **Global ARI is never used to choose or retain k.** The global structural screen (k≥3, largest share≤40%, D<3) remains a reported benchmark, not a local filter: imposing it would exclude all two-group regional descriptions by construction.

This extension was planned **after seeing the matched-resolution results**. It is an exploratory, adaptive development analysis, not an independently preregistered confirmation. The bounded shortlist is a computational compromise: untested k have unknown stability. Neither minimum inertia nor the elbow identifies a uniquely “true” or “optimal” k. We retain sets of benchmark-qualified resolutions, not one winner per region.

In [ ]:
# Objective: Implement the recorded local-k selection and fingerprint-checked cache/recovery helpers.
"""Development-only exploratory local resolution selection; no new geometry or holdout."""

LOCAL_SELECTION_DESIGN = {
    'version': 1, 'coarse_k': [2, 3],
    'elbow_eligible_k': list(range(3,10)),
    'elbow_rule': 'smallest eligible nonmatched k within 95% of maximum normalized chord-minus-inertia gap',
    'max_additional_k_per_region_representation': 3,
    'global_ari_used_for_selection': False,
    'structural_global_rules_used_as_local_filters': False,
    'representation_algorithm_changes': False,
    'primary_stability': 'existing median and lower-decile pairwise ARI >=0.80 in both tests',
    'cluster_recovery': 'weighted Hungarian-overlap matching to each regional reference; descriptive Jaccard, no new threshold',
    'search_status': 'exploratory development selection after observing matched-resolution results; not independent confirmation',
}

def base_analysis_load(root, base_output):
    root, base_output = Path(root), Path(base_output)
    manifest = json.loads((base_output/'run_manifest.json').read_text())
    data,audit,regions,references,provenance = prepare_development(root)
    if manifest['provenance'] != provenance:
        raise ValueError('Baseline data, saved models, or runtime changed. Do not reuse these caches.')
    # Selection and all new fits use the identical fitting code and declared 2..10 design.
    if manifest['regional_design'] != REGIONAL_DESIGN:
        raise ValueError('Regional fitting/sampling design differs from baseline.')
    reference_summary=pd.read_csv(base_output/'regional_reference_summary.csv')
    profiles=pd.read_csv(base_output/'regional_original_profiles.csv')
    return data,regions,references,provenance,manifest['context'],reference_summary,profiles

def reference_cache(frame, region, name, k, cache_dir, context):
    spec=SPECIFICATIONS[name]
    seed=REGIONAL_DESIGN['seed']+int(region)
    payload={'context':context,'spec':asdict(spec),'k':int(k),'seed':int(seed),
             'tag':f'region_{region}_{name}_k{k}_reference',
             'training_rows_hash':hashlib.sha256(np.arange(len(frame),dtype=np.int64).tobytes()).hexdigest()}
    key=digest_json(payload)
    path=Path(cache_dir)/(key+'.joblib')
    if not path.exists():
        raise FileNotFoundError('Reference fit is missing; selection does not silently refit it: '+str(path))
    cached=joblib.load(path)
    if cached.get('cache_key')!=key or cached['status']!='ok':
        raise ValueError('Reference cache failed validation.')
    return cached

def profile_pairwise_differences(profiles):
    rows=[]
    for (region,name,k),group in profiles.groupby(['region','candidate','k']):
        matrix=group.pivot(index='cluster',columns=['feature','category'],values='proportion')
        pair=[]
        for a,b in itertools.combinations(matrix.index,2):
            difference=abs(matrix.loc[a]-matrix.loc[b])
            feature_tv=difference.groupby(level='feature').sum()/2
            pair.append(float(feature_tv.mean()))
        shares=group.groupby('cluster').cluster_share.first()
        rows.append({'region':int(region),'candidate':name,'k':int(k),
            'minimum_original_feature_TV_between_clusters':min(pair),
            'median_original_feature_TV_between_clusters':float(np.median(pair)),
            'smallest_cluster_share':float(shares.min())})
    return pd.DataFrame(rows)

def select_local_resolutions(reference_summary, profiles):
    # Deliberately omit all global agreement columns from the selection calculation.
    allowed=reference_summary[['region','candidate','k','status','inertia','largest_share','dominance']].copy()
    semantic=profile_pairwise_differences(profiles)
    decisions=[]
    diagnostics=[]
    for (region,name),group in allowed.groupby(['region','candidate']):
        group=group.loc[group.status.eq('ok')].sort_values('k').copy()
        assert group.k.tolist()==list(range(2,11))
        x=(group.k.to_numpy()-2)/8
        inertia=group.inertia.to_numpy()
        gap=(1-x)-(inertia-inertia[-1])/(inertia[0]-inertia[-1])
        group['normalized_elbow_gap']=gap
        group['incremental_inertia_gain_pct']=100*(1-group.inertia/group.inertia.shift(1))
        eligible=group.loc[group.k.isin(LOCAL_SELECTION_DESIGN['elbow_eligible_k'])
                           &group.k.ne(SPECIFICATIONS[name].global_k)]
        elbow=int(eligible.loc[eligible.normalized_elbow_gap.ge(.95*eligible.normalized_elbow_gap.max()),'k'].min())
        chosen=sorted(set(LOCAL_SELECTION_DESIGN['coarse_k']+[elbow])-{SPECIFICATIONS[name].global_k})
        for k in chosen:
            reasons=[]
            if k in [2,3]:reasons.append('prespecified coarse alternative; avoids forcing global resolution')
            if k==elbow:reasons.append('near within-region inertia elbow; coarsest of near-tied elbow candidates')
            decisions.append({'region':int(region),'candidate':name,'k':k,'selection_reason':'; '.join(reasons),
                              'global_agreement_used':False,'reference_only_semantic_review_required':True})
        diagnostics.append(group)
    diagnostics=pd.concat(diagnostics,ignore_index=True).merge(semantic,on=['region','candidate','k'],validate='one_to_one')
    decisions=pd.DataFrame(decisions).merge(diagnostics,on=['region','candidate','k'],validate='one_to_one')
    assert decisions.groupby(['region','candidate']).size().le(3).all()
    return decisions,diagnostics

def selection_profile_review(data, decisions, fitted, output_dir):
    rows=[]
    for region,name,k in decisions[['region','candidate','k']].itertuples(index=False,name=None):
        frame=data.loc[data.GlobalRegion.eq(region)].reset_index(drop=True)
        labels=fitted[(region,name,k)]['labels'];weights=country_weights(frame)
        profiles=original_profiles(frame,labels,weights)
        for cluster,group in profiles.groupby('cluster'):
            ranked=group.sort_values('difference_vs_outside',ascending=False)
            anchors=[]
            for row in ranked.head(3).itertuples():
                category_label=('very' if row.category==2 else 'somewhat' if row.category==1 else 'not worried') if row.feature in WORRY else ('harm' if row.category==1 else 'no harm')
                anchors.append(f'{row.feature}: {category_label} {100*row.proportion:.1f}% (outside difference {100*row.difference_vs_outside:+.1f}pp)')
            inside=labels==cluster;ww=weights[inside]
            rows.append({'region':region,'candidate':name,'k':k,'cluster':int(cluster),
                'cluster_share':float(ww.sum()),'respondents':int(inside.sum()),
                'effective_weighted_n':float(ww.sum()**2/(ww**2).sum()),
                'strongest_original_contrasts':' | '.join(anchors),
                'max_absolute_category_contrast':float(group.difference_vs_outside.abs().max()),
                'review_status':'screening description; not a frozen profile name'})
    result=pd.DataFrame(rows)
    table_write(result,Path(output_dir)/'shortlisted_profile_review.csv')
    return result

def cached_repetition(frame,region,name,k,kind,repeat,training,cache_dir,context):
    spec=SPECIFICATIONS[name]
    payload={'context':context,'spec':asdict(spec),'k':int(k),
             'seed':REGIONAL_DESIGN['seed']+int(region)*10000+1000+int(repeat),
             'tag':f'region_{region}_{name}_k{k}_{kind}_{repeat:02d}',
             'training_rows_hash':hashlib.sha256(np.asarray(training,dtype=np.int64).tobytes()).hexdigest()}
    key=digest_json(payload);path=Path(cache_dir)/(key+'.joblib')
    if not path.exists():raise FileNotFoundError('Missing declared repetition: '+str(path))
    result=joblib.load(path)
    if result.get('cache_key')!=key:raise ValueError('Repetition cache fingerprint mismatch.')
    return result

def cluster_recovery_summary(data,shortlist,fitted,cache_dir,context,output_dir):
    all_rows=[]
    for region,name,k in shortlist:
        frame=data.loc[data.GlobalRegion.eq(region)].reset_index(drop=True)
        weights=country_weights(frame)
        reference=fitted[(region,name,k)]['labels']
        ref_mass=np.bincount(reference,weights=weights,minlength=k)
        plans,_,_=make_regional_plans(frame,region,REGIONAL_DESIGN)
        for kind,indices in plans.items():
            values={j:[] for j in range(k)}
            for repeat,training in enumerate(indices):
                result=cached_repetition(frame,region,name,k,kind,repeat,training,cache_dir,context)
                if result['status']!='ok':continue
                labels=result['labels']
                overlap=np.bincount(reference.astype('int64')*k+labels.astype('int64'),weights=weights,minlength=k*k).reshape(k,k)
                rr,cc=linear_sum_assignment(-overlap)
                pred_mass=overlap.sum(axis=0)
                for ref,pred in zip(rr,cc):
                    union=ref_mass[ref]+pred_mass[pred]-overlap[ref,pred]
                    values[ref].append(float(overlap[ref,pred]/union) if union else np.nan)
            for cluster,series in values.items():
                all_rows.append({'region':region,'candidate':name,'k':k,'test':kind,'reference_cluster':cluster,
                    'reference_share':ref_mass[cluster], 'successful_repetitions':len(series),
                    'median_reference_jaccard':float(np.median(series)) if series else np.nan,
                    'p10_reference_jaccard':float(np.quantile(series,.1)) if series else np.nan})
    result=pd.DataFrame(all_rows)
    table_write(result,Path(output_dir)/'cluster_recovery_summary.csv')
    return result

def supported_resolution_summary(data,regions,stability,cluster_recovery,reference_summary,output_dir):
    rows=[]
    names=regions.set_index('GlobalRegion').region_name.to_dict()
    for (region,name,k),group in stability.groupby(['region','candidate','k']):
        respondent=group.loc[group.test.eq('respondents')].iloc[0]
        country=group.loc[group.test.eq('countries')].iloc[0]
        country_count=int(data.loc[data.GlobalRegion.eq(region)].COUNTRY_ISO3.nunique())
        respondent_pass=bool(respondent.full_50_fit_stability_benchmark_met)
        country_pass=bool(country.full_50_fit_stability_benchmark_met)
        loo_pass=country_count in [2,3,4] and bool(country.numerical_ari_benchmark_met)
        if respondent_pass and country_pass:status='both full stability benchmarks met'
        elif respondent_pass and loo_pass:status='respondent benchmark + limited LOO diagnostics met'
        elif respondent_pass and country_count==1:status='respondent-only benchmark; geographic stability unavailable'
        else:status='not qualified by the applicable stability benchmarks'
        ref=reference_summary.loc[reference_summary.region.eq(region)&reference_summary.candidate.eq(name)&reference_summary.k.eq(k)].iloc[0]
        recovery=cluster_recovery.loc[cluster_recovery.region.eq(region)&cluster_recovery.candidate.eq(name)&cluster_recovery.k.eq(k)]
        rr=recovery.loc[recovery.test.eq('respondents')]
        cc=recovery.loc[recovery.test.eq('countries')]
        rows.append({'region':int(region),'region_name':names[region],'candidate':name,'k':int(k),
            'countries':country_count,'status':status,'respondent_median_ari':respondent.median_pairwise,
            'respondent_p10_ari':respondent.p10_pairwise,'country_median_ari':country.median_pairwise,
            'country_p10_ari':country.p10_pairwise,'distinct_country_subsets':country.distinct_country_fits,
            'worst_cluster_respondent_p10_jaccard':rr.p10_reference_jaccard.min(),
            'worst_cluster_country_p10_jaccard':cc.p10_reference_jaccard.min(),
            'largest_share':ref.largest_share,'dominance':ref.dominance,
            'global_structural_benchmark_pass':bool(ref.global_benchmark_pass),
            'local_global_ari_descriptive_only':ref.ari_unweighted})
    result=pd.DataFrame(rows)
    table_write(result,Path(output_dir)/'supported_local_resolutions.csv')
    return result

def plot_local_stability_grid(summary, regions, output_dir):
    import matplotlib.pyplot as plt
    for name in SPECIFICATIONS:
        fig,axes=plt.subplots(1,2,figsize=(14,7))
        for ax,column,title in zip(axes,['respondent_p10_ari','country_p10_ari'],['Respondent lower-decile ARI','Country/LOO lower-decile ARI']):
            sub=summary.loc[summary.candidate.eq(name)]
            matrix=sub.pivot(index='region',columns='k',values=column).reindex(index=regions.GlobalRegion,columns=range(2,11))
            image=ax.imshow(np.ma.masked_invalid(matrix.to_numpy()),vmin=0,vmax=1,cmap='viridis',aspect='auto')
            ax.set_facecolor('#d9d9d9');ax.set_xticks(range(9),range(2,11));ax.set_xlabel('Local k (grey = untested or unavailable)')
            ax.set_yticks(range(len(regions)),regions.region_name)
            for i in range(len(matrix)):
                for j in range(9):
                    value=matrix.iloc[i,j]
                    if np.isfinite(value):ax.text(j,i,f'{value:.2f}',ha='center',va='center',fontsize=7,color='white' if value<.55 else 'black')
            ax.set_title(title)
        fig.suptitle(f'{name} geometry: tested local resolutions (exploratory selection)',fontsize=14)
        fig.tight_layout();fig.subplots_adjust(bottom=.11,right=.93)
        fig.colorbar(image,cax=fig.add_axes([.95,.20,.012,.60]))
        fig.text(.02,.025,'Benchmark 0.80. Northern/Southern Africa: LOO, not 50 country fits. US/NZ: country test unavailable. Pairwise ARIs are dependent.',fontsize=8)
        output=Path(output_dir)/'figures';output.mkdir(exist_ok=True)
        fig.savefig(output/f'{name}_local_resolution_stability.png',dpi=150,bbox_inches='tight')
        plt.show()


## 12 Replay the additional tests and distinguish evidence levels

**Objective:** verify provenance and recover the completed regional tests without duplicate fitting.

**Output:** shortlist, all 120 tested resolutions, successful/failed fit counts, dependent pairwise-ARI summaries and cluster-recovery diagnostics.

The existing samples and fitting settings are reused for each new k: 20 initializations, maximum 300 iterations, 50 within-country 80% respondent subsamples without replacement; for regions with at least five countries, 50 samples of floor(0.8N) whole countries. Training weights are recomputed so included countries have equal total influence. Each fitted model assigns the **entire same regional cohort** by nearest centre, including omitted respondents/countries. These are partition sensitivity tests, not independent out-of-sample prediction-accuracy estimates. Data changes and randomized K-means initializations both vary; neither effect is isolated here.

Both tests require all 50 fits to succeed and **median AND lower-decile pairwise ARI≥0.80**, using unrounded values. Fifty fits generate 1,225 pairwise ARIs; those comparisons are dependent, not 1,225 independent validations. The standard ARI is unweighted even though fitting is equal-country weighted. For 2–4 countries we report all leave-one-country-out fits as limited diagnostics, not full 50-fit country evidence. For one country, geographic stability is unavailable. Repeated country subsets and the actual retained-country fractions remain visible in the sampling audit.

The resume switch is off. Turning it on is only appropriate after verifying that no fitting process is already running; successful fingerprint-checked fits are then reused. Fit failures would remain recorded rather than silently retried. The completed run has **zero model-fitting failures**. An initial preflight attempt stopped because it expected an optional implementation-hash key absent from the replayed v3 manifest; no fits ran in that attempt. Provenance verification was corrected to recompute the context from data/runtime, design and the actual baseline implementation hash. This was an orchestration/schema mismatch, not a failed clustering result.

In [ ]:
# Objective: Verify saved evidence and load completed tests; no duplicate fits by default.
LOCAL_OUTPUT = ROOT / 'course_artifacts/week05_local_resolutions_v4/f3145c704f52c156'
LOCAL_SELECTION_CONTEXT = 'f3145c704f52c156269125ab4174582e2b95aa4aac9ee71172ce146324938a18'
RESUME_ADDITIONAL_FITS = False
selection_manifest = json.loads((LOCAL_OUTPUT/'selection_manifest.json').read_text())
assert selection_manifest['base_context'] == CONTEXT
assert selection_manifest['selection_context'] == LOCAL_SELECTION_CONTEXT
assert selection_manifest['selection_design'] == LOCAL_SELECTION_DESIGN
assert selection_manifest['heldout_responses_used'] is False
evidence_manifest = json.loads((LOCAL_OUTPUT/'evidence_manifest.json').read_text())
assert evidence_manifest['selection_context'] == LOCAL_SELECTION_CONTEXT
for folder, key in [(OUTPUT, 'base_files'), (LOCAL_OUTPUT, 'selection_files')]:
    for filename, expected_hash in evidence_manifest[key].items():
        assert file_sha256(folder/filename) == expected_hash, 'Saved evidence changed: ' + filename
decisions, local_diagnostics = select_local_resolutions(regional_summary, local_profiles)
expected_shortlist = pd.read_csv(LOCAL_OUTPUT/'additional_local_shortlist.csv')
pd.testing.assert_frame_equal(decisions[['region','candidate','k']].reset_index(drop=True),
                              expected_shortlist[['region','candidate','k']].reset_index(drop=True))
calculated_context = digest_json({'base_context': CONTEXT, 'selection_design': LOCAL_SELECTION_DESIGN,
    'shortlist': decisions[['region','candidate','k']].to_dict('records'),
    'extension_code_sha256': selection_manifest['extension_code_sha256']})
assert calculated_context == LOCAL_SELECTION_CONTEXT
tested = sorted(set(list(decisions[['region','candidate','k']].itertuples(index=False,name=None))
                    + REGIONAL_STABILITY_SHORTLIST))
local_fitted = {}
for region, name, k in tested:
    frame = data.loc[data.GlobalRegion.eq(region)].reset_index(drop=True)
    local_fitted[(region,name,k)] = reference_cache(frame,region,name,k,OUTPUT/'fits',CONTEXT)
if RESUME_ADDITIONAL_FITS:
    additional = regional_stability(data,local_fitted,
        list(decisions[['region','candidate','k']].itertuples(index=False,name=None)),
        REGIONAL_DESIGN,LOCAL_OUTPUT,CONTEXT,n_jobs=2)
    combined_stability = pd.concat([stability_summary,additional],ignore_index=True)
    cluster_recovery = cluster_recovery_summary(data,tested,local_fitted,OUTPUT/'fits',CONTEXT,LOCAL_OUTPUT)
    supported = supported_resolution_summary(data,regions,combined_stability,cluster_recovery,regional_summary,LOCAL_OUTPUT)
else:
    combined_stability = pd.read_csv(LOCAL_OUTPUT/'combined_regional_stability_summary.csv')
    cluster_recovery = pd.read_csv(LOCAL_OUTPUT/'cluster_recovery_summary.csv')
    supported = pd.read_csv(LOCAL_OUTPUT/'supported_local_resolutions.csv')
assert len(supported) == 120 and len(combined_stability) == 240
assert not supported.duplicated(['region','candidate','k']).any()
completion = json.loads((LOCAL_OUTPUT/'local_selection_complete.json').read_text())
display(completion)
display(supported.groupby('status').size().rename('tested_resolutions').to_frame())
display(supported.loc[supported.status.eq('both full stability benchmarks met'),
    ['region_name','candidate','k','respondent_median_ari','respondent_p10_ari',
     'country_median_ari','country_p10_ari','largest_share','global_structural_benchmark_pass']].round(4))
print('All 270 reference fits and 10,456 matched/additional stability fits succeeded. No held-out responses used.')

## 13 Partition stability is not recovery of every profile

**Objective:** prevent large, repeatable groups from hiding fragile smaller groups or a poor saved reference.

**Output:** per-cluster weighted reference Jaccard, original-category distributions and contrasts, counts, weighted sample-size diagnostics, and local/global semantic associations for **every benchmark-qualified resolution**.

For each repeated fit, match its k clusters to the k reference clusters by maximizing total **equal-country-weighted overlap** (Hungarian assignment). Then compute \(J(A,B)=w(A\cap B)/w(A\cup B)\) for each matched reference cluster. Summarize its median and lower decile across fits. This is a one-to-one recovery diagnostic, not a new retention threshold; matching maximizes total overlap, not every cluster's Jaccard. Splits/merges can therefore leave an individual group poorly matched. Whole-partition ARI and individual-cluster Jaccard answer different questions and use different weighting.

Describe profiles using the **original three worry categories** and separate binary personal/known-person harm, not transformed centroids alone. For each local cluster save its share, respondent count, countries represented, strongest original-unit contrasts, and descriptive Kish weighted n. Kish n measures weight concentration; it is not a survey-design-adjusted effective sample size or confidence interval.

Associate a local cluster with its largest global overlap without forcing equal k or one-to-one global names. Report that overlap and mean feature-wise total variation from both the pooled global profile and that fixed profile **within the same region**. The latter helps distinguish general regional response differences from local regrouping. TV ignores dependence between features and has no semantic pass/fail threshold here. A near-pure-category flag (≥98% in a category and ≥50 percentage-point excess versus outside) identifies a possible single-feature anchor; it is **not** a cross-validated single-feature reconstruction test. These are descriptive interpretations, not validated social types.

In [ ]:
# Objective: Define reporting-only tables and figures; do not change selection or fit models.
"""Reporting only: does not select resolutions, change fits, or access held-out responses."""

FULL_SUPPORT = 'both full stability benchmarks met'
LOO_SUPPORT = 'respondent benchmark + limited LOO diagnostics met'
SINGLE_SUPPORT = 'respondent-only benchmark; geographic stability unavailable'

def local_support_table(supported, regions, output_dir):
    rows = []
    for region in regions.itertuples():
        row = {'region': int(region.GlobalRegion), 'region_name': region.region_name,
               'countries': int(region.countries)}
        for name in SPECIFICATIONS:
            group = supported.loc[supported.region.eq(region.GlobalRegion) & supported.candidate.eq(name)]
            row[name + '_tested_k'] = ', '.join(map(str, sorted(group.k)))
            for status, suffix in [(FULL_SUPPORT, 'full'), (LOO_SUPPORT, 'limited_LOO'),
                                   (SINGLE_SUPPORT, 'respondent_only')]:
                values = sorted(group.loc[group.status.eq(status), 'k'])
                row[name + '_' + suffix] = ', '.join(map(str, values)) if values else 'none'
        rows.append(row)
    result = pd.DataFrame(rows)
    table_write(result, Path(output_dir) / 'regional_supported_k_sets.csv')
    return result

def qualified_local_semantics(data, supported, profiles, overlaps, fixed_profiles, fitted, output_dir):
    """All qualified resolutions: category contrasts, sample size, overlap and semantic shifts."""
    qualified = supported.loc[supported.status.isin([FULL_SUPPORT, LOO_SUPPORT, SINGLE_SUPPORT])]
    rows, category_tables = [], []
    for resolution in qualified.itertuples():
        region, name, k = resolution.region, resolution.candidate, resolution.k
        frame = data.loc[data.GlobalRegion.eq(region)].reset_index(drop=True)
        labels = fitted[(region, name, k)]['labels']
        weights = country_weights(frame)
        selected = profiles.loc[profiles.region.eq(region) & profiles.candidate.eq(name) & profiles.k.eq(k)]
        category_tables.append(selected.assign(stability_support=resolution.status))
        ov = overlaps.loc[overlaps.region.eq(region) & overlaps.candidate.eq(name) & overlaps.k.eq(k)]
        for cluster, group in selected.groupby('cluster'):
            inside = labels == cluster
            ww = weights[inside]
            dominant = ov.loc[ov.local_cluster.eq(cluster)].sort_values('global_given_local', ascending=False).iloc[0]
            gc = int(dominant.global_cluster)
            original = group.set_index(['feature', 'category']).proportion
            # Include low/middle worry and absence of harm: low-burden profiles should not
            # be described by their least-negative "very worried" or harm contrasts.
            contrasts = group
            anchors = []
            for anchor in contrasts.sort_values('difference_vs_outside', ascending=False).head(3).itertuples():
                label = anchor.feature.replace('_', ' ')
                if anchor.feature in WORRY:
                    label += ': ' + {0: 'not worried', 1: 'somewhat worried', 2: 'very worried'}[anchor.category]
                else:
                    label += ': harmed' if anchor.category == 1 else ': no harm'
                anchors.append(f'{label} {100*anchor.proportion:.1f}% ({100*anchor.difference_vs_outside:+.1f}pp vs outside)')
            semantic_tv = {}
            for scope, area in [('global', None), ('region', region)]:
                ref = fixed_profiles.loc[fixed_profiles.candidate.eq(name) & fixed_profiles.scope.eq(scope)
                                         & fixed_profiles.cluster.eq(gc)]
                if area is not None:
                    ref = ref.loc[pd.to_numeric(ref.area_id, errors='coerce').eq(area)]
                if ref.empty:
                    semantic_tv[scope] = np.nan
                else:
                    difference = abs(original - ref.set_index(['feature', 'category']).proportion)
                    assert difference.notna().all()
                    semantic_tv[scope] = float((difference.groupby(level='feature').sum()/2).mean())
            # Near-pure category flags are descriptive anchors, not a reconstruction test.
            pure = group.loc[group.proportion.ge(.98) & group.difference_vs_outside.ge(.5)]
            flags = ' | '.join(f'{r.feature}, category {r.category}' for r in pure.itertuples())
            rows.append({'region': region, 'region_name': resolution.region_name, 'candidate': name,
                         'k': k, 'cluster': int(cluster), 'stability_support': resolution.status,
                         'cluster_share': float(ww.sum()), 'respondents': int(inside.sum()),
                         'countries_represented': int(frame.loc[inside, 'COUNTRY_ISO3'].nunique()),
                         'kish_weighted_n_descriptive': float(ww.sum()**2 / (ww**2).sum()),
                         'original_response_anchors': ' | '.join(anchors),
                         'near_pure_original_category_flags': flags,
                         'largest_overlap_global_cluster': gc,
                         'largest_overlap_fraction': float(dominant.global_given_local),
                         'mean_feature_TV_vs_global_profile': semantic_tv['global'],
                         'mean_feature_TV_vs_same_region_fixed_profile': semantic_tv['region']})
    result = pd.DataFrame(rows)
    table_write(result, Path(output_dir)/'qualified_local_profile_interpretations.csv')
    if category_tables:
        table_write(pd.concat(category_tables, ignore_index=True),
                    Path(output_dir)/'qualified_local_original_category_distributions.csv')
    return result

def compare_supported_local_resolutions(data, supported, fitted, output_dir):
    """Descriptive, weighted split/merge evidence; independently fitted k are not nested."""
    qualified = supported.loc[supported.status.isin([FULL_SUPPORT, LOO_SUPPORT, SINGLE_SUPPORT])]
    rows, cells = [], []
    for (region, name), group in qualified.groupby(['region', 'candidate']):
        frame = data.loc[data.GlobalRegion.eq(region)].reset_index(drop=True)
        weights = country_weights(frame)
        for coarse, fine in itertools.combinations(sorted(group.k), 2):
            co = fitted[(region, name, coarse)]['labels']
            fi = fitted[(region, name, fine)]['labels']
            stats, overlap = compare_partitions(fi, co, weights)
            # fine -> coarse purity = mass belonging to each fine group's dominant coarse group.
            rows.append({'region': region, 'candidate': name, 'coarse_k': coarse, 'fine_k': fine,
                         'ari_unweighted': stats['ari_unweighted'],
                         'fine_to_coarse_purity': stats['local_to_global_purity'],
                         'coarse_to_fine_purity': stats['global_to_local_purity'],
                         'note': 'approximate split/merge comparison; not a nested hierarchy'})
            overlap = overlap.rename(columns={'local_cluster': 'fine_cluster', 'global_cluster': 'coarse_cluster',
                                               'global_given_local': 'coarse_given_fine', 'local_given_global': 'fine_given_coarse'})
            cells.append(overlap.assign(region=region, candidate=name, coarse_k=coarse, fine_k=fine))
    result = pd.DataFrame(rows)
    table_write(result, Path(output_dir)/'supported_local_resolution_comparisons.csv')
    if cells:
        table_write(pd.concat(cells, ignore_index=True), Path(output_dir)/'supported_local_resolution_overlaps.csv')
    return result

def plot_qualified_local_profiles(profiles, supported, regions, output_dir):
    """A deliberately labelled illustrative subset; complete category tables remain available."""
    requested = [(2, 'B', 3), (13, 'B', 2), (13, 'E', 8), (7, 'E', 3)]
    qualified = set(supported.loc[supported.status.isin([FULL_SUPPORT, LOO_SUPPORT]),
                                 ['region', 'candidate', 'k']].itertuples(index=False, name=None))
    files = []
    for region, name, k in requested:
        if (region, name, k) not in qualified:
            continue
        sel = profiles.loc[profiles.region.eq(region) & profiles.candidate.eq(name) & profiles.k.eq(k)]
        matrices = [sel.loc[sel.feature.isin(WORRY) & sel.category.eq(2)].pivot(index='feature', columns='cluster', values='proportion').reindex(WORRY),
                    sel.loc[sel.feature.isin(PERSONAL+KNOWN) & sel.category.eq(1)].pivot(index='feature', columns='cluster', values='proportion').reindex(PERSONAL+KNOWN)]
        fig, axes = plt.subplots(1, 2, figsize=(15, 7), layout='constrained')
        for ax, matrix, title in zip(axes, matrices, ['Very worried (%)', 'Personal / known-person harm (%)']):
            im = ax.imshow(matrix, aspect='auto', cmap='YlGnBu', vmin=0, vmax=1)
            ax.set_yticks(range(len(matrix)), [x.replace('worry_', '').replace('harm_', '').replace('_', ' ') for x in matrix.index])
            shares = sel.groupby('cluster').cluster_share.first()
            ax.set_xticks(range(k), [f'C{c}\n{100*shares[c]:.1f}%' for c in matrix.columns])
            ax.set_xlabel('Local cluster ID and equal-country-weighted share')
            ax.set_title(title)
            for i in range(len(matrix)):
                for j in range(k):
                    value = matrix.iloc[i,j]
                    ax.text(j, i, f'{100*value:.0f}', ha='center', va='center', fontsize=8,
                            color='white' if value > .55 else 'black')
        region_name = regions.set_index('GlobalRegion').loc[region, 'region_name']
        fig.suptitle(f'{region_name}: {name} geometry, local k={k}\nIllustration, not a unique optimal resolution; development complete cases only', fontsize=13)
        fig.colorbar(im, ax=axes, shrink=.65, label='Weighted proportion')
        folder = Path(output_dir)/'figures'; folder.mkdir(exist_ok=True)
        path = folder/f'{name}_region{region}_k{k}_qualified_original_profiles.png'
        fig.savefig(path, dpi=160, bbox_inches='tight')
        plt.close(fig); files.append(path)
    return files

def plot_report_stability(supported, regions, output_dir):
    """Readable two-panel grid; explicit distinction between untested k and unavailable tests."""
    paths = []
    for name in SPECIFICATIONS:
        fig, axes = plt.subplots(1, 2, figsize=(15, 8), sharey=True)
        for ax, field, title in zip(axes, ['respondent_p10_ari', 'country_p10_ari'],
                                    ['Respondent lower-decile ARI', 'Country / LOO lower-decile ARI']):
            sub = supported.loc[supported.candidate.eq(name)]
            matrix = sub.pivot(index='region', columns='k', values=field).reindex(index=regions.GlobalRegion, columns=range(2,11))
            im = ax.imshow(np.ma.masked_invalid(matrix.to_numpy()), vmin=0, vmax=1, cmap='viridis', aspect='auto')
            ax.set_facecolor('#dedede'); ax.set_xticks(range(9), range(2,11)); ax.set_xlabel('Local k')
            ax.set_title(title)
            ax.set_yticks(range(len(regions)), regions.region_name)
            for i, region in enumerate(regions.GlobalRegion):
                for j, k in enumerate(range(2,11)):
                    value = matrix.iloc[i,j]
                    if np.isfinite(value):
                        ax.text(j, i, f'{value:.2f}', ha='center', va='center', fontsize=8,
                                color='white' if value < .55 else 'black')
                        # Outline only a resolution that meets BOTH applicable tests.
                        row = sub.loc[sub.region.eq(region) & sub.k.eq(k)]
                        if row.iloc[0].status in [FULL_SUPPORT, LOO_SUPPORT]:
                            from matplotlib.patches import Rectangle
                            ax.add_patch(Rectangle((j-.48,i-.48),.96,.96,fill=False,edgecolor='#d62728',linewidth=1.4))
                    elif k in sub.loc[sub.region.eq(region), 'k'].tolist():
                        ax.text(j, i, 'N/A', ha='center', va='center', fontsize=7)
        axes[1].tick_params(labelleft=False)
        fig.subplots_adjust(left=.25,right=.93,bottom=.13,top=.88,wspace=.12)
        fig.colorbar(im,cax=fig.add_axes([.945,.20,.012,.60]))
        fig.suptitle(f'{name}: exploratory local-resolution stability\nRed outline: both applicable benchmarks met (not a significance test)',fontsize=14)
        fig.text(.025,.055,'Grey blank = untested k. N/A = no country test. Threshold 0.80 uses unrounded values and median AND lower decile.\nNorthern / Southern Africa: only 4 / 3 LOO fits. US / NZ: one-country descriptions. Dependent pairwise ARIs are not independent validations.',fontsize=9)
        folder=Path(output_dir)/'figures';folder.mkdir(exist_ok=True)
        path=folder/f'{name}_local_resolution_stability.png'
        fig.savefig(path,dpi=160,bbox_inches='tight');plt.close(fig);paths.append(path)
    return paths


In [ ]:
# Objective: Save interpretable evidence for every qualified local resolution, not only illustrative winners.
support_table = local_support_table(supported, regions, LOCAL_OUTPUT)
profile_interpretations = qualified_local_semantics(data,supported,local_profiles,local_overlaps,
    fixed_original_profiles,local_fitted,LOCAL_OUTPUT)
resolution_comparisons = compare_supported_local_resolutions(data,supported,local_fitted,LOCAL_OUTPUT)
display(support_table[['region_name','countries','B_full','E_full','B_limited_LOO','E_limited_LOO',
                      'B_respondent_only','E_respondent_only']])
display(profile_interpretations[['region_name','candidate','k','cluster','cluster_share',
    'respondents','original_response_anchors','largest_overlap_global_cluster','largest_overlap_fraction']])
display(supported.loc[supported.status.ne('not qualified by the applicable stability benchmarks'),
    ['region_name','candidate','k','status','worst_cluster_respondent_p10_jaccard',
     'worst_cluster_country_p10_jaccard']].round(3))
display(resolution_comparisons.round(3))

## 14 Visual evidence for slides and the article

**Objective:** show the tested search space and original-response meanings without hiding unavailable or unsuccessful tests.

**Output:** B/E stability grids and illustrative original-unit profile panels, saved as PNGs and embedded below.

The grids show lower-decile ARI; both median and lower decile determine the outlined qualification status. Grey blank cells are untested k, not failures; N/A is an unavailable country test. Rounded labels must not be used for decisions. The selected original-unit examples illustrate African B3, European B2/E8 and Central Asian E3 when qualified; they are not a ranked best-of selection. Complete tables cover every qualified resolution. Each panel uses one 0–100% scale and distinguishes personal from known-person harm. The earlier country maps continue to use fixed global labels: no map pretends that unrelated local IDs have common cross-region meanings.

In [ ]:
# Objective: Save and embed stable-resolution evidence and original-unit examples, without further fits.
stability_figures = plot_report_stability(supported,regions,LOCAL_OUTPUT)
local_profile_figures = plot_qualified_local_profiles(local_profiles,supported,regions,LOCAL_OUTPUT)
for path in stability_figures + local_profile_figures:
    display(Image(filename=str(path)))

## 15 Updated analytical conclusions

**Objective:** state what the additional tests establish, what changes relative to v3, and what remains unproven.

**Output:** a bounded scientific interpretation suitable for adaptation into slides and an article.

### Coarser local descriptions recover structure that matched B4/E8 missed

Of **120 tested regional resolutions**, **14 meet both full partition-stability benchmarks**, across **six regions**. Thirteen are newly tested alternatives; Northern/Western Europe E8 was already qualified. The sets are:

| Development region | B geometry, supported k | E geometry, supported k |
|---|---:|---:|
| Eastern Africa | 2 | 2 |
| Central/Western Africa | 3 | 2 |
| Latin America/Caribbean | 3, 5 | none tested qualified |
| Central Asia | 2, 3 | 2, 3 |
| Southeastern Asia | none tested qualified | 5 |
| Northern/Western Europe | 2 | 2, 8 |

These are **sets**, not unique optima. The other seven multicountry regions have no full-benchmark-qualified tested resolution; this does not prove an absence of structure. Northern Africa E8 additionally meets the respondent benchmark and four-fit LOO numerical benchmark, which is weaker geographic evidence. The US-only E8 and New Zealand-only B3/E3 meet respondent benchmarks but cannot establish regional country stability. Of the 120 tested resolutions, 102 fail at least one applicable benchmark; there are no computational fitting failures. All 270 references and 10,456 matched/additional stability fits succeeded.

The Eastern Africa B2 country lower decile is **0.800272**: it barely meets the fixed 0.80 benchmark. Report it as borderline evidence, not a robust margin above the threshold. Latin America B3 and B5 both qualify; their cross-k overlaps show whether the finer description approximately splits the coarser groups. Separate K-means optima are not guaranteed to be nested, so this is not a strict hierarchy.

### Stability does not automatically establish a multidimensional sociological type

For Central/Western Africa B3, respondent median/lower-decile ARI is **0.959/0.933** and country **0.908/0.868**. It is stronger reproducibility evidence than the failed B4 result. Its substantive meaning still depends on the original-unit profiles below, not its ARI alone.

The regional interpretations are provisional descriptions of observed response patterns, not final names or explanations of their causes:

| Region | Original-response interpretation of the qualified local resolutions |
|---|---|
| Eastern Africa | B2 separates broad known-person harm exposure (36.0%) from lower exposure (64.0%). E2 instead contrasts food/water worry (58.6%) with traffic/severe-weather worry (41.4%). The different content is consistent with the different feature budgets; these are not interchangeable solutions. |
| Central/Western Africa | B3 has a broad harm-exposure group (24.7%), a lower-harm group with food/water concern (41.3%), and a traffic/weather/crime-concern group with low food/water worry (34.0%). E2 primarily contrasts food/water concern (56.9%) with traffic concern and lower food/water worry (43.1%). This is a repeatable, multidomain pattern, not merely agreement with the global labels. |
| Latin America/Caribbean | B3 distinguishes known-person harm exposure (20.0%), lower traffic/weather worry and harm (43.2%), and traffic-oriented concern with low food/water worry (36.7%). B5 adds distinct crime-oriented, mental-health/traffic-oriented, low-crime-worry, broad-harm, and food/water-oriented groups. B5 is not simply B3 with two exact splits: fine-to-coarse weighted purity is 82.8%, leaving 17.2% outside each finer group's dominant coarse association. Both resolutions qualify, so neither is labelled uniquely optimal. |
| Central Asia | B2 almost isolates known-person traffic harm (27.4%; 97.7% report it). B3 retains a similar harm group and divides the rest mainly by traffic/weather worry. E2 is an exact very-worried-traffic contrast (34.3% versus the remainder); E3 isolates near-pure very-worried-water (16.2%) and very-worried-traffic (23.9%) groups plus a lower-worry remainder (59.9%). These strong single-category anchors warrant caution despite excellent ARIs. Fifty country fits here cover only five distinct omitted-country subsets. |
| Southeastern Asia | E5 contains a lower-worry group (37.7%), a food/water-oriented group (14.3%), a traffic-oriented group (14.3%), a mental-health/crime/weather group (16.8%), and a crime/weather-oriented group (16.8%). The traffic group has the weakest reference recovery, so its boundaries need more caution than the whole-partition qualification suggests. |
| Northern/Western Europe | B2 mostly distinguishes known-person mental-health harm; E2 mostly distinguishes very-worried severe weather. E8 adds domain-specific food/water/traffic/weather/mental-health/crime worry groups alongside the two large low-worry groups differentiated by known-person mental-health harm. The large groups are highly repeatable; several smaller groups are less secure. |

The percentages above use the equal-country weights within each region. They are not population shares. The full original-category tables distinguish no/somewhat/very worry and personal/known-person harm for every group; the brief descriptions deliberately do not turn a dominant marginal feature into a causal interpretation.

Northern/Western Europe B2 is almost a division by **known-person mental-health harm**: one group is 31.9% of regional weight with 99.3% reporting it; the other is 68.1% with 0%. E2 instead separates a 18.4% group with 100% very-worried severe weather from the remainder. Both are highly repeatable under their geometries, but these simple categorical contrasts should not be advertised as newly discovered multidimensional social types. Different B/E groupings reflect both observed responses and our different coding/feature budgets. Categorical features can generate sharp, repeatable boundaries.

### High whole-partition ARI can hide weaker smaller profiles

Northern/Western Europe E8 still passes both full ARI benchmarks. Yet its weakest reference-cluster lower-decile Jaccard is **0.644** under respondent sampling and **0.594** under country sampling. The large low-worry/harm split is very repeatable, whereas some smaller thematic groups are less secure. Its eight-profile reference should therefore be described as partition-stable **with uneven individual-profile recovery**, not uniformly stable at every profile.

Southeastern Asia E5 also qualifies globally by ARI but has weakest-cluster lower-decile Jaccard **0.765/0.734**. Northern Africa E8 has still weaker recovery (approximately **0.149/0.148**) despite meeting the respondent and limited LOO ARI benchmarks. For the US-only E8, repeated fits agree well with each other (respondent pairwise median/lower decile **0.919/0.876**) but less with the saved reference (**0.660/0.643**); one reference cluster's matched lower-decile Jaccard is zero. This is evidence against interpreting that saved reference as a reliable eight-profile local description. It is not a numerical crash. Before freezing any such local model, investigate reference/initialization dependence and the affected clusters; no extra fits or post-hoc retention rule are introduced here.

### Global and local models answer different questions

The global B/E profiles provide a common classification for country/region comparisons, while local fits examine which distinctions can be recovered within an area. Low local/global ARI is not in itself weakness or a significant scientific discovery: different resolutions, local feature prevalence and optimization can produce it. The potentially valuable finding is a **repeatable, interpretable difference** supported by overlaps, original-category contrasts and cluster recovery. Global-profile frequencies and within-profile semantics also vary, as documented in sections 4–10. Neither analysis proves universal profiles or successful transfer to unseen countries.

For later country-distance analysis, retain a common global-profile basis, separately for B and E. Local profile coordinates cannot be compared directly across countries or regions without handling semantic alignment and splits/merges. No new regional classification or country clustering is performed here.

### Limits that belong in the article, not only in technical notes

The search is bounded, exploratory and adaptively extended. Threshold crossing is not a p-value, confidence level or guarantee of a true k. Reusing sampling plans helps compare resolutions but does not create independent evidence. With only five countries, 50 country fits cover just five distinct omissions; small-region LOO and one-country rows are weaker still. K-means samples include training respondents when comparing predicted partitions, so this is not held-out transfer validation. The regional complete-case cohort can be selective, equal-country weights define a country-balanced rather than population-proportional estimand, and ARI itself is unweighted. Original-category summaries describe marginals, not causal mechanisms, full joint distributions or validated latent social types. Borderline ARIs, minority-profile fragility and reference dependence must accompany positive findings.

## 15a Research decision: retain ARI 0.80 and distinguish stability from useful profiles

**Objective:** record the discussion and the user's decision without changing the scientific design.

**Output:** unchanged criterion, diagnostic threshold sensitivity, evidence about respondent/country perturbations and explicit interpretation limits. All calculations below use saved tables; no model is refitted.

**Decision, 12 October 2026:** retain the existing criterion: all required fits must succeed, and both the median and lower-decile pairwise ARI must be at least **0.80 in both respondent and country tests**, using unrounded values. LOO and one-country limitations remain unchanged. The 0.75/0.70 calculations are sensitivity descriptions only; they do not promote additional solutions, revise the shortlist, or change saved qualification statuses. No post-hoc Jaccard threshold is introduced either.

### Why the k=2 findings do not fulfil our profiling aim

The 14 resolutions meeting both full partition-stability benchmarks include **seven k=2 and seven k≥3 resolutions**. Binary partitions can reveal dominant contrasts and are worth preserving as diagnostics, but they are not satisfactory final answers to our aim of richer, multidimensional profiles—particularly when a partition nearly reproduces one survey variable. A benchmark-qualified partition is not automatically a scientifically useful typology. The earlier headline of 14 successes therefore overstates progress if read as 14 satisfactory rich profile solutions.

We do not force a larger k or weaken reproducibility to make the profiles appear more interesting. For local models, record separately: partition stability, resolution, individual-profile recovery, original-response meaning and global structural benchmarks. The seven full-benchmark-qualified k≥3 results remain the substantive regional set to review. This does not invalidate global B4/E8, which provide a common classification rather than a promise of independent recovery in every region.

### Could fewer data explain the limited local structure?

Possibly in part, but our present evidence does not establish that cause. Regional analysis has fewer respondents and fewer countries than global analysis, and small or rare profiles may contain little weighted information. However, many richer fits survive 20% respondent removal while every country remains represented, yet fail when whole countries disappear. Even regional cohorts of around 10,000–14,000 respondents show this behaviour. Respondent count alone is not an adequate explanation.

The two tests change different things. Respondent resampling retains every regional country. Country resampling changes country composition, total respondents, profile prevalence and sometimes the severity of the reduction: floor(0.8N) can retain substantially less than 80% of countries. Both also vary K-means random initialization. Consequently, country-test failure does not by itself isolate geographic differences, respondent-sample size or optimization effects.

The saved country tests permit one additional diagnostic without new fits: compare pairs fitted to the **same country subset** versus **different country subsets**. For an identical country subset the entire training cohort and weights are identical, so variation comes from initialization/optimization. For different subsets, country composition and respondent count also change. The comparisons are dependent and differently distributed across subsets; they are descriptive, not a causal experiment or a significance test. Where there are only a handful of same-subset pairs, their lower-decile summaries are especially weak evidence.

### What a lower threshold would admit—and why we retain 0.80

For full-country-test regions, requiring all four ARI summaries to reach 0.75 would increase the qualifying k≥3 set from 7 to 14 resolutions; 0.70 would increase it to 22. Some near-misses have meaningful original profiles and reasonable individual-cluster recovery. Others have reference groups that are almost never recovered under one-to-one overlap matching. Lowering overall ARI cannot resolve that problem.

The 0.80 benchmark is a research criterion, not an intrinsic pass/fail boundary of ARI. Nevertheless, changing it now to obtain richer results would be an adaptive relaxation requiring explicit justification. **We retain it.** Sensitivity results remain visible so conclusions do not treat a borderline failure as categorically different from a borderline pass, but all formal statuses and reported counts stay tied to 0.80.

Methodological background: [Shamir and Tishby, stability and model selection in K-means](https://doi.org/10.1007/s10994-010-5177-8); [official ARI definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html). The numerical evidence here is from our saved course analyses, not those sources.

In [ ]:
# Objective: Reproduce threshold sensitivity and country-subset diagnostics from saved outputs; retain 0.80.
DISCUSSION_OUTPUT = LOCAL_OUTPUT/'discussion_20261012'
DISCUSSION_OUTPUT.mkdir(exist_ok=True)
metric_columns = ['respondent_median_ari','respondent_p10_ari','country_median_ari','country_p10_ari']
full_country = supported.loc[supported.countries.ge(5)].copy()
assert not combined_stability.failed_fits.any(), 'These comparisons require the completed successful-fit record.'
threshold_rows, sensitivity_rows = [], []
primary = full_country[metric_columns].ge(.80).all(axis=1)
for threshold in [.80,.75,.70]:
    qualified = full_country[metric_columns].ge(threshold).all(axis=1)
    rich = qualified & full_country.k.ge(3)
    threshold_rows.append({'threshold':threshold, 'full_partition_benchmark_resolutions':int(qualified.sum()),
        'k_ge_3_resolutions':int(rich.sum()), 'regions_with_k_ge_3':int(full_country.loc[rich,'region'].nunique()),
        'status':'retained primary rule' if threshold==.80 else 'diagnostic sensitivity only; not adopted'})
    additional = full_country.loc[rich & ~primary].copy()
    additional['diagnostic_threshold'] = threshold
    sensitivity_rows.append(additional)
threshold_sensitivity = pd.DataFrame(threshold_rows)
sensitivity_candidates = pd.concat(sensitivity_rows,ignore_index=True)
table_write(threshold_sensitivity,DISCUSSION_OUTPUT/'ari_threshold_sensitivity.csv')
table_write(sensitivity_candidates,DISCUSSION_OUTPUT/'lower_threshold_richer_candidates_not_retained.csv')
display(threshold_sensitivity)
display(sensitivity_candidates.loc[sensitivity_candidates.diagnostic_threshold.eq(.75),
    ['region_name','candidate','k','respondent_p10_ari','country_p10_ari',
     'worst_cluster_country_p10_jaccard']].round(3))
rich = full_country.loc[full_country.k.ge(3)]
resp_pass = rich[['respondent_median_ari','respondent_p10_ari']].ge(.80).all(axis=1)
country_pass = rich[['country_median_ari','country_p10_ari']].ge(.80).all(axis=1)
test_comparison = pd.DataFrame([{'eligible_tested_richer_resolutions':len(rich),
    'respondent_benchmark_met':int(resp_pass.sum()),'country_benchmark_met':int(country_pass.sum()),
    'both_met':int((resp_pass&country_pass).sum()),'respondent_met_country_not_met':int((resp_pass&~country_pass).sum()),
    'country_met_respondent_not_met':int((~resp_pass&country_pass).sum())}])
table_write(test_comparison,DISCUSSION_OUTPUT/'respondent_vs_country_stability.csv')
display(test_comparison)

plans = pd.read_csv(LOCAL_OUTPUT/'regional_sampling_plan_audit.csv')
pairwise = pd.concat([pd.read_csv(OUTPUT/'regional_pairwise_ari.csv'),
                     pd.read_csv(LOCAL_OUTPUT/'regional_pairwise_ari.csv')],ignore_index=True)
assert not pairwise.duplicated(['region','candidate','k','test','pair_number']).any()
subset_rows = []
for (region,name,k), group in pairwise.loc[pairwise.test.eq('countries')].groupby(['region','candidate','k']):
    plan = plans.loc[plans.region.eq(region)&plans.test.eq('countries')].sort_values('repeat')
    if len(plan)!=50:
        continue  # LOO is not repeated 50-subset evidence.
    countries = plan.country_list.to_list()
    pairs = list(itertools.combinations(range(50),2))
    group = group.sort_values('pair_number')
    assert group.pair_number.to_list()==list(range(1225))
    same = np.asarray([countries[a]==countries[b] for a,b in pairs])
    values = group.ari_unweighted.to_numpy()
    for kind, mask in [('same training countries',same),('different training countries',~same)]:
        series = values[mask]
        subset_rows.append({'region':int(region),'candidate':name,'k':int(k),'comparison':kind,
            'pair_count':int(mask.sum()), 'median_ari':float(np.median(series)) if len(series) else np.nan,
            'p10_ari':float(np.quantile(series,.1)) if len(series) else np.nan,
            'interpretation':'dependent descriptive comparisons; country composition and n not independently controlled'})
subset_diagnostics = pd.DataFrame(subset_rows).merge(regions[['GlobalRegion','region_name']],
    left_on='region',right_on='GlobalRegion',validate='many_to_one').drop(columns='GlobalRegion')
table_write(subset_diagnostics,DISCUSSION_OUTPUT/'same_vs_different_country_subset_ari.csv')
display(subset_diagnostics.loc[subset_diagnostics.region.eq(8)&subset_diagnostics.candidate.eq('E')
    &subset_diagnostics.k.eq(8),['region_name','candidate','k','comparison','pair_count','median_ari','p10_ari']].round(3))
threshold_decision = {'date':'2026-10-12','decision':'retain existing ARI 0.80 unchanged',
    'authority':'explicit user instruction in this chat',
    'median_threshold':.80,'lower_decile_threshold':.80,'required_tests':['respondents','countries'],
    'required_successful_fits_per_full_test':50,'small_region_policy_unchanged':True,
    'lower_thresholds_adopted':False,'new_models_fitted':False,'heldout_responses_used':False,
    'final_global_scope_approved':False,'global_models_frozen':False,
    'k2_role':'diagnostic dominant contrasts; not satisfactory final rich-profile solution',
    'source_selection_context':LOCAL_SELECTION_CONTEXT}
atomic_json(threshold_decision,DISCUSSION_OUTPUT/'ari_decision.json')
assert threshold_sensitivity.k_ge_3_resolutions.tolist()==[7,14,22]
assert test_comparison.respondent_met_country_not_met.iloc[0]==23
assert int(supported.loc[supported.status.eq(FULL_SUPPORT),'k'].eq(2).sum())==7
assert int(supported.loc[supported.status.eq(FULL_SUPPORT),'k'].ge(3).sum())==7
display(threshold_decision)
print('0.80 retained; no qualification status changed, no new fitting, no held-out response access.')

## 15b Interpretation of the sensitivity discussion

**Objective:** distinguish plausible near-misses, fragile profiles and untested explanations.

**Output:** a critical interpretation of the saved diagnostic tables without changing the acceptance rule.

Among 66 tested k≥3 resolutions in the 11 regions eligible for full country resampling, 30 meet the respondent benchmark and eight meet the country benchmark; seven meet both. Twenty-three meet respondent stability but fail country stability. The principal observed limitation is therefore country-removal sensitivity, not demonstrated inability to estimate any richer structure from the available respondent counts. Regional size, country diversity, rare-profile information, geometry and randomized optimization remain possible contributing factors.

Eastern Asia E8 has country-test lower-decile ARI **0.854** among 245 pairs sharing identical training countries, versus **0.691** among 980 pairs with different country subsets. In the same-country comparisons, training respondents and weights are fixed and only randomized optimization changes. This supports an additional effect of changing the included countries, but differences in n and composition are not separated. The pair comparisons are dependent. Other regions have too few repeated identical subsets to support a reliable same-subset lower decile; the table retains their pair counts rather than treating them as equally persuasive.

At the unadopted 0.75 threshold, Eastern Africa B3/B4 would qualify with country lower-decile ARIs approximately **0.793/0.784** and weakest-cluster Jaccards **0.877/0.870**. Central/Western Africa B4 would qualify at approximately **0.772**, with weakest-cluster Jaccard **0.806**. These are informative near-misses under the unchanged criterion, not newly retained regional models. Conversely, Latin America E8 and Southern Europe E8 would also enter the 0.75 set despite weakest-cluster country Jaccards around **0.013/0.043**. A relaxed whole-partition ARI rule would not establish dependable recovery of every reference profile.

An optional, separate respondent-sample-size experiment could fit learning curves within a fixed country set and compare stability at several sample fractions. A complementary comparison could downsample global development respondents while preserving country coverage. Neither has been run or authorized here; neither is required to finish the current Objective 1 protocol. They would need a declared fitting/assignment design and would still not turn a count effect into proof of latent social types.

**Conclusion:** preserve 0.80, retain the seven full-benchmark-qualified richer resolutions with their profile-specific caveats, and report binary findings and near-misses as contextual evidence. The local analysis qualifies the geographic scope of the global models; it does not select replacement global centres or require every region to yield a rich standalone typology.

## 16 Evidence files, reproducibility and remaining Week 5 work

**Objective:** make results reusable while keeping exploratory analysis separate from final validation.

**Output:** a saved evidence inventory and a clear completion boundary.

Sections 1–10 preserve the global/regional/country prevalence, semantics, assignment-quality tables, maps, and matched-resolution evidence. The extension directory contains the exact selection design and shortlist, all-k reference diagnostics, all tested stability summaries and pairwise ARIs, sampling audits, per-cluster recovery, qualified original-category distributions and interpretations, supported-k sets, cross-k overlaps, figures, and an input-evidence hash manifest. No selected failure or inconvenient result is omitted. Intermediate fits are shared via a link to the fingerprinted baseline cache; copies of these results must preserve that cache or its relative folder structure.

**Still required before Objective 1 is concluded:** decide the scope of the final global profile specification(s); freeze the complete preprocessing/assignment procedure, development reference diagnostics and interpretation; only then apply and assess the untouched held-out-country responses. Regional resampling is not a substitute for those held-out tests. Regional/local profiles remain exploratory contextual evidence, not silently substituted global finalists. Optional country-similarity analysis is a separate question, not a condition for freezing profiles. The notebook makes no final freeze or held-out validation claim.

**Protocol clarification:** the 7 October validation protocol and the current Moodle description specify regional development analysis → final development decision/freeze → held-out assignment/transfer assessment. They do not impose a new repeated two-half Prediction Strength experiment as a prerequisite here. Earlier course descriptions or old search handoffs must not silently add that requirement back into this plan.

In [ ]:
# Objective: Save the article/slide evidence inventory and certify only the computations actually completed.
report_files = sorted(p.name for p in LOCAL_OUTPUT.glob('*.csv'))
report_inventory = {
    'base_context': CONTEXT, 'selection_context': LOCAL_SELECTION_CONTEXT,
    'development_respondents': len(data), 'development_countries': int(data.COUNTRY_ISO3.nunique()),
    'reference_fits': 270, 'stability_fits': int(combined_stability.successful_fits.sum()),
    'failed_stability_fits': int(combined_stability.failed_fits.sum()),
    'tested_resolutions': len(supported), 'qualified_full_partition_benchmarks': int(supported.status.eq(FULL_SUPPORT).sum()),
    'qualified_full_regions': int(supported.loc[supported.status.eq(FULL_SUPPORT),'region'].nunique()),
    'heldout_responses_used': False, 'final_specification_frozen': False,
    'unchanged_ari_decision': threshold_decision,
    'discussion_evidence': {str(p.relative_to(LOCAL_OUTPUT)): file_sha256(p) for p in sorted(DISCUSSION_OUTPUT.glob('*')) if p.is_file()},
    'reference_fit_implementation_sha256': IMPLEMENTATION_SHA256,
    'selection_code_sha256': selection_manifest['extension_code_sha256'],
    'tables': {name: file_sha256(LOCAL_OUTPUT/name) for name in report_files},
    'figures': {str(p.relative_to(LOCAL_OUTPUT)): file_sha256(p) for p in stability_figures+local_profile_figures},
    'notebook': 'week05_regional_validation_and_objective1_finalization.ipynb',
    'interpretation': 'exploratory development evidence; partition qualification is not uniform profile recovery or optimal k'}
assert report_inventory['stability_fits'] == 10456
assert report_inventory['failed_stability_fits'] == 0
atomic_json(report_inventory,LOCAL_OUTPUT/'development_evidence_inventory.json')
display(report_inventory)
print('Completed analysis saved. Earlier versions preserved. No held-out responses opened; no final model frozen.')

## 17 Historical pre-holdout plan: a development-only decision and freeze package

**Objective:** complete the pre-holdout decisions using existing evidence, not extend the local search indefinitely.

**Output to prepare next:** a reviewable global-model scope decision, frozen model/assignment manifests and development reference diagnostics. This section records the plan; it does not claim that freezing has happened.

The current 7 October protocol and Moodle Week 5 sequence make the next step **select and freeze the global specification(s)**, then validate them on the existing 29 held-out countries. Our regional search and its qualification decision are complete within their bounded scope. We should not automatically add more k searches, lower ARI, introduce another geographic grouping, or restart Prediction Strength from older descriptions.

The proposed development decision is to retain **global B4 and E8 as complementary descriptions**, not unrelated local solutions as replacements: B describes joint worry/harm patterns; E describes more differentiated worry themes and has Ward corroboration. This is a recommendation awaiting approval, not an approved final scope. The user has approved retaining 0.80 only. Regional results must accompany this scope decision, including binary dominant contrasts, country sensitivity, non-universal local recovery and fragile minor profiles. Neither global candidate is a universal regional typology.

Before held-out access, prepare and review:

1. **Model scope and meanings:** specify whether both B and E are retained, their roles, stable cluster IDs, original-response descriptions and limitations. Use existing complete-category distributions and assignment-quality results; do not invent new profile names from held-out responses.
2. **Exact assignment package:** preserve the existing fitted centres; save feature order, raw-code mappings, complete-case/eligibility rules, centring and appended mean, exact square-root multipliers, equal-country weighting, nearest-centre rule, seeds, software versions and fingerprints. Verify reloaded objects reproduce all development labels and join correctly by respondent ID. No refit is needed if the scope and models remain unchanged.
3. **Predeclared transfer interpretation:** freeze the development reference profiles, distance/margin summaries and profile-specific weighted distance p95 thresholds. Define what broad, partial or poor transfer means using the protocol's diagnostics. The p95 is an unusual-assignment flag, not an automatic overall transfer-failure rule; development ARI 0.80 is not a held-out assignment threshold. Resolve any additional numerical rule or uncertainty-reporting choice before opening held-out responses.

Only after approval and saving this package should we open the held-out response data, apply unchanged rules, report retention/exclusions and profile prevalence, and compare profile meanings, distances and margins overall and by country. Never update the global centres to improve held-out results. A partial or negative transfer finding is a valid conclusion, not permission to retune on the holdout.

The optional country-similarity analysis based on fixed-global profile proportions and maps can be pursued separately; it is not needed to establish the current freeze or to conclude Objective 1. No new model-fit or holdout experiment is performed in this documentation update.

**Subsequent approval:** the user explicitly authorized “Freeze B4/E8, then run held-out validation.” Sections 18–24 execute that sequence and report its results. This earlier plan and its pending-approval language are preserved as a decision history.

## 18 Final development decision and a sealed model package

**Objective:** freeze the approved B4/E8 specifications before reading any held-out responses.

**Output:** a sealed model/assignment package with preserved profile IDs, original-response definitions, development reference diagnostics and exact reload checks.

The user approved **freezing B4/E8 and then running held-out validation**. B4 is the joint worry/harm description; E8 is the complementary differentiated worry description, with existing Ward corroboration. We retain both, not a universal regional model or a strict hierarchy. Local k=2 findings remain diagnostics; they do not fulfil the richer local profiling aim. The 0.80 development ARI rule is unchanged and is not a held-out transfer rule.

The existing centres were **not refitted**. Reference seed 20231006 and n_init=20 are recorded from the Week 4 reference code; stability-resampling seeds belong to their separately recorded experiments. The package preserves original raw-code mappings, six worry and fourteen separate harm features, within-person centring and appended mean, exact multipliers \(\sqrt{\alpha/6}\) and \(\sqrt{(1-\alpha)/14}\), complete-case eligibility, equal-country weighting, software versions and fingerprints. B retains top=2, alpha=0.5, k=4; E retains top=3, alpha=0.75, k=8. There is no imputation, PCA or variance standardization.

Nearest-centre assignment uses the stored centre order and first centre on an exact tie; margin is (second−nearest)/second and zero if both distances are zero. Profile-specific weighted development p95 distances are frozen as **unusual-assignment flags**, not automatic transfer failures. Three largest positive original-category-versus-outside contrasts per development profile were fixed as descriptive anchors before held-out access; the full original-category distributions remain the main semantic evidence.

The freeze seal records **09:44:34 UTC (12:44:34 Helsinki)** on 12 October 2026. First held-out response access was **09:46:25 UTC (12:46:25 Helsinki)**. This order is verified from the saved records. The exported assignment code contains no fitting or tuning operation. Reloading the sealed objects reproduces all 109,488 saved development labels exactly. Earlier frozen k=3 artifacts under processed/ or week04_frozen_profile_model.joblib are unrelated legacy analyses, not this B/E validation.

The transfer interpretation and uncertainty plan were saved in the seal before opening responses: inspect profile identity, prevalence, distances, margins and country-specific fit; synthesize broad/partial/poor transfer without an invented numeric score. Use 1,000 fixed-model country-bootstrap draws, seed 20261012, with relative survey weights preserved within sampled complete-case countries. These intervals describe the observed country set, not population-proportional worldwide prevalence, complex survey-design uncertainty or model-fitting uncertainty.

In [ ]:
# Objective: Define portable frozen-model loading, recoding, weighting and assignment; no fitting or tuning.
"""Portable fixed-model assignment only. This module contains no fitting or tuning code."""
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from threadpoolctl import threadpool_limits

WORRY_SOURCES = dict(zip(['food','water','crime','severe_weather','traffic','mental_health'],
                       ['WP20720','WP20721','WP20722','WP20723','WP22213','WP20726']))
HARM_SOURCES = dict(zip(list(WORRY_SOURCES)+['work'],
                      ['WP22442','WP22443','WP22444','WP22445','WP22446','WP22447','WP22448']))

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for part in iter(lambda:handle.read(1024*1024),b''):h.update(part)
    return h.hexdigest()

def load_frozen_package(folder):
    folder=Path(folder)
    seal=json.loads((folder/'freeze_seal.json').read_text())
    assert seal['status']=='frozen_before_heldout_responses'
    for name,value in seal['files'].items():
        if sha256(folder/name)!=value:raise ValueError('Frozen package changed: '+name)
    bundle=joblib.load(folder/'models.joblib')  # Load only this trusted local course artifact.
    manifest=json.loads((folder/'specification.json').read_text())
    assert bundle['context']==manifest['context']==seal['context']
    return bundle,manifest

def derive_frozen_features(prepared):
    """Original raw codes -> original-unit features; missing values stay missing."""
    result=prepared.copy()
    for domain,source in WORRY_SOURCES.items():
        raw=result[source]
        if not raw.dropna().isin([1,2,3,98,99]).all():raise ValueError('Unexpected raw worry code: '+source)
        expected=raw.map({1:2.,2:1.,3:0.,98:np.nan,99:np.nan})
        column='worry_'+domain
        if column in result and not np.allclose(result[column],expected,equal_nan=True):
            raise ValueError('Prepared worry recoding differs: '+column)
        result[column]=expected
    for domain,source in HARM_SOURCES.items():
        raw=result[source]
        if not raw.dropna().isin([1,2,3,4,98,99]).all():raise ValueError('Unexpected raw harm code: '+source)
        valid=raw.isin([1,2,3,4])
        expected=raw.map({1:1.,2:1.,3:1.,4:0.,98:np.nan,99:np.nan})
        column='harm_'+domain
        if column in result and not np.allclose(result[column],expected,equal_nan=True):
            raise ValueError('Prepared combined harm recoding differs: '+column)
        result[column]=expected
        result[column+'_personal']=np.where(valid,raw.isin([1,3]).astype(float),np.nan)
        result[column+'_known_person']=np.where(valid,raw.isin([2,3]).astype(float),np.nan)
    return result

def frozen_transform(frame,model,feature_order):
    values=frame[feature_order].to_numpy(float)
    if not np.isfinite(values).all():raise ValueError('Assignment requires all 20 original-unit features observed.')
    if not np.isin(values[:,:6],[0,1,2]).all() or not np.isin(values[:,6:],[0,1]).all():
        raise ValueError('Unexpected original-unit feature values.')
    spec=model['spec']
    worry=np.where(values[:,:6]==2,spec['top'],values[:,:6])/spec['top']
    mean=worry.mean(axis=1,keepdims=True)
    return np.column_stack([(worry-mean)*np.sqrt(spec['alpha']/6),mean*np.sqrt(spec['alpha']/6),
                            values[:,6:]*np.sqrt((1-spec['alpha'])/14)])

def frozen_country_weights(frame):
    if not len(frame) or not np.isfinite(frame.WGT).all() or not frame.WGT.gt(0).all():
        raise ValueError('Weights require a nonempty cohort with positive finite WGT.')
    weights=frame.WGT/frame.groupby('COUNTRY_ISO3').WGT.transform('sum')/frame.COUNTRY_ISO3.nunique()
    assert np.isclose(weights.sum(),1)
    assert np.allclose(weights.groupby(frame.COUNTRY_ISO3).sum(),1/frame.COUNTRY_ISO3.nunique())
    return weights.to_numpy()

def assign_frozen(frame,bundle):
    """Keep cluster IDs unchanged; use first centre on exact ties; return geometric diagnostics."""
    output={}
    for name,model in bundle['models'].items():
        matrix=frozen_transform(frame,model,bundle['feature_order'])
        with threadpool_limits(limits=1):distances=cdist(matrix,model['centres'],metric='euclidean')
        labels=distances.argmin(axis=1)
        ordered=np.sort(distances,axis=1)
        first,second=ordered[:,0],ordered[:,1]
        margin=np.divide(second-first,second,out=np.zeros_like(second),where=second>0)
        cutoff=np.array([model['profile_distance_p95'][int(c)] for c in labels])
        result=frame[['WPID_RANDOM','COUNTRY_ISO3','GlobalRegion']].copy()
        result['candidate']=name;result['cluster']=labels.astype('int16')
        result['nearest_distance']=first;result['second_nearest_distance']=second
        result['margin']=margin;result['beyond_development_profile_p95']=first>cutoff
        result['equal_country_weight']=frozen_country_weights(frame)
        output[name]=result
    return output


In [ ]:
# Objective: Preserve the complete freeze/transfer implementation for inspection and reproducibility; execution below replays saved results.
"""Freeze existing development models before reading held-out responses; fixed-model transfer only."""
from datetime import datetime, timezone
import shutil

TRANSFER_PLAN = {
    'version':1,'global_scope':['B4','E8'],'roles':{'B':'joint worry/harm description','E':'complementary differentiated worry description'},
    'development_stability_median_min':.80,'development_stability_lower_decile_min':.80,
    'regional_k2':'diagnostic only; not a final rich regional typology',
    'development_model_refit':False,'heldout_model_refit':False,'retune_after_holdout':False,
    'heldout_expected_countries':29,'heldout_prepared_respondents':30090,
    'assignment':'nearest frozen centre; first stored centre on an exact tie',
    'distance_flag':'nearest distance > frozen assigned-profile development weighted p95',
    'distance_flag_is_overall_pass_fail':False,'ari_is_transfer_threshold':False,
    'profile_identity':'all original categories; three largest positive development category-vs-outside contrasts per profile are fixed descriptive anchors',
    'uncertainty':{'draws':1000,'seed':20261012,'unit':'countries, uniformly with replacement',
        'within_country':'preserve relative survey weights among complete cases',
        'model_fixed':True,'interval_percentiles':[2.5,97.5],
        'interpretation':'variation over the observed held-out countries; not worldwide population or fitting uncertainty'},
    'conclusion_rules':{
        'broad':'defining response contrasts remain recognizable without widespread deterioration in representation quality',
        'partial':'recognizable profiles coexist with identifiable profile/country-specific semantic or assignment-quality limitations',
        'poor':'substantial loss of defining patterns or widespread deterioration in representation quality',
        'decision':'analytical synthesis; no automatic numerical transfer score or undeclared cutoff'},
    'heldout_source':'course_artifacts/week03_heldout_country_data.parquet',
    'frozen_split_source':'course_artifacts/week03_country_split.csv',
}

def prepared_columns():
    return ['WPID_RANDOM','COUNTRY_ISO3','GlobalRegion','CountryIncomeLevel','WGT','country_split'] + \
        list(WORRY_SOURCES.values())+list(HARM_SOURCES.values())+WORRY+['harm_'+d for d in HARM_SOURCES]

def eligibility_audit(frame):
    audit=frame[['WPID_RANDOM','COUNTRY_ISO3','GlobalRegion']].copy()
    audit['observed_worry']=frame[WORRY].notna().sum(axis=1)
    audit['observed_harm_domains']=frame[['harm_'+d for d in HARM_SOURCES]].notna().sum(axis=1)
    audit['profile_eligible_any_observed']=audit.observed_worry.add(audit.observed_harm_domains).gt(0)
    audit['previous_4worry_4harm_eligible']=audit.observed_worry.ge(4)&audit.observed_harm_domains.ge(4)
    audit['complete_case']=frame[FEATURES].notna().all(axis=1)
    audit['exclusion_reason']=np.select([~audit.profile_eligible_any_observed,
        audit.profile_eligible_any_observed&~audit.previous_4worry_4harm_eligible,
        audit.previous_4worry_4harm_eligible&~audit.complete_case],
        ['zero observed profile responses','fewer than four observed worry or harm domains','remaining missing profile responses'],
        default='included complete case')
    countries=audit.groupby('COUNTRY_ISO3').agg(prepared=('WPID_RANDOM','size'),
        any_observed=('profile_eligible_any_observed','sum'),previous_4_4=('previous_4worry_4harm_eligible','sum'),
        complete=('complete_case','sum')).reset_index()
    countries['retained_fraction']=countries.complete/countries.prepared
    countries['excluded_zero_observed']=countries.prepared-countries.any_observed
    countries['excluded_4_4_stage']=countries.any_observed-countries.previous_4_4
    countries['excluded_remaining_missing']=countries.previous_4_4-countries.complete
    assert countries[['excluded_zero_observed','excluded_4_4_stage','excluded_remaining_missing']].sum().sum()==(~audit.complete_case).sum()
    return audit,countries

def freeze_existing_models(root):
    root=Path(root)
    data,old_audit,regions,references,provenance=prepare_development(root)
    base_info=json.loads((root/'.week05_build/points12_run.json').read_text())
    base=root/base_info['output_directory']
    base_manifest=json.loads((base/'run_manifest.json').read_text())
    assert base_manifest['provenance']==provenance
    raw=derive_frozen_features(pd.read_parquet(root/'course_artifacts/week03_development_data.parquet',columns=prepared_columns()))
    assert raw.country_split.eq('train').all() and raw.WPID_RANDOM.is_unique
    eligibility,country_audit=eligibility_audit(raw)
    complete=raw.loc[eligibility.complete_case].reset_index(drop=True)
    assert np.array_equal(complete.WPID_RANDOM,data.WPID_RANDOM)
    assert np.array_equal(complete[FEATURES].to_numpy(),data[FEATURES].to_numpy())
    assert len(complete)==109488
    source_names=['fixed_global_original_profiles_by_region.csv','fixed_global_profile_prevalence.csv',
        'fixed_global_assignment_quality.csv','development_distance_thresholds_provisional.csv']
    source_hashes={name:file_sha256(base/name) for name in source_names}
    profile_table=pd.read_csv(base/'fixed_global_original_profiles_by_region.csv')
    development_profiles=profile_table.loc[profile_table.scope.eq('global')].copy()
    prevalence=pd.read_csv(base/'fixed_global_profile_prevalence.csv')
    global_prevalence=prevalence.loc[prevalence.scope.eq('global')]
    thresholds=pd.read_csv(base/'development_distance_thresholds_provisional.csv')
    stability=previous_stability(root)
    assert stability.median_pairwise.ge(.8).all() and stability.p10_pairwise.ge(.8).all()
    core_hash=file_sha256(root/'.week05_build/week05_core.py')
    assignment_hash=file_sha256(root/'.week05_build/frozen_assignment.py')
    implementation_hash=file_sha256(root/'.week05_build/frozen_transfer.py')
    freeze_inputs={'development_provenance':provenance,'development_curated_sha256':file_sha256(root/'course_artifacts/week03_development_data.parquet'),
        'base_context':base_manifest['context'],'source_tables':source_hashes,'transfer_plan':TRANSFER_PLAN,
        'assignment_implementation_sha256':assignment_hash,'transfer_implementation_sha256':implementation_hash,
        'baseline_implementation_sha256':core_hash}
    context=digest_json(freeze_inputs)
    folder=root/'course_artifacts/week05_frozen_objective1_v1'/context[:16]
    if (folder/'freeze_seal.json').exists():
        bundle,manifest=load_frozen_package(folder)
        assert manifest['context']==context
        return folder,bundle,manifest,complete,regions
    folder.mkdir(parents=True,exist_ok=True)
    names=global_prevalence.set_index(['candidate','cluster']).working_description.to_dict()
    bundle={'context':context,'feature_order':FEATURES,'transform_order':TRANSFORM_COLUMNS,'models':{}}
    summary=[];anchors=[]
    for name,spec in SPECIFICATIONS.items():
        model=references[name]
        shares=structure(model['labels'],country_weights(complete))
        assert shares['global_benchmark_pass']
        bundle['models'][name]={'spec':asdict(spec),'centres':np.array(model['centres'],copy=True),
            'profile_names':{int(c):names[(name,c)] for c in range(spec.global_k)},
            'profile_distance_p95':{int(r.cluster):float(r.development_distance_p95) for r in thresholds.loc[thresholds.candidate.eq(name)].itertuples()},
            'reference_seed':20231006,'n_init':20,'source_sha256':model['source_sha256']}
        for cluster in range(spec.global_k):
            group=development_profiles.loc[development_profiles.candidate.eq(name)&development_profiles.cluster.eq(cluster)]
            top=group.sort_values('difference_vs_outside',ascending=False).head(3)
            for row in top.itertuples():
                anchors.append({'candidate':name,'cluster':cluster,'profile_name':names[(name,cluster)],
                    'feature':row.feature,'category':int(row.category),'development_proportion':row.proportion,
                    'development_outside_proportion':row.outside_proportion,'development_contrast':row.difference_vs_outside})
            subset=global_prevalence.loc[global_prevalence.candidate.eq(name)&global_prevalence.cluster.eq(cluster)].iloc[0]
            summary.append({'candidate':name,'cluster':cluster,'profile_name':names[(name,cluster)],
                'development_share':subset.equal_country_weighted_share,'development_respondents':int(subset.respondents),
                'interpretation':'descriptive development response pattern; not a latent natural population type'})
    atomic_joblib(bundle,folder/'models.joblib')
    shutil.copy2(root/'.week05_build/frozen_assignment.py',folder/'assignment.py')
    table_write(pd.DataFrame(summary),folder/'profile_definitions.csv')
    table_write(pd.DataFrame(anchors),folder/'development_defining_anchors.csv')
    table_write(development_profiles,folder/'development_original_category_profiles.csv')
    table_write(global_prevalence,folder/'development_global_prevalence.csv')
    table_write(pd.read_csv(base/'fixed_global_assignment_quality.csv'),folder/'development_assignment_quality.csv')
    table_write(thresholds,folder/'development_profile_distance_p95.csv')
    table_write(stability,folder/'development_stability.csv')
    table_write(country_audit,folder/'development_country_retention.csv')
    assigned=assign_frozen(complete,bundle)
    for name in SPECIFICATIONS:
        assert np.array_equal(assigned[name].cluster,references[name]['labels'])
        assigned[name]['profile_name']=assigned[name].cluster.map(bundle['models'][name]['profile_names'])
        assigned[name].to_parquet(folder/f'development_{name}_assignments.parquet',index=False)
    wide=eligibility.copy()
    for name in SPECIFICATIONS:
        wide=wide.merge(assigned[name][['WPID_RANDOM','cluster','nearest_distance','margin']].rename(columns={
            'cluster':name+'_cluster','nearest_distance':name+'_nearest_distance','margin':name+'_margin'}),
            on='WPID_RANDOM',how='left',validate='one_to_one')
        assert wide.loc[wide.complete_case,name+'_cluster'].notna().all()
        assert wide.loc[~wide.complete_case,name+'_cluster'].isna().all()
    wide.to_parquet(folder/'development_eligibility_and_assignments_all_respondents.parquet',index=False)
    table_write(pd.read_csv(root/'course_artifacts/week03_country_split.csv'),folder/'frozen_country_split.csv')
    for name in SPECIFICATIONS:
        for r in pd.DataFrame(bundle['models'][name]['centres']).itertuples(index=False,name=None):assert len(r)==21
    manifest={'context':context,'status':'frozen_before_heldout_responses','frozen_at_utc':datetime.now(timezone.utc).isoformat(),
        'approved_scope':['B4','E8'],'approval':'explicit user instruction: Freeze B4/E8, then run held-out validation',
        'data_population':'2023 complete-case respondents in development countries; missing responses are not imputed',
        'eligibility':'all six worry and seven original harm-domain responses substantive; equivalently all 20 derived features observed',
        'prior_screen':'at least four observed worry and four harm domains is reported; complete-case rule is stricter and determines assignment',
        'raw_codes':{'worry':{'1':2,'2':1,'3':0,'98':None,'99':None},
            'harm_personal':{'1':1,'2':0,'3':1,'4':0,'98':None,'99':None},
            'harm_known_person':{'1':0,'2':1,'3':1,'4':0,'98':None,'99':None}},
        'raw_sources':{'worry':WORRY_SOURCES,'harm':HARM_SOURCES},'feature_order':FEATURES,'transform_order':TRANSFORM_COLUMNS,
        'transformation':'recode top category, divide by top, subtract respondent mean in six worry coordinates, append mean; multiply both by sqrt(alpha/6); harm by sqrt((1-alpha)/14)',
        'imputation':None,'variance_standardization':False,'PCA':False,
        'weights':'relative WGT within included complete-case country; equal total influence per included country, separately within each split',
        'development_model_refit':False,'profile_ids':'preserved exactly from existing reference caches; not relabelled by prevalence',
        'regional_limitations':'local structure not universal; k2 only diagnostic; several smaller E profiles have weaker recovery; global/local ARI is not a retention or significance test',
        'global_stability_caution':'B has stronger whole-partition and reference-profile recovery; E meets ARI benchmarks but pooled cluster Jaccard lower tails are weak and must be reported',
        'transfer_plan':TRANSFER_PLAN,'freeze_inputs':freeze_inputs,'development_reload_verified':True,
        'development_country_count':113,'development_prepared_count':116820,'development_assigned_count':109488,
        'heldout_responses_used_at_freeze':False,'other_old_frozen_artifacts':'older course_artifacts/week04_frozen_profile_model.joblib and processed/*heldout*k3 are not this B/E analysis'}
    atomic_json(manifest,folder/'specification.json')
    files={p.name:file_sha256(p) for p in sorted(folder.iterdir()) if p.is_file()}
    atomic_json({'context':context,'status':'frozen_before_heldout_responses','frozen_at_utc':manifest['frozen_at_utc'],
        'files':files,'development_replay_verified':True,'heldout_responses_used':False},folder/'freeze_seal.json')
    reloaded,verified=load_frozen_package(folder)
    replay=assign_frozen(complete,reloaded)
    for name in SPECIFICATIONS:assert np.array_equal(replay[name].cluster,assigned[name].cluster)
    atomic_json({'package_directory':str(folder.relative_to(root)),'context':context},root/'.week05_build/freeze_run.json')
    return folder,reloaded,verified,complete,regions

def transfer_tables(frame,assigned,bundle,region_names,output_dir,split_name):
    prevalence=[];profiles=[];quality=[]
    for name,model in bundle['models'].items():
        labels=assigned[name].cluster.to_numpy()
        distance=assigned[name].nearest_distance.to_numpy();margin=assigned[name].margin.to_numpy()
        exceed=assigned[name].beyond_development_profile_p95.to_numpy()
        k=model['spec']['global_k']
        scopes=[('global',split_name,split_name,np.ones(len(frame),bool))]
        scopes += [('region',str(int(r)),region_names[r],frame.GlobalRegion.eq(r).to_numpy()) for r in sorted(frame.GlobalRegion.unique())]
        scopes += [('country',c,c,frame.COUNTRY_ISO3.eq(c).to_numpy()) for c in sorted(frame.COUNTRY_ISO3.unique())]
        for scope,key,area,mask in scopes:
            local=frame.loc[mask];ll=labels[mask];weights=frozen_country_weights(local)
            for cluster in range(k):
                inside=ll==cluster
                prevalence.append({'split':split_name,'candidate':name,'scope':scope,'area_id':key,'area_name':area,
                    'cluster':cluster,'profile_name':model['profile_names'][cluster],'respondents':int(inside.sum()),
                    'area_respondents':int(mask.sum()),'area_countries':int(local.COUNTRY_ISO3.nunique()),
                    'respondent_share':float(inside.mean()),'equal_country_weighted_share':float(weights[inside].sum())})
            pp=original_profiles(local,ll,weights)
            profiles.append(pp.assign(split=split_name,candidate=name,scope=scope,area_id=key,area_name=area))
            for cluster in [-1]+list(range(k)):
                inside=np.ones(len(local),bool) if cluster==-1 else ll==cluster
                if not inside.any():continue
                ww=weights[inside]
                quality.append({'split':split_name,'candidate':name,'scope':scope,'area_id':key,'area_name':area,
                    'cluster':cluster,'respondents':int(inside.sum()),
                    'kish_weighted_n':float(ww.sum()**2/(ww**2).sum()),
                    'distance_median':weighted_quantile(distance[mask][inside],ww,.5),
                    'distance_p90':weighted_quantile(distance[mask][inside],ww,.9),
                    'margin_median':weighted_quantile(margin[mask][inside],ww,.5),
                    'margin_p10':weighted_quantile(margin[mask][inside],ww,.1),
                    'profile_p95_exceedance':float(np.average(exceed[mask][inside],weights=ww))})
    prevalence=pd.DataFrame(prevalence);profiles=pd.concat(profiles,ignore_index=True);quality=pd.DataFrame(quality)
    for _,g in prevalence.groupby(['candidate','scope','area_id']):
        assert np.isclose(g.equal_country_weighted_share.sum(),1) and np.isclose(g.respondent_share.sum(),1)
    table_write(prevalence,Path(output_dir)/f'{split_name}_prevalence.csv')
    table_write(profiles,Path(output_dir)/f'{split_name}_original_category_profiles.csv')
    table_write(quality,Path(output_dir)/f'{split_name}_assignment_quality.csv')
    return prevalence,profiles,quality

def country_bootstrap_profiles(frame,assigned,bundle,output_dir):
    """Conditional fixed-model intervals; bootstrap country masses, not respondent/model fits."""
    plan=TRANSFER_PLAN['uncertainty'];countries=sorted(frame.COUNTRY_ISO3.unique());n=len(countries)
    rng=np.random.default_rng(plan['seed'])
    counts=rng.multinomial(n,np.ones(n)/n,size=plan['draws'])
    country_index={c:i for i,c in enumerate(countries)}
    ci=frame.COUNTRY_ISO3.map(country_index).to_numpy()
    within=(frame.WGT/frame.groupby('COUNTRY_ISO3').WGT.transform('sum')).to_numpy()
    rows=[]
    for name,model in bundle['models'].items():
        labels=assigned[name].cluster.to_numpy();k=model['spec']['global_k']
        mass=np.zeros((n,k));np.add.at(mass,(ci,labels),within)
        draws=counts@mass/n
        for cluster in range(k):
            values=draws[:,cluster]
            rows.append({'candidate':name,'cluster':cluster,'quantity':'profile_share','feature':'','category':-1,
                'estimate':float(mass[:,cluster].mean()),'p025':float(np.quantile(values,.025)),
                'p975':float(np.quantile(values,.975)),'valid_draws':len(values)})
        for feature in FEATURES:
            for category in ([0,1,2] if feature in WORRY else [0,1]):
                numerator=np.zeros((n,k));mask=frame[feature].to_numpy()==category
                np.add.at(numerator,(ci[mask],labels[mask]),within[mask])
                weighted=counts@numerator/n
                conditional=np.divide(weighted,draws,out=np.full_like(weighted,np.nan),where=draws>0)
                for cluster in range(k):
                    values=conditional[:,cluster];values=values[np.isfinite(values)]
                    total=mass[:,cluster].sum()
                    rows.append({'candidate':name,'cluster':cluster,'quantity':'original_category_probability',
                        'feature':feature,'category':category,
                        'estimate':float(numerator[:,cluster].sum()/total) if total else np.nan,
                        'p025':float(np.quantile(values,.025)) if len(values) else np.nan,
                        'p975':float(np.quantile(values,.975)) if len(values) else np.nan,'valid_draws':len(values)})
    result=pd.DataFrame(rows)
    table_write(result,Path(output_dir)/'heldout_country_bootstrap_intervals.csv')
    atomic_json({**plan,'countries':countries,'draw_count':len(counts),'bootstrap_counts_sha256':hashlib.sha256(counts.tobytes()).hexdigest()},
                Path(output_dir)/'bootstrap_design.json')
    return result

def run_heldout_transfer(root,folder,bundle,manifest,development,regions):
    root,folder=Path(root),Path(folder)
    # A valid pre-response freeze seal is a hard execution boundary.
    bundle,manifest=load_frozen_package(folder)
    assert manifest['heldout_responses_used_at_freeze'] is False
    output=root/'course_artifacts/week05_heldout_transfer_v1'/manifest['context'][:16]
    output.mkdir(parents=True,exist_ok=True)
    raw=pd.read_parquet(root/TRANSFER_PLAN['heldout_source'],columns=prepared_columns())
    first_read=datetime.now(timezone.utc).isoformat()
    assert first_read>manifest['frozen_at_utc']
    split=pd.read_csv(folder/'frozen_country_split.csv')
    expected=set(split.loc[split.country_split.eq('heldout_country'),'COUNTRY_ISO3'])
    development_ids=pd.read_parquet(root/'course_artifacts/week03_development_data.parquet',columns=['WPID_RANDOM']).WPID_RANDOM
    assert len(raw)==30090 and raw.COUNTRY_ISO3.nunique()==29 and set(raw.COUNTRY_ISO3)==expected
    assert raw.country_split.eq('heldout_country').all() and raw.WPID_RANDOM.is_unique
    assert not set(raw.WPID_RANDOM).intersection(development_ids)
    assert not set(raw.COUNTRY_ISO3).intersection(development.COUNTRY_ISO3)
    frame=derive_frozen_features(raw)
    audit,country_audit=eligibility_audit(frame)
    complete=frame.loc[audit.complete_case].reset_index(drop=True)
    zero=sorted(expected-set(complete.COUNTRY_ISO3))
    assert np.isfinite(complete.WGT).all() and complete.WGT.gt(0).all()
    region_names=regions.set_index('GlobalRegion').region_name.to_dict()
    # The split registry, not response patterns, determines country/region identities.
    lookup=split.set_index('COUNTRY_ISO3')
    assert (complete.GlobalRegion.to_numpy()==complete.COUNTRY_ISO3.map(lookup.GlobalRegion).to_numpy()).all()
    assignments=assign_frozen(complete,bundle)
    for name,table in assignments.items():
        table['profile_name']=table.cluster.map(bundle['models'][name]['profile_names'])
        table.to_parquet(output/f'heldout_{name}_assignments.parquet',index=False)
        audit=audit.merge(table[['WPID_RANDOM','cluster','nearest_distance','margin']].rename(columns={
            'cluster':name+'_cluster','nearest_distance':name+'_nearest_distance','margin':name+'_margin'}),
            on='WPID_RANDOM',how='left',validate='one_to_one')
        assert audit.loc[audit.complete_case,name+'_cluster'].notna().all()
        assert audit.loc[~audit.complete_case,name+'_cluster'].isna().all()
    audit.to_parquet(output/'heldout_eligibility_and_assignments_all_respondents.parquet',index=False)
    country_audit=country_audit.merge(split[['COUNTRY_ISO3','GlobalRegion','CountryIncomeLevel']],on='COUNTRY_ISO3',validate='one_to_one')
    table_write(country_audit,output/'heldout_country_retention.csv')
    table_write(audit.groupby('exclusion_reason').size().rename('respondents').reset_index(),output/'heldout_exclusion_counts.csv')
    prevalence,profiles,quality=transfer_tables(complete,assignments,bundle,region_names,output,'heldout')
    devprofiles=pd.read_csv(folder/'development_original_category_profiles.csv')
    devquality=pd.read_csv(folder/'development_assignment_quality.csv')
    devprev=pd.read_csv(folder/'development_global_prevalence.csv')
    differences=profiles.merge(devprofiles[['candidate','cluster','feature','category','proportion','difference_vs_outside']],
        on=['candidate','cluster','feature','category'],suffixes=('_heldout','_development'),validate='many_to_one')
    differences['difference_pp']=100*(differences.proportion_heldout-differences.proportion_development)
    table_write(differences,output/'heldout_vs_development_category_differences.csv')
    tv=differences.assign(absdiff=lambda g:abs(g.proportion_heldout-g.proportion_development)).groupby(
        ['candidate','scope','area_id','area_name','cluster','feature']).absdiff.sum().div(2).rename('feature_TV').reset_index()
    semantic=tv.groupby(['candidate','scope','area_id','area_name','cluster']).agg(mean_feature_TV=('feature_TV','mean'),
        maximum_feature_TV=('feature_TV','max')).reset_index()
    table_write(tv,output/'heldout_original_feature_TV.csv');table_write(semantic,output/'heldout_profile_semantic_TV.csv')
    anchors=pd.read_csv(folder/'development_defining_anchors.csv').merge(
        profiles.loc[profiles.scope.eq('global'),['candidate','cluster','feature','category','proportion','outside_proportion','difference_vs_outside']],
        on=['candidate','cluster','feature','category'],validate='one_to_one')
    anchors['heldout_contrast_direction_preserved']=anchors.difference_vs_outside.gt(0)
    anchors['proportion_shift_pp']=100*(anchors.proportion-anchors.development_proportion)
    table_write(anchors,output/'heldout_frozen_anchor_comparison.csv')
    comparison=quality.loc[quality.scope.eq('global')].merge(devquality.loc[devquality.scope.eq('global')].drop(columns=['scope','area_id','area_name']),
        on=['candidate','cluster'],suffixes=('_heldout','_development'),validate='one_to_one')
    comparison['distance_median_ratio']=comparison.distance_median_heldout/comparison.distance_median_development
    comparison['distance_p90_ratio']=comparison.distance_p90_heldout/comparison.distance_p90_development
    comparison['margin_median_difference']=comparison.margin_median_heldout-comparison.margin_median_development
    table_write(comparison,output/'heldout_vs_development_quality.csv')
    shares=prevalence.loc[prevalence.scope.eq('global')].merge(devprev[['candidate','cluster','equal_country_weighted_share']],
        on=['candidate','cluster'],suffixes=('_heldout','_development'),validate='one_to_one')
    shares['prevalence_difference_pp']=100*(shares.equal_country_weighted_share_heldout-shares.equal_country_weighted_share_development)
    table_write(shares,output/'heldout_vs_development_prevalence.csv')
    intervals=country_bootstrap_profiles(complete,assignments,bundle,output)
    join_checks={}
    for name in SPECIFICATIONS:
        saved=pd.read_parquet(output/f'heldout_{name}_assignments.parquet')
        check=complete[['WPID_RANDOM']].merge(saved,on='WPID_RANDOM',validate='one_to_one')
        assert len(check)==len(complete) and check.cluster.notna().all()
        replay=assign_frozen(complete,bundle)[name]
        assert np.array_equal(saved.cluster,replay.cluster)
        join_checks[name]={'unique_ids':True,'reload_assignment_identical':True,'all_prepared_ids_preserved_in_eligibility_file':len(audit)==len(raw)}
    load_frozen_package(folder)  # Verify the sealed development package did not change during transfer.
    atomic_json({'status':'completed_fixed_model_transfer','freeze_context':manifest['context'],
        'freeze_seal_sha256':file_sha256(folder/'freeze_seal.json'),'first_heldout_response_read_at_utc':first_read,
        'frozen_at_utc':manifest['frozen_at_utc'],'heldout_input_sha256':file_sha256(root/TRANSFER_PLAN['heldout_source']),
        'prepared_respondents':len(raw),'complete_respondents':len(complete),'excluded_respondents':len(raw)-len(complete),
        'prepared_countries':29,'assigned_countries':int(complete.COUNTRY_ISO3.nunique()),'zero_eligible_countries':zero,
        'model_refit':False,'retuning':False,'all_assignment_joins_verified':join_checks,
        'interpretation':'descriptive transfer, not ARI threshold testing or independent recovery by refitting',
        'tables':{p.name:file_sha256(p) for p in sorted(output.glob('*.csv'))},
        'assignments':{p.name:file_sha256(p) for p in sorted(output.glob('*.parquet'))}},output/'transfer_complete.json')
    atomic_json({'output_directory':str(output.relative_to(root)),'package_directory':str(folder.relative_to(root)),
        'context':manifest['context']},root/'.week05_build/transfer_run.json')
    return output,complete,assignments


In [ ]:
# Objective: Verify the sealed package and immutable transfer record before replaying their evidence.
FROZEN_PACKAGE = ROOT / 'course_artifacts/week05_frozen_objective1_v1/123d248adda6115f'
TRANSFER_OUTPUT = ROOT / 'course_artifacts/week05_heldout_transfer_v1/123d248adda6115f'
HANDOFF_OUTPUT = ROOT / 'course_artifacts/week05_objective1_handoff_v1/123d248adda6115f'
EXPECTED_FREEZE_CONTEXT = '123d248adda6115fc1fa6f7a6779fc6c30696a5e6e3c46399d8abf84a72e94c4'
frozen_bundle, frozen_manifest = load_frozen_package(FROZEN_PACKAGE)
transfer_record = json.loads((TRANSFER_OUTPUT/'transfer_complete.json').read_text())
assert frozen_manifest['context']==transfer_record['freeze_context']==EXPECTED_FREEZE_CONTEXT
assert frozen_manifest['heldout_responses_used_at_freeze'] is False
assert transfer_record['first_heldout_response_read_at_utc'] > frozen_manifest['frozen_at_utc']
assert transfer_record['freeze_seal_sha256']==file_sha256(FROZEN_PACKAGE/'freeze_seal.json')
assert frozen_manifest['transfer_plan']==TRANSFER_PLAN
assert not transfer_record['model_refit'] and not transfer_record['retuning']
assert file_sha256(ROOT/TRANSFER_PLAN['heldout_source'])==transfer_record['heldout_input_sha256']
for key in ['tables','assignments']:
    for name,expected in transfer_record[key].items():
        assert file_sha256(TRANSFER_OUTPUT/name)==expected,'Transfer evidence changed: '+name
development_replay=assign_frozen(data,frozen_bundle)
for name in SPECIFICATIONS:
    assert np.array_equal(development_replay[name].cluster,global_references[name]['labels'])
display(pd.read_csv(FROZEN_PACKAGE/'profile_definitions.csv'))
display(pd.read_csv(FROZEN_PACKAGE/'development_stability.csv')[
    ['candidate','experiment','median_pairwise','p10_pairwise','p10_cluster_jaccard']])
print('Sealed scope:',frozen_manifest['approved_scope'])
print('Frozen before held-out access:',frozen_manifest['frozen_at_utc'], '<', transfer_record['first_heldout_response_read_at_utc'])
print('All development assignments reproduce exactly; no refit.')

## 19 Held-out integrity, missingness and assignment coverage

**Objective:** apply the frozen rules to the existing 29-country split and retain excluded respondents explicitly.

**Output:** verified country/respondent separation, sequential exclusions, country retention, assignment replay and a demographic selection audit.

The prepared holdout has **30,090 respondents**. Of these, **27,972 (92.96%)** are complete cases and receive both B/E labels. All **29 countries** retain eligible respondents. The 2,118 excluded respondents comprise one with zero observed profile responses, 202 removed at the historical four-worry/four-harm screening stage, and 1,915 with remaining missing profile responses. The ultimate rule is all 20 derived features observed, equivalently substantive answers to all six worry and seven original harm questions. The historical four/four screen is weaker than this final rule. Excluded respondents are not assigned through imputation.

Conditional assigned coverage is 100% **of complete cases**, not of all prepared respondents. Development retention was 93.72% by respondent count versus 92.96% in holdout; country-balanced mean retention is 93.57% versus 92.83%. These are different estimands and are labelled separately.

Country retention varies from **78.7% to 99.8%**. Sierra Leone retains 78.7%, Laos 80.5%, Moldova 80.6%, Tajikistan 81.2% and Somalia 82.9%. The complete-case profiles therefore describe a selected respondent population. The demographic audit compares prepared, included and excluded cohorts using available age, gender, education and urbanicity variables. Each cohort's equal-country weights are recomputed; the excluded development cohort includes 112 rather than 113 countries. Differences are descriptive, not proof that missingness is ignorable or a causal selection mechanism.

Frozen raw-code checks are replayed on holdout: worry raw 1/2/3 becomes 2/1/0; harm raw 1/2/3/4 becomes personal (1/0/1/0) and known-person (0/1/1/0); 98/99 remain missing. No encoding or weighting is learned anew from responses. Survey-weight normalization is recomputed within the assigned held-out cohort, as specified before access.

For the demographic diagnostic tables, the harmonized dictionary defines Gender 1=male, 2=female; Education 1=primary or less, 2=secondary, 3=tertiary, 9=DK/refused; Urbanicity2 1=rural/small town, 2=city/suburb, 9=DK/refused; AgeGroups3 1=15–29, 2=30–49, 3=50+. DK/refused is shown separately, not interpreted as substantive education or urbanicity. Source Age=100 was already recoded to missing in the curated data. These variables are selection diagnostics, not inputs to B/E.

**Why 27,972 rather than 30,090?** We require complete responses to six worry and seven original harm questions. The seven harm answers generate fourteen personal/known-person indicators, so 13 original questions produce 20 features. A missing harm answer makes both corresponding indicators missing. The frozen complete-case rule—not a failed fit—leaves 2,118 respondents unassigned. The one/202/1,915 counts are mutually exclusive bookkeeping stages; all three groups ultimately lack complete profile information. These exclusions are not distance flags: atypically distant complete-case respondents still receive their nearest profile. The model does not assign incomplete cases by ignoring missing coordinates or introducing a new imputer at validation time.

In [ ]:
# Objective: Replay held-out recoding and assignment, verify disjoint IDs/countries and display all exclusions.
holdout_raw=pd.read_parquet(ROOT/TRANSFER_PLAN['heldout_source'],columns=prepared_columns())
holdout_features=derive_frozen_features(holdout_raw)
holdout_audit,holdout_country_audit=eligibility_audit(holdout_features)
heldout_complete=holdout_features.loc[holdout_audit.complete_case].reset_index(drop=True)
frozen_split=pd.read_csv(FROZEN_PACKAGE/'frozen_country_split.csv')
expected_holdout=set(frozen_split.loc[frozen_split.country_split.eq('heldout_country'),'COUNTRY_ISO3'])
assert len(holdout_raw)==30090 and holdout_raw.WPID_RANDOM.is_unique
assert set(holdout_raw.COUNTRY_ISO3)==expected_holdout and len(expected_holdout)==29
assert not set(holdout_raw.COUNTRY_ISO3).intersection(data.COUNTRY_ISO3)
development_all_ids=pd.read_parquet(ROOT/'course_artifacts/week03_development_data.parquet',columns=['WPID_RANDOM']).WPID_RANDOM
assert not set(holdout_raw.WPID_RANDOM).intersection(development_all_ids)
heldout_replay=assign_frozen(heldout_complete,frozen_bundle)
for name in SPECIFICATIONS:
    saved=pd.read_parquet(TRANSFER_OUTPUT/f'heldout_{name}_assignments.parquet')
    assert np.array_equal(saved.WPID_RANDOM,heldout_replay[name].WPID_RANDOM)
    assert np.array_equal(saved.cluster,heldout_replay[name].cluster)
    assert np.allclose(saved.nearest_distance,heldout_replay[name].nearest_distance)
assert len(heldout_complete)==27972 and heldout_complete.COUNTRY_ISO3.nunique()==29
display(pd.read_csv(TRANSFER_OUTPUT/'heldout_exclusion_counts.csv'))
display(pd.read_csv(TRANSFER_OUTPUT/'heldout_country_retention.csv').sort_values('retained_fraction'))
selection_audit=pd.read_csv(TRANSFER_OUTPUT/'missingness_selection_demographic_audit.csv')
display(selection_audit)
print('100% assignment of 27,972 complete cases; 2,118 prepared respondents explicitly remain unassigned.')

## 20 Transfer of prevalence, profile meaning and representation quality

**Objective:** compare fixed-global development and held-out profiles without treating prevalence equality as a success condition.

**Output:** complete original-category comparisons, frozen-anchor contrasts, all-profile quality summaries and fixed-model country-bootstrap intervals.

The largest absolute change in profile prevalence is **1.45 percentage points for B** and **1.55 for E**. B's broad-harm group is 18.53% in development and 17.16% in holdout; its lower-worry/harm group is 33.77% and 34.34%. E's lower-worry group is 36.11% and 34.56%, and broad-high-worry group 10.37% and 10.69%. Frequencies are close at the aggregate, but matching prevalence is neither required nor sufficient for transfer, and the intervals do not establish statistical equivalence.

All **36 preselected development anchor contrasts retain their positive direction** in aggregate holdout profiles. Their meanings remain recognizable: B's broad exposure is especially through known people; the other B profiles distinguish food/water concern, crime/weather/traffic concern and lower worry/harm. E's domain-specific worry themes remain identifiable. Mean feature-wise categorical TV against development ranges **0.008–0.025 for B** and **0.011–0.030 for E**. These are average marginal shifts, not a test of the complete joint distribution or a declared semantic threshold.

Assignment quality is similar overall. B's median distance increases by 0.4% and p90 by 1.5%; E's corresponding increases are 1.0% and 0.6%. Median margins are B 0.134 versus development 0.137 and E 0.273 versus 0.273. Distances and margins are geometric diagnostics, not posterior probabilities, and B/E distances are not compared across their different geometries.

Beyond the frozen profile-specific development p95, **6.34%** of held-out B assignments are flagged versus **4.99%** in development; E is **5.44% versus 4.98%**. B's food/water C2 has the clearest profile-level rise, to **8.18%** flagged, and p90 distance approximately 4.8% higher. This is a warning about representation, not a post-hoc automatic failure.

Some very sharp E anchors follow partly from the frozen assignment geometry: a rule that selects a very-worried-traffic group will predictably preserve that defining category when applied to another cohort. The more informative evidence is the full remaining worry/harm distributions, non-defining contrasts, distance/margin behaviour and country-specific results. We validate the transfer of a **fixed descriptive classification**, not independent rediscovery of latent types by reclustering the holdout. No holdout refit was performed.

In [ ]:
# Objective: Display complete held-out/development comparisons and their fixed-model uncertainty summaries.
prevalence_transfer=pd.read_csv(TRANSFER_OUTPUT/'prevalence_comparison_with_country_intervals.csv')
quality_transfer=pd.read_csv(TRANSFER_OUTPUT/'heldout_vs_development_quality.csv')
semantic_transfer=pd.read_csv(TRANSFER_OUTPUT/'semantic_differences_with_sample_sizes.csv')
anchor_transfer=pd.read_csv(TRANSFER_OUTPUT/'heldout_frozen_anchor_comparison.csv')
bootstrap_intervals=pd.read_csv(TRANSFER_OUTPUT/'heldout_country_bootstrap_intervals.csv')
display(prevalence_transfer[['candidate','cluster','profile_name','equal_country_weighted_share_development',
    'equal_country_weighted_share_heldout','prevalence_difference_pp','p025','p975']].round(4))
display(quality_transfer[['candidate','cluster','distance_median_ratio','distance_p90_ratio',
    'margin_median_heldout','margin_median_development','profile_p95_exceedance_heldout','profile_p95_exceedance_development']].round(4))
display(semantic_transfer.loc[semantic_transfer.scope.eq('global')])
display(anchor_transfer[['candidate','cluster','feature','category','development_proportion','proportion',
    'development_contrast','difference_vs_outside','heldout_contrast_direction_preserved']])
assert anchor_transfer.heldout_contrast_direction_preserved.all() and len(anchor_transfer)==36
assert bootstrap_intervals.valid_draws.eq(1000).all()
display(bootstrap_intervals)
print('Intervals: 1,000 fixed-model country draws; no fit uncertainty, worldwide population inference or equivalence claim.')

## 20a Discussion clarification: what a distance flag means

**Objective:** distinguish atypical representation from failed assignment, exclusion or classification error.

**Output:** a self-contained definition of the frozen distance diagnostic and its limits.

For each assigned development profile c, measure the Euclidean distance to its stored centre in that model's **scaled 21-coordinate space**. Freeze a separate equal-country/survey-weighted 95th-percentile distance \(t_c\) for each profile. A held-out respondent i is flagged if \(d(x_i,\mu_{c_i})>t_{c_i}\). Using a separate threshold accommodates profiles with different dispersion. The cutoff is learned only from development; it is not recalibrated to make the held-out rate 5%.

Approximately 5% of development assignments exceed their cutoffs by construction; discrete distances and percentile ties can make the actual rate slightly below 5%. A higher held-out rate means more respondents are atypically far from their nearest available frozen profile compared with development members of that same profile. It does not by itself say why: differences in response combinations, country composition, profile representation or selection may contribute.

The flag **does not remove the respondent or change their label**, prove the assignment wrong, establish a new cluster, or identify a statistical outlier under a fitted probability distribution. All complete cases receive their nearest profile, flagged or not. It is not an assignment probability or accuracy measure. For example, Sierra Leone's 25.3% B flag rate is a weighted fraction of its assigned complete-case respondents, not a statement that 25.3% were misclassified. It is unrelated to the 2,118 holdout respondents excluded for missing profile responses.

## 21 Country-specific limitations and smaller-profile reliability

**Objective:** prevent reassuring global averages from hiding geographic weaknesses, sparse cells or uncertain profile boundaries.

**Output:** all-country retention/quality tables, country/profile semantic shifts with sample sizes, anchor reversals and the existing per-profile development recovery record.

The most conspicuous distance flags occur in **Sierra Leone, Somalia and Namibia**:

| Held-out country | B beyond frozen p95 | E beyond frozen p95 | Prepared respondents retained |
|---|---:|---:|---:|
| Sierra Leone | 25.3% | 14.8% | 78.7% |
| Somalia | 24.0% | 12.5% | 82.9% |
| Namibia | 17.9% | 12.4% | 94.4% |

The fixed models represent these countries less well than the aggregate. Sierra Leone and Somalia also have appreciable complete-case selection, but Namibia's high retention shows that missingness alone is not an adequate explanation. None of these descriptive associations identifies a causal mechanism. The p95 flags are not a country pass/fail threshold or evidence that every flagged respondent needs a new profile.

Country-level profile meanings vary more than pooled meanings. Mean feature-wise TV reaches 0.264 for Somalia E C7, involving 104 respondents, and approximately 0.264 for Sweden E C7, involving **only three**. Sweden E C3 also has only three respondents; its large semantic shift is highly uncertain and is not evidence of broad national transfer failure. Saved tables include raw n and Kish weighted n so these cases are not interpreted alike. Across country-specific frozen anchors, all 348 B contrasts remain positive; E preserves 687 of 696, with nine reversals involving secondary features and, in some cases, sparse profiles. These counts are descriptive and partly reflect the assignment geometry, not independent hypothesis tests.

Development individual-profile recovery also matters after a model is frozen. Existing Week 4 diagnostics show B's country-test per-profile Jaccard lower deciles around **0.91–0.96**. E is less uniform: its crime/traffic C3 lower decile is about **0.088** in both respondent and country perturbations; crime-focused C4 is about **0.431–0.433**. Whole-partition ARI still passes the unchanged rule. The sealed classification can transfer descriptively while these specific eight-profile distinctions remain uncertain under re-estimation. Good fixed-model transfer does **not** erase that development limitation. E remains a complementary, more tentative thematic resolution, not eight equally established natural types.

No model, profile ID or threshold was modified in response to these findings. B/E and regional results are interpreted together rather than selecting only the most favourable statistic.

In [ ]:
# Objective: Inspect all country limitations and the already-computed individual-profile recovery evidence.
country_transfer=pd.read_csv(TRANSFER_OUTPUT/'heldout_country_quality_and_retention.csv')
country_anchor_transfer=pd.read_csv(TRANSFER_OUTPUT/'country_frozen_anchor_diagnostics.csv')
development_profile_recovery=pd.read_csv(TRANSFER_OUTPUT/'development_existing_per_profile_recovery.csv')
display(country_transfer[['Country','candidate','respondents','retained_fraction','distance_median','distance_p90',
    'margin_median','profile_p95_exceedance']].sort_values('profile_p95_exceedance',ascending=False))
display(semantic_transfer.loc[semantic_transfer.scope.eq('country')].sort_values('mean_feature_TV',ascending=False).head(30))
display(country_anchor_transfer.loc[~country_anchor_transfer.heldout_country_contrast_positive,
    ['candidate','area_id','cluster','feature','category','respondents','kish_weighted_n','difference_vs_outside_heldout']])
display(development_profile_recovery)
print('Sparse-cell differences and weak development recovery remain qualifications; no post-hoc rejection threshold added.')

## 21a Discussion clarification: how good is the held-out result?

**Objective:** distinguish favourable aggregate transfer from uniform country quality, forced label assignment and latent-type discovery.

**Output:** an analytical synthesis and a reproducible explanatory concentration diagnostic.

The evidence supports **useful fixed descriptive classifications that transfer broadly in aggregate, with localized weaknesses**. All 36 frozen aggregate anchor contrasts remain positive, the full marginal category distributions change little, nearest-centre distances and margins change modestly, and profile prevalence differs by at most about 1.6 percentage points. Frequency similarity alone is not sufficient evidence: the country split balanced geographic/income metadata, and defining E categories can also be partly preserved by assignment geometry.

The distance deterioration is not spread evenly across countries. The median held-out country has a flag rate of **4.46% for B** and **4.51% for E**, despite overall country-balanced rates of 6.34% and 5.44%. Sierra Leone, Somalia and Namibia are the three highest-flag countries for both models. The remaining 26 countries average **4.49% for B** and **4.53% for E**. This is an **explanatory calculation performed after inspection**, not a new acceptance criterion: the three countries remain in every official validation result and are not removed to improve the summary. The complete country table and the concentrated failures remain part of the scientific conclusion.

There is no independent ground-truth profile label, so we cannot report classification accuracy. Receiving a mandatory nearest-centre label is not evidence of a good match, and 92.96% prepared-sample coverage is not 92.96% accuracy. Nor does this validation independently rediscover clusters through holdout refitting. It evaluates application of the frozen classification to new countries.

The scope is complete-case respondents; selective missingness remains a limitation. Rare country/profile cells require caution. B is the stronger core result because its individual development profiles recover more consistently; E adds useful thematic detail, but some boundaries remain tentative under re-estimation. E's lower aggregate distance-flag rate is not, by itself, a reason to call it the better model: it uses more centres and a different geometry.

The defensible conclusion is: **the frozen profiles retain their aggregate interpretation and broadly similar representation quality in unseen countries, but their usefulness is not geographically uniform.** They are qualified descriptive classifications, not universally reproducible natural population types.

In [ ]:
# Objective: Reproduce the explanatory country concentration diagnostic without excluding any country or changing model acceptance.
concentration_rows=[]
country_flag_table=pd.read_csv(TRANSFER_OUTPUT/'heldout_country_quality_and_retention.csv')
for name,group in country_flag_table.groupby('candidate'):
    assert len(group)==29 and group.COUNTRY_ISO3.is_unique
    top=group.nlargest(3,'profile_p95_exceedance')
    remaining=group.loc[~group.COUNTRY_ISO3.isin(top.COUNTRY_ISO3)]
    assert len(remaining)==26
    concentration_rows.append({'candidate':name,'all_countries':29,
        'overall_equal_country_flag_rate':float(group.profile_p95_exceedance.mean()),
        'median_country_flag_rate':float(group.profile_p95_exceedance.median()),
        'three_highest_flag_countries':'; '.join(top.Country),
        'remaining_countries':26,'remaining_26_mean_flag_rate':float(remaining.profile_p95_exceedance.mean()),
        'top_three_contribution_to_all29_rate':float(top.profile_p95_exceedance.sum()/29),
        'status':'post-validation explanatory diagnostic only; all 29 countries remain in official results'})
country_flag_concentration=pd.DataFrame(concentration_rows)
table_write(country_flag_concentration,TRANSFER_OUTPUT/'heldout_country_distance_flag_concentration.csv')
display(country_flag_concentration)
assert set(country_flag_table.COUNTRY_ISO3)==set(expected_holdout)
print('No countries excluded, no numerical rejection threshold introduced, no profile or model changed.')

## 22 Visual evidence and reporting failures

**Objective:** provide reusable visual comparisons without concealing selection or geographic limitations.

**Output:** paired prevalence/interval plots, original-response anchors, country distance flags and all-country B/E prevalence maps.

The maps use the same frozen global profile IDs in all 142 countries, one 0–100% colour scale, and black borders for held-out countries. Small countries use dots/diamonds. They show profile prevalence, not a new country clustering or geographic classification; country/profile tables remain authoritative. Natural Earth de facto borders are not a political claim.

One initial map-generation attempt failed because the survey's **PSE** identifier did not directly match Natural Earth's **PSX** administrative key. It was resolved using the cartography's explicit ISO_A3=PSE field, without changing geometry, scientific results or models. The key audit now verifies all 142 analysis-country codes. This was a reporting failure, not a failed clustering or transfer computation. The earlier Week 4 computational failures and approximations remain in their preserved notebooks; no new model-fitting failure occurred here.

In [ ]:
# Objective: Include the reporting-only table and plot implementation; it contains no fitting, retuning or threshold changes.
"""Post-freeze reporting only; never changes models, transformations or transfer thresholds."""

def transfer_reporting_tables(root,package,output):
    import pandas as pd
    from pathlib import Path
    root,package,output=Path(root),Path(package),Path(output)
    definitions=pd.read_csv(package/'profile_definitions.csv')
    differences=pd.read_csv(output/'heldout_vs_development_category_differences.csv')
    country_anchors=definitions[['candidate','cluster','profile_name']].merge(
        pd.read_csv(package/'development_defining_anchors.csv').drop(columns='profile_name'),
        on=['candidate','cluster'],validate='one_to_many').merge(
        differences.loc[differences.scope.eq('country')],
        on=['candidate','cluster','feature','category'],validate='one_to_many')
    country_anchors['heldout_country_contrast_positive']=country_anchors.difference_vs_outside_heldout.gt(0)
    country_quality=pd.read_csv(output/'heldout_assignment_quality.csv')
    country_anchors=country_anchors.merge(country_quality.loc[country_quality.scope.eq('country'),
        ['candidate','cluster','area_id','respondents','kish_weighted_n']],
        on=['candidate','cluster','area_id'],validate='many_to_one')
    table_write(country_anchors,output/'country_frozen_anchor_diagnostics.csv')
    semantic=pd.read_csv(output/'heldout_profile_semantic_TV.csv').merge(
        country_quality[['candidate','scope','area_id','cluster','respondents','kish_weighted_n']],
        on=['candidate','scope','area_id','cluster'],validate='one_to_one')
    semantic=semantic.merge(definitions[['candidate','cluster','profile_name']],on=['candidate','cluster'],validate='many_to_one')
    table_write(semantic,output/'semantic_differences_with_sample_sizes.csv')
    holdout=pd.read_parquet(root/'course_artifacts/week03_heldout_country_data.parquet',
        columns=['COUNTRY_ISO3','Country','Age','AgeGroups3','Gender','Education','Urbanicity2','WGT','WPID_RANDOM'])
    development=pd.read_parquet(root/'course_artifacts/week03_development_data.parquet',
        columns=['COUNTRY_ISO3','Country','Age','AgeGroups3','Gender','Education','Urbanicity2','WGT','WPID_RANDOM'])
    country_names=pd.concat([development[['COUNTRY_ISO3','Country']],holdout[['COUNTRY_ISO3','Country']]]).drop_duplicates()
    assert country_names.COUNTRY_ISO3.is_unique
    country=pd.read_csv(output/'heldout_country_retention.csv').merge(country_names,on='COUNTRY_ISO3',validate='one_to_one')
    quality_country=country_quality.loc[country_quality.scope.eq('country')&country_quality.cluster.eq(-1)].merge(
        country,left_on='area_id',right_on='COUNTRY_ISO3',validate='many_to_one')
    table_write(quality_country,output/'heldout_country_quality_and_retention.csv')
    selection=[]
    for split,raw,audit_path in [('development',development,package/'development_eligibility_and_assignments_all_respondents.parquet'),
                               ('heldout',holdout,output/'heldout_eligibility_and_assignments_all_respondents.parquet')]:
        eligibility=pd.read_parquet(audit_path,columns=['WPID_RANDOM','complete_case'])
        raw=raw.merge(eligibility,on='WPID_RANDOM',validate='one_to_one')
        for cohort,frame in [('prepared',raw),('complete_case',raw.loc[raw.complete_case]),('excluded',raw.loc[~raw.complete_case])]:
            weights=frozen_country_weights(frame)
            for feature in ['Gender','Education','Urbanicity2','AgeGroups3']:
                values=frame[feature].fillna('missing').astype(str)
                for category in sorted(values.unique()):
                    mask=values.eq(category).to_numpy()
                    selection.append({'split':split,'cohort':cohort,'feature':feature,'category':category,
                        'respondents':int(mask.sum()),'countries_in_cohort':int(frame.COUNTRY_ISO3.nunique()),
                        'equal_country_weighted_share':float(weights[mask].sum()),
                        'interpretation':'descriptive selection audit; weights recomputed within cohort; country sets may differ'})
            observed=frame.Age.notna().to_numpy()
            selection.append({'split':split,'cohort':cohort,'feature':'Age','category':'observed mean',
                'respondents':int(observed.sum()),'countries_in_cohort':int(frame.COUNTRY_ISO3.nunique()),
                'equal_country_weighted_share':float(np.average(frame.Age.to_numpy()[observed],weights=weights[observed])),
                'interpretation':'weighted mean age among observed ages; not a category proportion'})
    selection=pd.DataFrame(selection)
    table_write(selection,output/'missingness_selection_demographic_audit.csv')
    prev=pd.read_csv(output/'heldout_vs_development_prevalence.csv')
    intervals=pd.read_csv(output/'heldout_country_bootstrap_intervals.csv')
    share_intervals=intervals.loc[intervals.quantity.eq('profile_share'),['candidate','cluster','p025','p975','valid_draws']]
    comparison=prev.merge(share_intervals,on=['candidate','cluster'],validate='one_to_one')
    table_write(comparison,output/'prevalence_comparison_with_country_intervals.csv')
    recovery=[]
    for name,encoding,balance,k in [('B','012','balanced',4),('E','013','worry_emphasized',8)]:
        source=root/f'course_artifacts/week04_experiment1_complete_overall_worry_v3/{encoding}/cluster_recovery.csv'
        original=pd.read_csv(source)
        selected=original.loc[original.balance.eq(balance)&original.k.eq(k)]
        result=selected.groupby(['experiment','cluster']).jaccard.agg(median_jaccard='median',
            p10_jaccard=lambda g:g.quantile(.1),fits='size').reset_index()
        result['candidate']=name;result['source_sha256']=file_sha256(source)
        recovery.append(result)
    table_write(pd.concat(recovery,ignore_index=True),output/'development_existing_per_profile_recovery.csv')
    return comparison,quality_country,semantic,country_anchors,selection

def plot_transfer_figures(root,package,output):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon,Patch
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable
    from pathlib import Path
    root,package,output=Path(root),Path(package),Path(output)
    figures=output/'figures';figures.mkdir(exist_ok=True)
    files=[]
    shares=pd.read_csv(output/'prevalence_comparison_with_country_intervals.csv')
    fig,axes=plt.subplots(1,2,figsize=(15,7),layout='constrained',sharex=True)
    for ax,name in zip(axes,['B','E']):
        group=shares.loc[shares.candidate.eq(name)].sort_values('cluster');y=np.arange(len(group))
        dev=100*group.equal_country_weighted_share_development.to_numpy();held=100*group.equal_country_weighted_share_heldout.to_numpy()
        ax.barh(y+.17,dev,height=.32,color='#bdbdbd',label='Development: fixed reference')
        ax.barh(y-.17,held,height=.32,color='#327c9e',label='Held-out: unchanged assignments')
        ax.errorbar(held,y-.17,xerr=np.vstack([held-100*group.p025,100*group.p975-held]),fmt='none',ecolor='black',capsize=3)
        labels=['C'+str(r.cluster)+': '+r.profile_name for r in group.itertuples()]
        ax.set_yticks(y,labels,fontsize=9);ax.invert_yaxis();ax.set_xlim(0,48);ax.set_xlabel('Equal-country-weighted profile share (%)')
        ax.set_title(name+' global profiles');ax.grid(axis='x',alpha=.2)
    handles,labels=axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='lower center',bbox_to_anchor=(.5,-.045),ncol=2,fontsize=9)
    fig.suptitle('Frozen profiles: development versus 29 held-out countries\nBars: point estimates; whiskers: 95% fixed-model country-bootstrap intervals',fontsize=13)
    path=figures/'development_vs_heldout_profile_prevalence.png';fig.savefig(path,dpi=160,bbox_inches='tight');plt.close(fig);files.append(path)
    quality=pd.read_csv(output/'heldout_country_quality_and_retention.csv')
    devq=pd.read_csv(package/'development_assignment_quality.csv')
    order=quality.groupby('Country').profile_p95_exceedance.max().sort_values(ascending=False).index.tolist()
    fig,axes=plt.subplots(1,2,figsize=(13,10),sharey=True)
    for ax,name,color in zip(axes,['B','E'],['#327c9e','#b46b32']):
        g=quality.loc[quality.candidate.eq(name)].set_index('Country').reindex(order)
        ax.scatter(100*g.profile_p95_exceedance,np.arange(len(g)),c=color,s=35)
        baseline=devq.loc[devq.candidate.eq(name)&devq.scope.eq('global')&devq.cluster.eq(-1),'profile_p95_exceedance'].iloc[0]
        ax.axvline(100*baseline,color='#777777',linestyle='--',label='Development overall rate')
        ax.set_title(name+': unusually distant assignments');ax.set_xlim(0,28);ax.set_xlabel('Beyond assigned-profile development p95 (%)')
        ax.set_yticks(range(len(order)),order,fontsize=9);ax.grid(axis='x',alpha=.2);ax.legend(fontsize=8)
    axes[0].invert_yaxis()  # Shared axis: invert once, not once per subplot.
    axes[1].tick_params(labelleft=False)
    fig.subplots_adjust(left=.24,right=.98,bottom=.09,top=.90,wspace=.12)
    fig.suptitle('Country-specific transfer limitations hidden by the aggregate\nEach cutoff was frozen in development; these flags are not automatic failure thresholds',fontsize=13)
    path=figures/'heldout_country_distance_flags.png';fig.savefig(path,dpi=160,bbox_inches='tight');plt.close(fig);files.append(path)
    anchors=pd.read_csv(output/'heldout_frozen_anchor_comparison.csv')
    intervals=pd.read_csv(output/'heldout_country_bootstrap_intervals.csv')
    anchors=anchors.merge(intervals.loc[intervals.quantity.eq('original_category_probability'),['candidate','cluster','feature','category','p025','p975']],
        on=['candidate','cluster','feature','category'],validate='one_to_one')
    fig,axes=plt.subplots(1,2,figsize=(15,12),layout='constrained',sharex=True)
    for ax,name in zip(axes,['B','E']):
        g=anchors.loc[anchors.candidate.eq(name)].sort_values(['cluster','feature','category']).reset_index(drop=True)
        y=np.arange(len(g));held=100*g.proportion.to_numpy()
        ax.scatter(100*g.development_proportion,y+.12,color='#888888',marker='x',label='Development')
        ax.errorbar(held,y-.12,xerr=np.vstack([held-100*g.p025,100*g.p975-held]),fmt='o',markersize=3,color='#327c9e',capsize=2,label='Held-out + country interval')
        labels=[]
        for r in g.itertuples():
            category={0:'not',1:'somewhat',2:'very'}[r.category] if r.feature in WORRY else ('harm' if r.category==1 else 'no harm')
            labels.append(f'C{r.cluster}: {r.feature.replace("worry_","").replace("harm_","").replace("_"," ")} ({category})')
        ax.set_yticks(y,labels,fontsize=8);ax.invert_yaxis();ax.set_xlim(-2,103);ax.set_xlabel('Within-profile original response (%)');ax.set_title(name+' frozen descriptive anchors');ax.grid(axis='x',alpha=.2);ax.legend(fontsize=8)
    fig.suptitle('Original-response transfer: anchors selected before held-out access\nSome sharp E anchors follow from the geometry; preservation is not independent discovery of natural types',fontsize=13)
    path=figures/'frozen_anchor_original_response_transfer.png';fig.savefig(path,dpi=160,bbox_inches='tight');plt.close(fig);files.append(path)
    # All-profile country maps use the same global cluster IDs and a single 0–100 scale.
    split=pd.read_csv(package/'frozen_country_split.csv')
    heldout=set(split.loc[split.country_split.eq('heldout_country'),'COUNTRY_ISO3'])
    info=json.loads((root/'.week05_build/points12_run.json').read_text())
    dev=pd.read_csv(root/info['output_directory']/'fixed_global_profile_prevalence.csv')
    held=pd.read_csv(output/'heldout_prevalence.csv')
    all_countries=pd.concat([dev.loc[dev.scope.eq('country')],held.loc[held.scope.eq('country')]],ignore_index=True)
    table_write(all_countries,output/'fixed_global_all_country_prevalence.csv')
    world=json.loads((root/'course_artifacts/cartography/ne_50m_admin_0_countries.geojson').read_text())
    mapping={}
    for feature in world['features']:
        code=feature['properties'].get('ADM0_A3')
        # PSX is the cartography's administrative key; its ISO_A3 is explicitly PSE.
        # This matches identifiers only and does not alter any boundary geometry.
        aliases={'KOS':'XKX','SDS':'SSD','PSX':'PSE'}
        if code=='PSX':assert feature['properties'].get('ISO_A3')=='PSE'
        code=aliases.get(code,code);mapping[code]=feature
    assert not set(all_countries.area_id)-set(mapping)
    atomic_json({'cartography_source':'Natural Earth cached GeoJSON','country_keys_matched':int(all_countries.area_id.nunique()),
        'aliases':{'KOS':'XKX','SDS':'SSD','PSX':'PSE'},'boundaries_changed':False,
        'initial_plot_failure':'PSE had no direct ADM0_A3 match; resolved using verified ISO_A3=PSE on ADM0_A3=PSX',
        'models_and_transfer_results_changed':False},output/'map_key_audit.json')
    names=pd.read_csv(package/'profile_definitions.csv').set_index(['candidate','cluster']).profile_name.to_dict()
    cmap=plt.get_cmap('viridis');norm=Normalize(0,100)
    for name,k in [('B',4),('E',8)]:
        fig,axes=plt.subplots(k//2,2,figsize=(15,4*k//2),squeeze=False)
        for cluster,ax in enumerate(axes.ravel()):
            g=all_countries.loc[all_countries.candidate.eq(name)&all_countries.cluster.eq(cluster)]
            values=(100*g.set_index('area_id').equal_country_weighted_share).to_dict()
            for code,feature in mapping.items():
                if code=='ATA':continue
                geom=feature['geometry'];polygons=geom['coordinates'] if geom['type']=='MultiPolygon' else [geom['coordinates']]
                color=cmap(norm(values[code])) if code in values else '#eeeeee'
                for polygon in polygons:
                    ax.add_patch(Polygon(np.array(polygon[0]),closed=True,facecolor=color,
                        edgecolor='black' if code in heldout else '#888888',linewidth=.55 if code in heldout else .18))
                if code in values:
                    # Use the largest polygon, not the first (which may be a small
                    # island belonging to a large country such as Canada/Russia).
                    largest_extent=max(np.ptp(np.array(p[0])[:,0])*np.ptp(np.array(p[0])[:,1]) for p in polygons)
                    if largest_extent<2:
                        prop=feature['properties'];ax.scatter(float(prop['LABEL_X']),float(prop['LABEL_Y']),
                            s=18,marker='D' if code in heldout else 'o',color=color,edgecolor='black',linewidth=.35,zorder=5)
            ax.set_xlim(-180,180);ax.set_ylim(-60,85);ax.set_aspect('equal');ax.set_xticks([]);ax.set_yticks([])
            ax.set_title(f'{name} C{cluster}: {names[(name,cluster)]}',fontsize=10);ax.set_facecolor('#f5f8fa')
        fig.subplots_adjust(left=.02,right=.90,bottom=.08,top=.94,hspace=.2)
        fig.colorbar(ScalarMappable(norm=norm,cmap=cmap),cax=fig.add_axes([.92,.20,.015,.60])).set_label('Within-country survey-weighted profile share (%)')
        fig.legend(handles=[Patch(facecolor='white',edgecolor='black',label='Black border: held-out country'),
                            Patch(facecolor='#eeeeee',label='No 2023 analysis data')],loc='lower center',bbox_to_anchor=(.5,.04),ncol=2)
        fig.suptitle(name+': frozen global profiles across development and held-out countries',fontsize=15)
        fig.text(.02,.012,'Complete cases; shared 0–100% scale. Tiny countries use dots/diamonds. Natural Earth de facto boundaries are not a political claim.',fontsize=8)
        path=figures/f'{name}_frozen_all_country_profile_maps.png';fig.savefig(path,dpi=160,bbox_inches='tight');plt.close(fig);files.append(path)
    return files


In [ ]:
# Objective: Display the verified completed figures and cartographic key audit without regenerating fits or models.
transfer_figures=[TRANSFER_OUTPUT/'figures'/name for name in [
    'development_vs_heldout_profile_prevalence.png','frozen_anchor_original_response_transfer.png',
    'heldout_country_distance_flags.png','B_frozen_all_country_profile_maps.png','E_frozen_all_country_profile_maps.png']]
for path in transfer_figures:display(Image(filename=str(path)))
display(json.loads((TRANSFER_OUTPUT/'map_key_audit.json').read_text()))

## 23 Objective 1 conclusion: useful fixed profiles, qualified geographic transfer

**Objective:** close the research question with a claim proportional to the evidence, including mixed findings.

**Output:** final model scope, transfer interpretation and limits for the slides/article.

**Final scope:** frozen global B4 and E8 are retained as complementary descriptive classifications of 2023 complete-case respondents. B combines worry-domain patterns and personal/known-person harm exposure; E distinguishes more worry themes. They do not form a strict hierarchy. Ward corroborates E under related Euclidean/centroid-extension geometry, not as wholly independent evidence of latent classes.

**Development:** both global candidates pass the structural screen and both full median/lower-decile ARI≥0.80 tests. The bounded regional search shows that richer global partitions do not recur independently everywhere; several regions support different resolutions or mainly a simple dominant contrast. Among the regional full-benchmark results, seven have k≥3 and seven are diagnostic binary partitions. Smaller-profile recovery is uneven, especially for some E distinctions. These results qualify the global interpretation rather than compel regional replacement models.

**Held-out transfer:** applying the sealed models to 27,972 complete cases in all 29 untouched countries preserves the aggregate defining contrasts and closely similar marginal response distributions, with broadly similar distances/margins. There are important geographic limits, especially unusually distant assignments in Sierra Leone, Somalia and Namibia, and sparse country/profile cells. The appropriate synthesis is **broad aggregate descriptive transfer with partial country-specific representation limitations**. This is an analytical judgement under the predeclared diagnostics, not a numerical acceptance score, p-value or proof of statistical equivalence.

**Scientific contribution:** the profiles provide a reproducible common language for distinguishing domain-specific worry and exposure patterns rather than only an overall burden gradient. The regional analyses reveal repeatable local distinctions, semantic variation and limitations of a universal resolution. That contribution is a structured descriptive framework, not evidence that people fall into four or eight natural social types. Some pure worry anchors are partly induced by modelling geometry and assignment. Claims about outcomes, causal explanations and demographic predictability belong to later objectives.

**Limitations:** complete-case selection; country-balanced rather than population-proportional estimates; conditional survey-weighting without full complex-design variance; marginal semantics rather than the full joint distribution; geometric assignment ambiguity; adaptive development k selection; small regional country counts; uneven individual-profile recovery; and selected country-level deterioration. Fixed-model country-bootstrap intervals reflect only variation over the observed held-out countries. No all-country refit, holdout refit, threshold relaxation or model redesign was performed.

**Completion:** Objective 1's current protocol is fulfilled: regional evidence, final scope/freeze, geographic validation, explicit limitations and a reproducible assignment handoff are documented. Completion means the research question has a defensible, qualified answer—not guaranteed universal success. The holdout has now been used; any later redesign is a new development cycle and cannot reuse these same countries as untouched confirmation. The optional country-similarity extension is now completed in sections 26–28. It is descriptive, not additional untouched validation or a replacement geographic classification.

**Strength of validation claim:** held-out assessment was not a formal quantitative acceptance test. The diagnostic plan and qualitative broad/partial/poor-transfer criteria were fixed before response access, but no overall numerical rejection threshold was declared. We therefore report favourable, geographically qualified performance under those diagnostics, not that the models passed a prespecified quantitative test. The remaining analytical discretion is a methodological limitation. We must not create numerical rejection boundaries retrospectively from these favourable results.

## 23a Discussion clarification: what would have rejected the models?

**Objective:** state the prespecified poor-transfer conditions and the limits of our decision procedure.

**Output:** a transparent falsifiability/rejection discussion without adding post-hoc numerical rules.

The frozen transfer plan specified poor transfer as **substantial loss of defining response patterns or widespread deterioration in representation quality**, assessed for B and E separately. It explicitly said that the final decision was an analytical synthesis, not an automatic numerical score.

Evidence against suitability for the intended global descriptive use would include:

1. **Lost profile meanings:** the full original-unit distributions no longer support the descriptions—for example, a lower-worry profile no longer showing lower worry, or broad-harm members no longer being distinguished by harm exposure. Reversed or vanished non-mechanically constrained contrasts and major shifts across features would weigh against semantic transfer.
2. **Substantially poorer representation:** widespread increases in nearest-centre distances and profile-specific flags, especially with smaller nearest/second-nearest margins, suggesting respondents are forced into poorly fitting available profiles. Mandatory assignment would not rescue such a result.
3. **Geographically widespread weakness:** reassuring aggregate averages conceal substantial loss of meaning or fit across many countries. Concentrated country/profile weaknesses instead support partial or geographically qualified transfer, rather than uniform success.

Different profile frequencies alone, a low-prevalence profile, a few distant respondents, or a sparse country/profile cell would not automatically reject a model. Large complete-case exclusions would limit population coverage; they are not themselves classification errors or proof of centre failure. The development ARI 0.80 screen is not a held-out rejection threshold.

**Important methodological qualification:** we did **not** predeclare an overall numerical rejection boundary, a required number of preserved anchors, or a statistical equivalence test. Thus we cannot claim that B/E passed a prespecified quantitative acceptance test. We can claim that they performed favourably, with explicit geographic qualifications, under the prespecified diagnostics. The discretion left in the broad/partial/poor judgement must be acknowledged in the article. No threshold is retroactively selected to certify these observed results.

The actual evidence—preserved aggregate meanings, modest aggregate geometric changes and concentrated country limitations—supports the qualified conclusion already reported. Negative transfer would have justified concluding that the frozen model was unsuitable or only partially useful for the intended scope, not retuning its centres on the holdout and calling the revised model externally validated. Any redesign now requires a new development cycle and genuinely new validation evidence. Objective 1 can be completed with a negative or mixed answer; completion is not synonymous with model acceptance.

## 24 Reproducible handoff to Objectives 2 and 3

**Objective:** preserve model identities, eligibility and respondent linkage so later analyses cannot silently change the profiles or weighting estimand.

**Output:** portable frozen-model objects, all-respondent and assigned-only Parquet tables, a completion/reuse manifest and verified hashes.

The handoff preserves **146,910 prepared respondent IDs across 142 countries**: 137,460 receive both frozen profile labels and **9,450 remain unassigned** because they lack complete profile responses. Join by unique WPID_RANDOM, keep COUNTRY_ISO3 and country_split, and preserve B_C0–B_C3 and E_C0–E_C7. Do not relabel profiles by their prevalence in a new subgroup or impute labels for excluded respondents.

Two weighting columns are explicitly different: **equal_country_weight_within_split** sums to one separately for development and holdout; **equal_country_weight_all_included_countries** sums to one over the complete assigned cohort across all 142 countries. Use the latter only for a deliberately pooled country-balanced analysis. Concatenating the two split-normalized weights unchanged would give each split equal total mass rather than each country equal mass.

The frozen package is copied into the handoff and its original seal is reverified. Reusing global descriptive labels does not authorize supervised validation leakage: later predictive modelling still needs an explicit country-level design. The package records geographic and E-profile limitations alongside the assignments, rather than handing off labels without their evidential qualifications.

In [ ]:
# Objective: Verify the final ID-linked handoff, frozen models and completed reporting evidence.
handoff_record=json.loads((HANDOFF_OUTPUT/'objective1_completion_and_reuse.json').read_text())
handoff_all=pd.read_parquet(HANDOFF_OUTPUT/'all_respondents_eligibility_and_frozen_profiles.parquet')
assert len(handoff_all)==146910 and handoff_all.WPID_RANDOM.is_unique
assert int(handoff_all.complete_case.sum())==137460 and int((~handoff_all.complete_case).sum())==9450
for name in SPECIFICATIONS:
    assert handoff_all.loc[handoff_all.complete_case,name+'_cluster'].notna().all()
    assert handoff_all.loc[~handoff_all.complete_case,name+'_cluster'].isna().all()
assert np.isclose(handoff_all.equal_country_weight_all_included_countries.sum(),1)
for split,group in handoff_all.groupby('country_split'):
    assert np.isclose(group.equal_country_weight_within_split.sum(),1)
load_frozen_package(HANDOFF_OUTPUT/'frozen_models')
for name,expected in handoff_record['files'].items():assert file_sha256(HANDOFF_OUTPUT/name)==expected
load_frozen_package(FROZEN_PACKAGE)
final_inventory={
    'status':'Objective 1 completed with qualified descriptive transfer conclusions',
    'freeze_context':EXPECTED_FREEZE_CONTEXT,'notebook':'week05_regional_validation_and_objective1_finalization.ipynb',
    'frozen_scope':['B4','E8'],'models_refitted':False,'development_ARI_threshold':.80,
    'heldout_prepared':30090,'heldout_assigned':27972,'heldout_countries':29,
    'conclusion':'Broad aggregate descriptive transfer; material country-specific representation limitations',
    'freeze_seal_sha256':file_sha256(FROZEN_PACKAGE/'freeze_seal.json'),
    'transfer_record_sha256':file_sha256(TRANSFER_OUTPUT/'transfer_complete.json'),
    'handoff_record_sha256':file_sha256(HANDOFF_OUTPUT/'objective1_completion_and_reuse.json'),
    'tables':{p.name:file_sha256(p) for p in sorted(TRANSFER_OUTPUT.glob('*.csv'))},
    'figures':{str(p.relative_to(TRANSFER_OUTPUT)):file_sha256(p) for p in transfer_figures},
    'reporting_failure':'PSE/PSX map key mismatch repaired using ISO_A3; model/transfer results unchanged',
    'major_cautions':['complete-case selection','country-balanced estimand','geometry-constrained anchors',
        'weak development recovery of some E profiles','country-specific distance deterioration','no universal latent-type claim']}
atomic_json(final_inventory,TRANSFER_OUTPUT/'objective1_final_evidence_inventory.json')
display(handoff_record)
display(final_inventory)
print('Objective 1 complete: frozen models verified, fixed-model held-out transfer documented, all IDs retained for reuse.')

## 25 What profiles add beyond burden, and how heterogeneous their members are

**Objective:** finish the protocol's explicit requirement to describe within-profile variation and distinguish profiles from overall worry/harm scores.

**Output:** full burden distributions, original-pattern frequencies, assignment margins and an exact-burden information-loss table, separately for development and held-out respondents.

Use original 0/1/2 worry responses, not the model-specific transformed coordinates. Define \(S_W=\sum_{j=1}^{6} W_j\), ranging from 0 to 12, and \(S_H=\sum_{j=1}^{7}\max(H_{j,\mathrm{personal}},H_{j,\mathrm{known}})\), ranging from 0 to 7. A known-person indicator does not count the number of known people; each domain contributes at most one to the combined total. Also report separate personal and known-person domain counts. Profile summaries retain the original country-balanced analysis weights, normalized within the profile; countries do not receive equal weight again inside each profile.

For every observed exact pair \((S_W,S_H)\), tabulate all frozen profile labels. The largest joint profile mass in each cell gives the **best possible in-sample weighted accuracy of any deterministic label lookup using only those two totals**:
\[
A_{\mathrm{lookup}}=\sum_{s,t}\max_c P(C=c,S_W=s,S_H=t).
\]
This is a descriptive upper bound computed and evaluated on the same cohort, not a trained classifier, cross-validated accuracy or an external-outcome test. Its complement quantifies label information that these two totals cannot reconstruct. It concerns these particular summaries, not every possible definition of burden; it combines domain-pattern and personal/known-person information. Sparse cells may inflate the in-sample upper bound. We make no significance claim.

**Results:** the development ceiling is **62.95% for B** and **51.93% for E**; held-out ceilings are **62.80%** and **51.61%**. Thus approximately 37% and 48% of weighted profile membership cannot be reconstructed from these two exact totals, even with an in-sample lookup. The conditional label entropy is 1.12 bits for B and 1.81 bits for E in development. This supports the narrower descriptive claim that the profiles preserve domain/relationship distinctions absent from these totals. Because clustering used those richer features, some additional information is expected by construction; this is not proof of sociological importance or superior outcome prediction. Objective 3 must evaluate that separately.

**Profiles are not identical people.** The modal exact original response pattern accounts for only **0.5–15.4%** of a development B profile and **2.1–14.4%** of an E profile. Counts of distinct patterns depend strongly on sample size; do not compare them as a standardized diversity measure. Report distributions, rather than treating a centroid or profile mean as every member's response. Low margins remain geometric boundary ambiguity, not probabilities.

For example, B C0's development mean is 4.56 affected domains versus 1.27, 1.54 and 0.71 in C1–C3; its known-person count is 3.99 compared with a personal count of 1.30. E C4 and C5 have almost equal mean worry (0.90 and 0.89 on 0–2) but distinguish crime and severe-weather themes. Transfer is not exact equality: B C2's affected-domain mean increases from 1.54 to 1.77, and E C7's personal-harm mean from 1.28 to 1.63. These changes remain visible despite preserved broad aggregate interpretations.

In [ ]:
# Objective: Preserve the full country-distance and burden-summary implementation; no respondent-profile fitting or retuning.
"""Post-validation descriptive extensions; never fit or modify respondent profiles."""
from pathlib import Path
import hashlib
import json
import platform
import traceback
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, cut_tree, leaves_list
from scipy.stats import spearmanr
from sklearn.metrics import silhouette_score, adjusted_rand_score

EXTENSION_DESIGN = {
    'version': 1, 'seed': 20261012, 'country_bootstrap_repetitions': 200,
    'country_basis': 'within-country survey-weighted frozen global B or E proportions; never local cluster IDs',
    'distance': 'Hellinger = Euclidean distance of square-root proportions / sqrt(2)',
    'scopes': ['development', 'all_countries_post_validation'],
    'geography': 'unchanged dataset GlobalRegion and approximate great-circle map-label-point distances',
    'country_clustering': 'average linkage; descriptive cuts 2..8; largest silhouette selects an illustrative cut only',
    'tie_break': 'smaller k on equal silhouette; country ISO order for nearest-neighbour ties',
    'uncertainty': '200 IID respondent bootstraps within each country; retain WGT and frozen labels; conditional sensitivity only',
    'inference': 'effect sizes only; no independent-pair p-values, no causal claims, no geographic regrouping',
    'burden_basis': 'original worry sum (0..12) and combined affected-domain count (0..7)',
    'holdout_use': 'post-validation descriptive extension, not untouched confirmation or model selection',
    'respondent_models_refitted': False,
}

def extension_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def extension_json(value, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False))
    temporary.replace(path)

def extension_csv(frame, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix+'.tmp')
    frame.to_csv(temporary, index=False); temporary.replace(path)

def hellinger_matrix(proportions):
    p = np.asarray(proportions, float)
    assert np.isfinite(p).all() and (p >= 0).all() and np.allclose(p.sum(axis=1), 1)
    d = squareform(pdist(np.sqrt(p), metric='euclidean')) / np.sqrt(2)
    np.fill_diagonal(d, 0.)
    assert np.allclose(d, d.T) and d.max() <= 1+1e-12
    return d

def weighted_percentile(values, weights, q):
    order = np.argsort(values, kind='stable')
    return float(np.asarray(values)[order][np.searchsorted(np.cumsum(np.asarray(weights)[order])/sum(weights),q,side='left')])

def map_features(root):
    world = json.loads((root/'course_artifacts/cartography/ne_50m_admin_0_countries.geojson').read_text())
    mapping = {}
    for item in world['features']:
        code = item['properties']['ADM0_A3']
        if code == 'PSX': assert item['properties']['ISO_A3'] == 'PSE'
        mapping[{'KOS':'XKX','SDS':'SSD','PSX':'PSE'}.get(code,code)] = item
    return mapping

def great_circle_matrix(metadata, mapping):
    coords = np.array([[mapping[c]['properties']['LABEL_Y'],mapping[c]['properties']['LABEL_X']] for c in metadata.COUNTRY_ISO3],float)
    lat,lon = np.radians(coords).T
    angle = np.sin((lat[:,None]-lat[None,:])/2)**2 + np.cos(lat[:,None])*np.cos(lat[None,:])*np.sin((lon[:,None]-lon[None,:])/2)**2
    return 6371.0088*2*np.arcsin(np.sqrt(np.clip(angle,0,1)))

def descriptive_country_tree(d, metadata, candidate, scope):
    tree = linkage(squareform(d, checks=True), method='average', optimal_ordering=True)
    rows, labels = [], {}
    for k in range(2,9):
        lab = cut_tree(tree,n_clusters=k).ravel()
        labels[k] = lab
        counts = np.bincount(lab)
        rows.append({'candidate':candidate,'scope':scope,'k':k,
            'silhouette':float(silhouette_score(d,lab,metric='precomputed')),
            'smallest_country_group':int(counts.min()),'largest_country_group':int(counts.max()),
            'singleton_groups':int((counts==1).sum()),
            'ARI_with_dataset_regions':float(adjusted_rand_score(metadata.GlobalRegion,lab)),
            'status':'exploratory descriptive cut, not stability-qualified or an optimal country typology'})
    screen = pd.DataFrame(rows)
    selected = int(screen.sort_values(['silhouette','k'],ascending=[False,True]).iloc[0].k)
    return tree, screen, selected, labels[selected]

def burden_and_heterogeneity(raw, assigned, definitions, output):
    worry = ['worry_'+d for d in ['food','water','crime','severe_weather','traffic','mental_health']]
    domains = ['food','water','crime','severe_weather','traffic','mental_health','work']
    personal = ['harm_'+d+'_personal' for d in domains]
    known = ['harm_'+d+'_known_person' for d in domains]
    raw = raw.merge(assigned[['WPID_RANDOM','B_cluster','E_cluster']],on='WPID_RANDOM',validate='one_to_one')
    raw['worry_sum'] = raw[worry].sum(axis=1).astype(int)
    raw['combined_harm_count'] = np.maximum(raw[personal].to_numpy(),raw[known].to_numpy()).sum(axis=1).astype(int)
    raw['personal_harm_count'] = raw[personal].sum(axis=1)
    raw['known_harm_count'] = raw[known].sum(axis=1)
    # An exact original response pattern, rather than a mean or a transformed coordinate.
    pattern = (raw[worry].to_numpy(dtype=np.int64) @ (3**np.arange(6,dtype=np.int64)))
    harm_pairs = raw[personal].to_numpy(dtype=np.int64)+2*raw[known].to_numpy(dtype=np.int64)
    raw['exact_pattern'] = pattern+3**6*(harm_pairs @ (4**np.arange(7,dtype=np.int64)))
    summaries, distributions, cells, ceilings = [], [], [], []
    names = definitions.set_index(['candidate','cluster']).profile_name.to_dict()
    for split,frame in raw.groupby('country_split'):
        weights = frame.WGT/frame.groupby('COUNTRY_ISO3').WGT.transform('sum')/frame.COUNTRY_ISO3.nunique()
        frame = frame.assign(weight=weights)
        for candidate,k in [('B',4),('E',8)]:
            label = candidate+'_cluster'
            for cluster,group in frame.groupby(label):
                w = group.weight.to_numpy(); total = w.sum()
                pattern_mass = group.groupby('exact_pattern').weight.sum()
                summaries.append({'split':split,'candidate':candidate,'cluster':int(cluster),
                    'profile_name':names[(candidate,int(cluster))], 'respondents':len(group),
                    'distinct_original_response_patterns':int(len(pattern_mass)),
                    'most_common_exact_pattern_share':float(pattern_mass.max()/total),
                    'median_margin':weighted_percentile(group[candidate+'_margin'],w,.5),
                    'p10_margin':weighted_percentile(group[candidate+'_margin'],w,.1),
                    'mean_worry_original_0_to_2':float(np.average(group.worry_sum/6,weights=w)),
                    'mean_combined_harmed_domains':float(np.average(group.combined_harm_count,weights=w)),
                    'mean_personal_harmed_domains':float(np.average(group.personal_harm_count,weights=w)),
                    'mean_known_person_harmed_domains':float(np.average(group.known_harm_count,weights=w))})
                for feature in ['worry_sum','combined_harm_count','personal_harm_count','known_harm_count']:
                    for value,g in group.groupby(feature):
                        distributions.append({'split':split,'candidate':candidate,'cluster':int(cluster),
                            'feature':feature,'value':float(value),'conditional_weighted_share':float(g.weight.sum()/total)})
            table = frame.groupby(['worry_sum','combined_harm_count',label]).weight.agg(['sum','size']).reset_index()
            table.columns = ['worry_sum','combined_harm_count','cluster','joint_weight','respondents']
            table['cell_weight'] = table.groupby(['worry_sum','combined_harm_count']).joint_weight.transform('sum')
            table['within_burden_cell_profile_share'] = table.joint_weight/table.cell_weight
            table['candidate'] = candidate; table['split'] = split; cells.append(table)
            maxima = table.groupby(['worry_sum','combined_harm_count']).joint_weight.max().sum()
            entropy = -float((table.joint_weight*np.log2(table.within_burden_cell_profile_share)).sum())
            ceilings.append({'split':split,'candidate':candidate,'burden_lookup_in_sample_accuracy_ceiling':float(maxima),
                'mass_not_reconstructible_from_two_totals':float(1-maxima),
                'conditional_profile_entropy_bits':entropy,
                'status':'descriptive exact-cell upper bound; not cross-validated performance or external-outcome benefit'})
    extension_csv(pd.DataFrame(summaries),output/'profile_burden_and_heterogeneity.csv')
    extension_csv(pd.DataFrame(distributions),output/'profile_burden_distributions.csv')
    extension_csv(pd.concat(cells,ignore_index=True),output/'same_burden_profile_composition.csv')
    extension_csv(pd.DataFrame(ceilings),output/'burden_only_information_loss.csv')
    return raw

def compute_country_extensions(root, output, assigned, audit, metadata, mapping):
    repetitions = EXTENSION_DESIGN['country_bootstrap_repetitions']
    rng = np.random.default_rng(EXTENSION_DESIGN['seed'])
    boot = {name:np.zeros((repetitions,len(metadata),k)) for name,k in [('B',4),('E',8)]}
    vectors = {}; sources = []
    for candidate,k in [('B',4),('E',8)]:
        weighted = assigned.assign(mass=assigned.WGT/assigned.groupby('COUNTRY_ISO3').WGT.transform('sum'))
        p = weighted.groupby(['COUNTRY_ISO3',candidate+'_cluster']).mass.sum().unstack(fill_value=0).reindex(index=metadata.COUNTRY_ISO3,columns=range(k),fill_value=0).to_numpy()
        vectors[candidate] = p
        for i,row in metadata.iterrows():
            group = assigned.loc[assigned.COUNTRY_ISO3.eq(row.COUNTRY_ISO3)]
            weights = group.WGT.to_numpy()
            for c in range(k):
                sources.append({'candidate':candidate,**row.to_dict(),'cluster':c,'profile_share':float(p[i,c]),
                    'profile_respondents':int(group[candidate+'_cluster'].eq(c).sum())})
    extension_csv(pd.DataFrame(sources),output/'country_profile_vectors.csv')
    # IID respondent bootstrap: common multiplicities for B and E, original WGT retained.
    for i,row in metadata.iterrows():
        group = assigned.loc[assigned.COUNTRY_ISO3.eq(row.COUNTRY_ISO3)]
        counts = rng.multinomial(len(group),np.full(len(group),1/len(group)),size=repetitions)
        weighted_counts = counts*group.WGT.to_numpy()[None,:]
        denominator = weighted_counts.sum(axis=1)
        for candidate,k in [('B',4),('E',8)]:
            for c in range(k):
                boot[candidate][:,i,c] = weighted_counts[:,group[candidate+'_cluster'].eq(c).to_numpy()].sum(axis=1)/denominator
        if (i+1)%30==0: print('Conditional country-proportion bootstrap:',i+1,'of',len(metadata),flush=True)
    pair_rows, neighbour_rows, summary_rows, cut_rows, group_rows, mean_rows, stability_rows, nn_uncertainty, agreement_rows = [],[],[],[],[],[],[],[],[]
    for scope in EXTENSION_DESIGN['scopes']:
        indices = np.flatnonzero(metadata.country_split.eq('train')) if scope=='development' else np.arange(len(metadata))
        meta = metadata.iloc[indices].reset_index(drop=True)
        geo = great_circle_matrix(meta,mapping)
        triangle = np.triu_indices(len(meta),1)
        scope_distances, scope_nearest = {},{}
        for candidate in ['B','E']:
            p = vectors[candidate][indices]; d = hellinger_matrix(p)
            scope_distances[candidate] = d
            extension_csv(pd.DataFrame(d,index=meta.COUNTRY_ISO3,columns=meta.COUNTRY_ISO3).reset_index(),output/f'{candidate}_{scope}_hellinger_matrix.csv')
            same = meta.GlobalRegion.to_numpy()[:,None]==meta.GlobalRegion.to_numpy()[None,:]
            tmp = d.copy();np.fill_diagonal(tmp,np.inf)
            rank = np.argsort(tmp,axis=1,kind='stable'); nearest = rank[:,0];scope_nearest[candidate] = nearest
            within, between = [],[]
            for i,row in meta.iterrows():
                same_peers = np.flatnonzero(same[i] & (np.arange(len(meta))!=i))
                if len(same_peers):
                    within.append(float(d[i,same_peers].mean()));between.append(float(d[i,~same[i]].mean()))
                for position,j in enumerate(rank[i,:5],1):
                    other=meta.iloc[j]
                    neighbour_rows.append({'candidate':candidate,'scope':scope,'country':row.Country,'iso3':row.COUNTRY_ISO3,
                        'region_name':row.region_name,'rank':position,'neighbour':other.Country,'neighbour_iso3':other.COUNTRY_ISO3,
                        'neighbour_region':other.region_name,'same_region':bool(same[i,j]),'hellinger':float(d[i,j]),
                        'map_label_distance_km':float(geo[i,j]),'retention':row.retained_fraction,
                        'neighbour_retention':other.retained_fraction})
            for i,j in zip(*triangle):
                pair_rows.append({'candidate':candidate,'scope':scope,'iso3_a':meta.iloc[i].COUNTRY_ISO3,
                    'iso3_b':meta.iloc[j].COUNTRY_ISO3,'same_region':bool(same[i,j]),'hellinger':float(d[i,j]),
                    'map_label_distance_km':float(geo[i,j])})
            tree,screen,selected,labels = descriptive_country_tree(d,meta,candidate,scope)
            extension_csv(pd.DataFrame(tree,columns=['child_a','child_b','average_hellinger','countries']),output/f'{candidate}_{scope}_average_linkage.csv')
            cut_rows.append(screen)
            region_counts = meta.groupby('GlobalRegion').COUNTRY_ISO3.transform('size').to_numpy()
            summary_rows.append({'candidate':candidate,'scope':scope,'countries':len(meta),
                'same_region_nearest_fraction':float(same[np.arange(len(meta)),nearest].mean()),
                'same_region_random_peer_reference':float(((region_counts-1)/(len(meta)-1)).mean()),
                'same_region_top5_fraction':float(same[np.arange(len(meta))[:,None],rank[:,:5]].mean()),
                'country_balanced_within_region_distance':float(np.mean(within)),
                'country_balanced_between_region_distance_matched_countries':float(np.mean(between)),
                'countries_with_same_region_peers':len(within),
                'profile_vs_geographic_distance_spearman':float(spearmanr(d[triangle],geo[triangle]).statistic),
                'illustrative_country_k':selected,
                'illustrative_silhouette':float(screen.loc[screen.k.eq(selected),'silhouette'].iloc[0]),
                'country_group_vs_region_ARI':float(adjusted_rand_score(meta.GlobalRegion,labels))})
            for i,row in meta.iterrows():
                group_rows.append({'candidate':candidate,'scope':scope,**row.to_dict(),'country_group':int(labels[i])+1,'illustrative_k':selected})
            for group in sorted(set(labels)):
                for c in range(p.shape[1]):
                    mean_rows.append({'candidate':candidate,'scope':scope,'country_group':int(group)+1,
                        'countries':int((labels==group).sum()),'cluster':c,'mean_country_profile_share':float(p[labels==group,c].mean())})
            same_neighbour_count=np.zeros(len(meta),int);outside_count=np.zeros(len(meta),int)
            for repeat in range(repetitions):
                bd = hellinger_matrix(boot[candidate][repeat,indices])
                bt = linkage(squareform(bd),method='average')
                bl = cut_tree(bt,n_clusters=selected).ravel()
                stability_rows.append({'candidate':candidate,'scope':scope,'repeat':repeat,'fixed_illustrative_k':selected,
                    'country_partition_ARI':float(adjusted_rand_score(labels,bl))})
                np.fill_diagonal(bd,np.inf);bn=bd.argmin(axis=1)
                same_neighbour_count += bn==nearest
                outside_count += ~same[np.arange(len(meta)),bn]
            for i,row in meta.iterrows():
                nn_uncertainty.append({'candidate':candidate,'scope':scope,'iso3':row.COUNTRY_ISO3,'country':row.Country,
                    'reference_neighbour':meta.iloc[nearest[i]].Country,'reference_neighbour_same_region':bool(same[i,nearest[i]]),
                    'exact_nearest_repeated_fraction':float(same_neighbour_count[i]/repetitions),
                    'nearest_outside_region_fraction':float(outside_count[i]/repetitions),
                    'bootstrap_repetitions':repetitions,'status':'conditional IID respondent sensitivity; not survey-design confidence or model-fit uncertainty'})
        agreement_rows.append({'scope':scope,'countries':len(meta),
            'B_E_distance_rank_spearman':float(spearmanr(scope_distances['B'][triangle],scope_distances['E'][triangle]).statistic),
            'B_E_exact_nearest_agreement':float((scope_nearest['B']==scope_nearest['E']).mean())})
    for table,name in [(pair_rows,'country_pair_distances'),(neighbour_rows,'nearest_country_neighbours'),
                       (summary_rows,'country_geography_summary'),(group_rows,'illustrative_country_groups'),
                       (mean_rows,'country_group_profile_meanings'),(stability_rows,'conditional_country_group_bootstrap'),
                       (nn_uncertainty,'conditional_nearest_neighbour_bootstrap'),(agreement_rows,'B_E_country_distance_agreement')]:
        extension_csv(pd.DataFrame(table),output/(name+'.csv'))
    extension_csv(pd.concat(cut_rows,ignore_index=True),output/'country_tree_cut_diagnostics.csv')

def extension_plots(root,output,definitions):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon,Patch
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable
    from scipy.cluster.hierarchy import dendrogram
    figures=output/'figures';figures.mkdir(exist_ok=True)
    mapping=map_features(root)
    summary=pd.read_csv(output/'country_geography_summary.csv')
    groups=pd.read_csv(output/'illustrative_country_groups.csv')
    pairs=pd.read_csv(output/'country_pair_distances.csv')
    burden=pd.read_csv(output/'profile_burden_and_heterogeneity.csv')
    dist=pd.read_csv(output/'profile_burden_distributions.csv')
    for candidate,k in [('B',4),('E',8)]:
        scope='all_countries_post_validation'
        meta=groups.loc[groups.candidate.eq(candidate)&groups.scope.eq(scope)].sort_values('COUNTRY_ISO3').reset_index(drop=True)
        matrix=pd.read_csv(output/f'{candidate}_{scope}_hellinger_matrix.csv').iloc[:,1:].to_numpy()
        tree=pd.read_csv(output/f'{candidate}_{scope}_average_linkage.csv').to_numpy()
        order=leaves_list(tree)
        fig,ax=plt.subplots(figsize=(12,11),layout='constrained')
        im=ax.imshow(matrix[np.ix_(order,order)],vmin=0,vmax=1,cmap='viridis')
        ticks=np.arange(0,len(meta),3)
        ax.set_xticks(ticks,meta.iloc[order[ticks]].COUNTRY_ISO3,rotation=90,fontsize=6)
        ax.set_yticks(ticks,meta.iloc[order[ticks]].COUNTRY_ISO3,fontsize=6)
        ax.set_title(candidate+': country similarity in frozen global-profile proportions\nAverage-linkage order; all 142 countries, post-validation description')
        fig.colorbar(im,ax=ax,label='Hellinger distance (0 = identical proportions; 1 = disjoint support)',shrink=.75)
        fig.savefig(figures/f'{candidate}_country_distance_heatmap.png',dpi=140);plt.close(fig)
        fig,ax=plt.subplots(figsize=(17,7),layout='constrained')
        dendrogram(tree,labels=meta.COUNTRY_ISO3.to_list(),leaf_rotation=90,leaf_font_size=6,ax=ax,color_threshold=0,above_threshold_color='#327c9e')
        ax.set_ylabel('Average Hellinger distance at merge')
        ax.set_title(candidate+': exploratory country hierarchy, not a replacement for the dataset regions')
        fig.savefig(figures/f'{candidate}_country_dendrogram.png',dpi=150);plt.close(fig)
        same=meta.set_index('COUNTRY_ISO3').country_group.to_dict()
        selected=int(meta.illustrative_k.iloc[0]); cmap=plt.get_cmap('tab10')
        fig,ax=plt.subplots(figsize=(15,7))
        for code,item in mapping.items():
            if code=='ATA':continue
            geom=item['geometry'];polys=geom['coordinates'] if geom['type']=='MultiPolygon' else [geom['coordinates']]
            color=cmap((same[code]-1)%10) if code in same else '#eeeeee'
            held=code in set(meta.loc[meta.country_split.eq('heldout_country'),'COUNTRY_ISO3'])
            for poly in polys:ax.add_patch(Polygon(np.array(poly[0]),closed=True,facecolor=color,edgecolor='black' if held else '#888888',linewidth=.6 if held else .2))
            if code in same and max(np.ptp(np.array(p[0])[:,0])*np.ptp(np.array(p[0])[:,1]) for p in polys)<2:
                ax.scatter(item['properties']['LABEL_X'],item['properties']['LABEL_Y'],s=20,c=[color],edgecolor='black',linewidth=.3,zorder=5)
        ax.set(xlim=(-180,180),ylim=(-60,85),aspect='equal');ax.set_xticks([]);ax.set_yticks([])
        handles=[Patch(facecolor=cmap(g),label=f'Illustrative country group {g+1}') for g in range(selected)]
        handles += [Patch(facecolor='white',edgecolor='black',label='Black border: held-out'),Patch(facecolor='#eeeeee',label='No analysis data')]
        ax.legend(handles=handles,loc='lower center',bbox_to_anchor=(.5,-.14),ncol=3,fontsize=8)
        ax.set_title(f'{candidate}: illustrative {selected}-group cut of country-profile similarity\nNot new geographical areas or stability-qualified social types')
        fig.text(.02,.01,'All 142 complete-case country distributions; average-linkage/silhouette exploration after validation. Natural Earth de facto borders are not a political claim.',fontsize=8)
        fig.savefig(figures/f'{candidate}_illustrative_country_groups_map.png',dpi=150,bbox_inches='tight');plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(14,6),layout='constrained')
    for ax,candidate in zip(axes,['B','E']):
        g=pairs.loc[pairs.candidate.eq(candidate)&pairs.scope.eq('all_countries_post_validation')]
        ax.hist([g.loc[g.same_region,'hellinger'],g.loc[~g.same_region,'hellinger']],bins=np.linspace(0,1,26),density=True,
                alpha=.6,label=['Same dataset region','Different dataset regions'],color=['#327c9e','#c47d35'])
        ax.set(title=candidate+': geography overlaps, but does not define similarity',xlabel='Hellinger distance',ylabel='Pairwise distance density');ax.legend(fontsize=9)
    fig.suptitle('Descriptive pair distributions only; pairs share countries and are not independent observations')
    fig.savefig(figures/'same_vs_different_region_distances.png',dpi=150);plt.close(fig)
    fig,axes=plt.subplots(2,2,figsize=(14,10),layout='constrained')
    for row,(candidate,k) in enumerate([('B',4),('E',8)]):
        for col,feature in enumerate(['worry_sum','combined_harm_count']):
            g=dist.loc[dist.split.eq('train')&dist.candidate.eq(candidate)&dist.feature.eq(feature)]
            t=g.pivot(index='cluster',columns='value',values='conditional_weighted_share').reindex(index=range(k),fill_value=0).fillna(0)
            axes[row,col].imshow(t.to_numpy(),vmin=0,vmax=1,cmap='Blues',aspect='auto')
            axes[row,col].set_xticks(range(len(t.columns)),t.columns.astype(int));axes[row,col].set_yticks(range(k),[f'{candidate} C{c}' for c in range(k)])
            axes[row,col].set_xlabel('Original worry sum (0–12)' if feature=='worry_sum' else 'Affected domains, personal OR known person (0–7)')
    fig.colorbar(ScalarMappable(norm=Normalize(0,1),cmap='Blues'),ax=axes.ravel().tolist(),label='Within-profile weighted share',shrink=.8)
    fig.suptitle('Profiles contain distributions of burden, not a single burden value\nFrozen development labels; equal-country-weighted complete cases')
    fig.savefig(figures/'profile_burden_distributions.png',dpi=150);plt.close(fig)
    return sorted(figures.glob('*.png'))

def run_week05_extensions(root):
    root=Path(root)
    package=root/'course_artifacts/week05_frozen_objective1_v1/123d248adda6115f'
    handoff=root/'course_artifacts/week05_objective1_handoff_v1/123d248adda6115f'
    transfer=root/'course_artifacts/week05_heldout_transfer_v1/123d248adda6115f'
    inputs=[package/'freeze_seal.json',handoff/'profile_assigned_respondents.parquet',handoff/'all_respondents_eligibility_and_frozen_profiles.parquet',
        root/'course_artifacts/week03_development_data.parquet',root/'course_artifacts/week03_heldout_country_data.parquet',
        root/'course_artifacts/week03_country_split.csv',root/'23_wrp/WRP_2023_data_dictionary.csv',
        root/'course_artifacts/cartography/ne_50m_admin_0_countries.geojson',transfer/'fixed_global_all_country_prevalence.csv',
        root/'.week05_build/complete_week05_extensions.py']
    fingerprint={'design':EXTENSION_DESIGN,'inputs':{str(p.relative_to(root)):extension_sha(p) for p in inputs}}
    context=hashlib.sha256(json.dumps(fingerprint,sort_keys=True).encode()).hexdigest()
    output=root/'course_artifacts/week05_country_similarity_and_profile_synthesis_v1'/context[:16]
    output.mkdir(parents=True,exist_ok=True)
    load_frozen_package(package)
    record_path=output/'extension_complete.json'
    if record_path.exists():
        record=json.loads(record_path.read_text())
        assert record['context']==context
        for name,digest in record['files'].items():assert extension_sha(output/name)==digest,name
        print('Reusing fingerprint-checked completed descriptive extension:',output,flush=True)
        return output
    extension_json(fingerprint,output/'extension_design_and_inputs.json')
    try:
        assigned=pd.read_parquet(handoff/'profile_assigned_respondents.parquet')
        audit=pd.read_parquet(handoff/'all_respondents_eligibility_and_frozen_profiles.parquet')
        assert len(assigned)==137460 and assigned.WPID_RANDOM.is_unique
        country=pd.read_csv(root/'course_artifacts/week03_country_split.csv').sort_values('COUNTRY_ISO3').reset_index(drop=True)
        counts=audit.groupby('COUNTRY_ISO3').complete_case.agg(complete_respondents='sum',prepared_respondents='size').reset_index()
        counts['retained_fraction']=counts.complete_respondents/counts.prepared_respondents
        country=country.merge(counts,on='COUNTRY_ISO3',validate='one_to_one')
        dictionary=pd.read_csv(root/'23_wrp/WRP_2023_data_dictionary.csv')
        labels=dictionary.loc[dictionary.variable.eq('GlobalRegion'),'value_labels'].iloc[0]
        region_names={int(s.split(' = ',1)[0]):s.split(' = ',1)[1] for s in labels.split(' | ')}
        country['region_name']=country.GlobalRegion.map(region_names)
        mapping=map_features(root);assert not set(country.COUNTRY_ISO3)-set(mapping)
        columns=['WPID_RANDOM','COUNTRY_ISO3','WGT','country_split']+list(WORRY_SOURCES.values())+list(HARM_SOURCES.values())
        raw=pd.concat([pd.read_parquet(root/f'course_artifacts/{filename}',columns=columns) for filename in ['week03_development_data.parquet','week03_heldout_country_data.parquet']],ignore_index=True)
        raw=derive_frozen_features(raw.loc[raw.WPID_RANDOM.isin(assigned.WPID_RANDOM)].copy())
        assert len(raw)==len(assigned) and raw.WPID_RANDOM.is_unique
        raw=raw.merge(assigned[['WPID_RANDOM','B_margin','E_margin']],on='WPID_RANDOM',validate='one_to_one')
        definitions=pd.read_csv(package/'profile_definitions.csv')
        burden_and_heterogeneity(raw,assigned,definitions,output)
        compute_country_extensions(root,output,assigned,audit,country,mapping)
        # Reproduce the existing country shares exactly instead of trusting a new aggregation.
        old=pd.read_csv(transfer/'fixed_global_all_country_prevalence.csv')
        new=pd.read_csv(output/'country_profile_vectors.csv')
        joined=new.merge(old,left_on=['candidate','COUNTRY_ISO3','cluster'],right_on=['candidate','area_id','cluster'],validate='one_to_one')
        assert len(joined)==1704 and np.allclose(joined.profile_share,joined.equal_country_weighted_share,atol=1e-12)
        extension_plots(root,output,definitions)
        load_frozen_package(package)
        assert all(extension_sha(root/name)==digest for name,digest in fingerprint['inputs'].items())
        files=[p for p in output.rglob('*') if p.is_file() and p.name!='extension_complete.json']
        record={'status':'completed post-validation descriptive extensions','context':context,'design':EXTENSION_DESIGN,
            'software':{'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,'sklearn':sklearn.__version__},
            'respondent_models_refitted':False,'heldout_validation_reinterpreted':False,
            'files':{str(p.relative_to(output)):extension_sha(p) for p in sorted(files)}}
        extension_json(record,record_path)
        print('Completed country similarity and burden/heterogeneity extension:',output,flush=True)
        return output
    except Exception:
        extension_json({'status':'failed; preserve diagnostics','context':context,'traceback':traceback.format_exc()},output/'extension_failure.json')
        raise



In [ ]:
# Objective: Verify completed descriptive extensions and all source fingerprints; replay outputs without recomputing or fitting.
EXTENSION_OUTPUT=ROOT/'course_artifacts/week05_country_similarity_and_profile_synthesis_v1/d694096841b06e54'
extension_record=json.loads((EXTENSION_OUTPUT/'extension_complete.json').read_text())
assert extension_record['context']=='d694096841b06e5478353d691e3e545570d50d71afbe35bb93cbd25b5145679a'
assert not extension_record['respondent_models_refitted'] and not extension_record['heldout_validation_reinterpreted']
extension_inputs=json.loads((EXTENSION_OUTPUT/'extension_design_and_inputs.json').read_text())
for relative_path,digest in extension_inputs['inputs'].items():
    assert extension_sha(ROOT/relative_path)==digest,'Extension input changed: '+relative_path
for relative_path,digest in extension_record['files'].items():
    assert extension_sha(EXTENSION_OUTPUT/relative_path)==digest,'Extension evidence changed: '+relative_path
load_frozen_package(FROZEN_PACKAGE)
burden_synthesis=pd.read_csv(EXTENSION_OUTPUT/'profile_burden_and_heterogeneity.csv')
burden_information=pd.read_csv(EXTENSION_OUTPUT/'burden_only_information_loss.csv')
display(burden_information)
display(burden_synthesis)
display(Image(filename=str(EXTENSION_OUTPUT/'figures/profile_burden_distributions.png')))
print('Original profile labels, split, raw codes, survey weights and all frozen package hashes remain unchanged.')

## 26 Country similarity without inventing new geographical areas

**Objective:** ask whether countries have similar distributions over the same frozen global profiles, even when the dataset assigns them to different regions.

**Output:** separate B/E country vectors, complete distance matrices, five closest countries per country, dataset-region comparisons and conditional nearest-neighbour sensitivity.

For country \(a\), define \(p_{ac}\) as the survey-weighted proportion of its complete-case respondents assigned to frozen profile \(c\). The vector sums to one. Every country contributes one vector, irrespective of its population or survey sample size; relative survey weights are retained within that country. Calculate B's four-component and E's eight-component vectors **separately**; never align unrelated local-cluster numbers or concatenate these bases.

Use Hellinger distance:
\[
H(a,b)=\frac{1}{\sqrt{2}}\sqrt{\sum_c(\sqrt{p_{ac}}-\sqrt{p_{bc}})^2},\qquad 0\le H\le1.
\]
Zero means identical profile proportions; one means disjoint profile support. Square roots reduce the dominance of large components and retain sensitivity to smaller profile shares. Zero shares are valid: no pseudocount or log transform is needed. This distance compares **profile mixtures**, not all original responses, within-profile semantics, respondent-to-centroid distances, or causal/social equivalence of countries. Different response distributions can map to the same profile mixture. E's finer, sometimes fragile boundaries remain a limitation of its distance basis.

We report the 113 development countries separately from all 142 countries. The latter analysis includes the already-opened holdout and is **post-validation description**, not new external confirmation. The original 15 GlobalRegion categories remain unchanged.

For within/between-region means, first average distances to each country's same-region peers and other-region peers, then average those country means. Development has two singleton regions: both comparisons use the same 111 countries with same-region peers. All-country comparisons include all 142. The random-peer baseline accounts for the available number of countries in each region; it is not a statistical test. Pairwise distances share countries and are dependent. No ordinary correlation p-value or independent-pair significance test is reported.

**Geographical proximity:** compare profile distance with approximate great-circle distance between Natural Earth's country **map-label points**. These points are not capitals, population centroids, distances between borders or evidence of direct adjacency. For large countries/islands they are especially coarse. This answers a broad spatial-distance question only; we do not claim to have tested shared land borders. The dataset's regional comparison is the principal geographic description.

**Results across all 142 countries:** mean same-region versus other-region distance is **0.161 vs 0.211 for B** and **0.202 vs 0.273 for E**. Regional peers are closer on average. However, only **22.5% (B)** and **30.3% (E)** have their closest profile-mixture neighbour in the same region—above the random-peer reference of **8.1%**, but still leaving most nearest neighbours outside the region. Geographic-distance rank correlations are modest: **0.157** and **0.142**. Development-only findings have the same broad direction.

This is not a paradox: there are many more countries outside each region, and a within-region average need not determine the single minimum. B/E pair-distance rankings correlate **0.719**, yet they choose the same exact nearest country for only **12.7%** of countries. Broad similarity patterns are more concordant than individual nearest-country claims.

**Sampling sensitivity:** use 200 IID respondent bootstrap draws within every country, keep original WGT and frozen labels, recompute proportions/distances and keep the same country-tree cut. The same respondent multiplicities are shared across B/E. This is conditional sampling sensitivity, not full complex-survey variance: it ignores PSU/stratification, imputation/nonresponse uncertainty, model fitting and uncertainty in country selection. Exact nearest-neighbour identity recurs in a median **35.5% of B** and **46.0% of E** draws. Close ranks are often interchangeable; report the closest five and sensitivity rather than calling one partner definitive.

For example, B links Serbia with Iran at H=0.0135 despite different dataset regions; the exact neighbour recurs in 74.0% and 87.5% of their conditional draws respectively. Italy/Ivory Coast are even closer in B (H=0.0094), but their exact pairing recurs only 31.0%/33.5%. This illustrates why numerical proximity alone is insufficient to claim a robust sociological twin. Neither example says their original response patterns or circumstances are identical.

In [ ]:
# Objective: Display country-mixture geography results and complete nearest-neighbour evidence with conditional sampling limits.
country_geography=pd.read_csv(EXTENSION_OUTPUT/'country_geography_summary.csv')
country_distance_agreement=pd.read_csv(EXTENSION_OUTPUT/'B_E_country_distance_agreement.csv')
country_neighbours=pd.read_csv(EXTENSION_OUTPUT/'nearest_country_neighbours.csv')
neighbour_sensitivity=pd.read_csv(EXTENSION_OUTPUT/'conditional_nearest_neighbour_bootstrap.csv')
display(country_geography)
display(country_distance_agreement)
display(neighbour_sensitivity.groupby(['scope','candidate']).exact_nearest_repeated_fraction.agg(['median','mean']))
display(country_neighbours.loc[country_neighbours.scope.eq('all_countries_post_validation')&country_neighbours['rank'].eq(1)])
display(Image(filename=str(EXTENSION_OUTPUT/'figures/same_vs_different_region_distances.png')))
print('Full vectors, all pair distances and all top-five neighbours are saved; no region assignments changed.')

## 27 Country clustering: record the attempt, but do not force a new typology

**Objective:** explore whether the country-distance geometry suggests reproducible groups, without substituting them for dataset regions or respondent profiles.

**Output:** average-linkage trees, all cuts k=2–8, silhouette/size/region-ARI diagnostics, group profile meanings and conditional bootstrap recovery.

Average linkage merges the groups with smallest mean pairwise Hellinger distance. It clusters countries, **not respondents**, and does not change the B/E models. Choose the highest-silhouette cut among 2–8 only to illustrate the tree; ties prefer smaller k. The bounded search and selected cut are exploratory, not a validated estimate of an optimal number of country types. No respondent k≥3/40%/D<3 filter is transplanted to country groups. All cuts, including singleton-heavy ones, remain reported.

**B's selected k=2 is one country (Malawi) versus 141 others.** Malawi has 58.1% in B's broad-harm profile and only 5.2% in the lower-worry/harm profile, versus group means of 18.0% and 34.1% among the other countries. Retention is 94.8%. This is a descriptive extreme in profile mixtures, not automatically a data error or a discovered two-type world. A silhouette of 0.414 does not make the 141-country remainder a coherent class.

**E's selected k=2 divides 64 and 78 countries**, largely by lower-worry versus broader/higher-worry composition. The two groups average 53.1% versus 21.6% in E C1 and 4.8% versus 15.1% in E C7. Its silhouette is 0.351 and agreement with the 15 dataset regions is ARI=0.089. This coarse mixture contrast does not capture all thematic diversity and is not a causal geographic explanation.

Both cuts are sensitive to respondent sampling. All-country partition bootstrap ARI median/lower decile is **0.659/−0.011 for B** and **0.863/0.172 for E**. Development-only lower deciles are also weak. A high E median hides fragile alternative tree splits; B's singleton/tree structure can shift sharply. These draws vary estimated country mixtures with respondent models fixed. They do not vary country inclusion or repeat model fitting. We do **not** transfer the 0.80 respondent-model screen to this different experiment, but the observed spread is plainly insufficient to promote these groups as reliable country types.

**Decision:** retain distances, top-five neighbours, continuous heatmaps and the unchanged geographical categories as the useful results. Preserve the dendrograms and illustrative coloured maps as diagnostics of why a new country classification is not warranted. Do not lower a threshold, remove Malawi, search additional linkages or pick another k merely to produce attractive groups.

Implementation references: [SciPy average linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html) and [scikit-learn silhouettes with precomputed distances](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_samples.html). Actual installed software versions and input fingerprints are saved in extension_complete.json.

In [ ]:
# Objective: Inspect all country-tree cuts, profile-mixture meanings and conditional recovery before interpreting coloured groups.
country_cuts=pd.read_csv(EXTENSION_OUTPUT/'country_tree_cut_diagnostics.csv')
country_groups=pd.read_csv(EXTENSION_OUTPUT/'illustrative_country_groups.csv')
country_group_meanings=pd.read_csv(EXTENSION_OUTPUT/'country_group_profile_meanings.csv')
country_group_recovery=pd.read_csv(EXTENSION_OUTPUT/'conditional_country_group_bootstrap.csv')
display(country_cuts)
display(country_groups.groupby(['scope','candidate','country_group']).size().rename('countries').reset_index())
display(country_group_meanings)
display(country_group_recovery.groupby(['scope','candidate']).country_partition_ARI.agg(
    median='median',p10=lambda x:x.quantile(.1),minimum='min',maximum='max'))
for name in ['B','E']:
    display(Image(filename=str(EXTENSION_OUTPUT/'figures'/f'{name}_country_distance_heatmap.png')))
    display(Image(filename=str(EXTENSION_OUTPUT/'figures'/f'{name}_country_dendrogram.png')))
    display(Image(filename=str(EXTENSION_OUTPUT/'figures'/f'{name}_illustrative_country_groups_map.png')))
print('Maps are diagnostic tree cuts, not retained country classifications or new regions.')

## 28 Final scientific synthesis after the remaining Week 5 work

**Objective:** integrate the new descriptions without retrospectively changing validation, model identity or claims.

**Output:** the final interpretation for course discussion, slides and the article.

1. **Keep the common global classification.** B4 is the stronger core description; E8 gives complementary worry themes with known fragile boundaries. Country distances use these shared bases, not unmatched locally optimized cluster IDs. The regional models describe context and local splits/merges, not replacement global labels.
2. **Geography matters, but is not sufficient.** Countries in the same dataset region are closer on average. Most exact nearest mixture neighbours nevertheless cross those boundaries; many precise nearest ranks are unstable. This supports studying cross-region similarities, not claiming geography is irrelevant or proposing new geographic areas.
3. **Do not force country clusters.** B's silhouette winner mainly isolates Malawi; E resolves a broad composition contrast but lacks consistent lower-tail recovery. Neither exploratory cut is adopted as a country typology. The negative grouping result is a useful research finding, not a reason to change thresholds after seeing it.
4. **Profiles contain more than two totals, but remain broad summaries.** Exact worry/harm totals cannot recover all labels; domain and personal/known-person information remains. Members are heterogeneous. This does not establish added outcome-prediction value or natural classes; those are different claims.
5. **Keep the held-out conclusion geographically qualified.** The new all-country description is performed after validation and adds no untouched confirmation. Complete-case selection, within-profile semantic variation, country-specific fit limits and survey-design uncertainty remain. Country mixtures compress those complexities; similar mixtures do not imply interchangeable societies.

Objective 1 is complete within the approved protocol and the now-completed descriptive extension. No fixed-model holdout refit, native-Ward rerun, new regional definitions, later-wave validation or supervised Objective 2 modelling is required to close this stage. The older Moodle wording about an independent holdout refit is not silently reintroduced: the current 7 October protocol explicitly treats it as optional. Any later model redesign requires a new development/validation cycle.

## 29 Completion checklist and evidence integrity

**Objective:** make the completed scope and reusable files explicit, with a final integrity check.

**Output:** a saved Week 5 completion checklist, extension manifest and links from the existing final evidence inventory.

The original frozen package, held-out transfer record and ID-linked handoff are immutable. New tables/figures live in a fingerprinted extension directory and are referenced from the reporting inventory only. All previous scientific failures and qualifications remain visible. The pre-extension notebook snapshot is kept under earlier iterations. There is one active canonical Week 5 notebook; replay verifies saved evidence and does not run clustering computations.

In [ ]:
# Objective: Save the final completion checklist and link the verified extension without altering sealed models or original validation results.
completion_checklist=pd.DataFrame([
    {'item':'Development global/regional prevalence, semantics, distances and maps','status':'complete','evidence':'Sections 1–10'},
    {'item':'Bounded regional k search and unchanged 0.80 stability qualification','status':'complete','evidence':'Sections 11–17'},
    {'item':'Final B4/E8 decision, model freeze and exact development replay','status':'complete','evidence':'Section 18; sealed package'},
    {'item':'Held-out eligibility, fixed assignment, uncertainty and geographic limits','status':'complete','evidence':'Sections 19–23a; original transfer record'},
    {'item':'Within-profile heterogeneity and comparison with burden totals','status':'complete','evidence':'Section 25; new summary/distribution tables'},
    {'item':'Country distances, regional/spatial comparisons and nearest-neighbour sensitivity','status':'complete descriptive extension','evidence':'Section 26; country vectors/distance matrices'},
    {'item':'Exploratory country grouping and maps','status':'complete; no country typology adopted','evidence':'Section 27; all cuts and bootstrap qualifications'},
    {'item':'ID-linked assignment handoff for later objectives','status':'complete','evidence':'Section 24; 146,910 IDs, 137,460 assigned'},
    {'item':'Held-out refits, later waves, native Ward or new geographic definitions','status':'not required; not performed','evidence':'Current protocol scope; no silent expansion'},
])
extension_csv(completion_checklist,TRANSFER_OUTPUT/'week05_completion_checklist.csv')
load_frozen_package(FROZEN_PACKAGE)
load_frozen_package(HANDOFF_OUTPUT/'frozen_models')
for group in ['tables','assignments']:
    for name,digest in transfer_record[group].items():assert file_sha256(TRANSFER_OUTPUT/name)==digest
for name,digest in handoff_record['files'].items():assert file_sha256(HANDOFF_OUTPUT/name)==digest
for name,digest in extension_record['files'].items():assert extension_sha(EXTENSION_OUTPUT/name)==digest
final_inventory['descriptive_extension']={
    'directory':str(EXTENSION_OUTPUT.relative_to(ROOT)),
    'context':extension_record['context'],
    'manifest_sha256':extension_sha(EXTENSION_OUTPUT/'extension_complete.json'),
    'respondent_models_refitted':False,'country_groups_adopted':False,
    'status':'completed; exploratory country similarity and burden/heterogeneity synthesis',
    'interpretation':'country-mixture similarity, not causal equivalence or new held-out confirmation'}
final_inventory['tables']={p.name:file_sha256(p) for p in sorted(TRANSFER_OUTPUT.glob('*.csv'))}
atomic_json(final_inventory,TRANSFER_OUTPUT/'objective1_final_evidence_inventory.json')
display(completion_checklist)
print('Week 5 complete. Frozen package, original transfer results and handoff hashes verified unchanged.')

## 30 Continuous country-similarity colours with the dataset regions overlaid

**Objective:** show countries with similar frozen global-profile mixtures in similar colours on a world map, rather than show distance to a single selected country or adopt new country clusters.

**Output:** B/E world maps with continuous similarity colours, outlines of the original geographical areas, a two-dimensional colour key, and measured approximation quality. The interactive companion provides an approximate global overview and an original-distance-from-selected-country view, with regional outlines available in both. Selecting a country leaves overview colours fixed; in the distance view it updates colours to that country's original distances. No scientific fit changes.

One scalar colour scale cannot represent every pairwise relationship in a multidimensional distance matrix. We therefore make an explicit **display approximation**. For each model separately, represent countries by \(\sqrt{p_a}/\sqrt{2}\), whose Euclidean distances equal the saved Hellinger distances. Centre these country vectors and obtain the two-dimensional classical metric-MDS coordinates by singular-value decomposition. Map the two coordinates isotropically to CIELAB chromatic coordinates at a common lightness, reducing colour spread until it lies inside the display gamut. Nearby points in the colour key then have similar colours. Coordinate signs are fixed for reproducibility, not given substantive meaning.

This is **not PCA preprocessing of respondent profiles**. The no-PCA B/E specifications, labels and original distances remain unchanged. The dimensional reduction is only for presenting already-computed country distances. Hue is not a worry or harm score, colours do not define country groups, and B/E have separate colour keys: identical-looking colours across the two maps do not imply identical compositions.

**Measured display quality:** the two coordinates retain **81.4% of variation for B** and **73.6% for E** in the centred Hellinger-coordinate representation. Original/projected pair-distance rank correlation is **0.913 for B** and **0.928 for E**. Relative distance error is 0.196/0.229. These are descriptive approximation diagnostics, not confidence levels or an inferential test. The exact nearest neighbour is preserved for only **29.6% of B countries and 10.6% of E countries**. The overview maps therefore show broad similarity patterns; **do not infer exact closest partners from overview colour**. Use the saved full distances and closest-five tables for that purpose. The interactive list defaults to 10 nearest countries, with options for 20 or all 141 others, separately for B and E. It excludes the selected country itself and orders by original unrounded Hellinger distance, resolving exact ties by ISO3 order. Display values are rounded; the rankings are not derived from colour coordinates. The optional distance map uses the original distances on a common 0–1 scale, not the MDS approximation. There is no adopted threshold for "similar countries": nearest is a relative rank and even the nearest country can be substantively different.

The overlay dissolves countries according to the **unchanged dataset GlobalRegion membership**, producing fifteen separate area unions. It does not use response-derived country groups or construct geographical envelopes across seas. Areas may contain disconnected territories. Countries outside the study remain uncoloured and are not assigned invented region memberships. The published 110m cartography omits polygons for Bahrain, Comoros, Hong Kong, Malta, Mauritius and Singapore; verified Natural Earth map-label points show their colours as dots. Boundaries are presentation geometry, not political claims.

The colour view complements, rather than replaces, the exact distance heatmaps. Similar profile mixtures need not mean similar original responses within profiles or interchangeable societies. Complete-case selection and the weaker E distinctions remain limitations. Both maps include the already-used held-out countries and are post-validation description, not new untouched confirmation.

**Presentation history:** an initial reference-country map was superseded after the user clarified that all-country similarity colours were wanted. A URL-encoded local-path error in the initial map builder was corrected; no scientific fit or distance calculation failed. The discarded reference-map design is not the intended final visualization.

In [ ]:
# Objective: Verify and display the requested continuous-colour maps, preserving the original regional and frozen-profile definitions.
CONTINUOUS_MAP_OUTPUT=ROOT/'course_artifacts/week05_continuous_country_similarity_maps_v1'
continuous_map_manifest=json.loads((CONTINUOUS_MAP_OUTPUT/'map_manifest.json').read_text())
assert not continuous_map_manifest['model_refitting'] and not continuous_map_manifest['region_membership_changed']
assert not continuous_map_manifest['respondent_PCA_used']
for name,digest in continuous_map_manifest['files'].items():
    assert extension_sha(CONTINUOUS_MAP_OUTPUT/name)==digest,'Map evidence changed: '+name
load_frozen_package(FROZEN_PACKAGE)
display(pd.read_csv(CONTINUOUS_MAP_OUTPUT/'colour_projection_quality.csv'))
for name in ['B','E']:
    display(Image(filename=str(CONTINUOUS_MAP_OUTPUT/f'{name}_continuous_similarity_with_region_outlines.png')))
final_inventory['continuous_country_similarity_maps']={
    'directory':str(CONTINUOUS_MAP_OUTPUT.relative_to(ROOT)),
    'manifest_sha256':extension_sha(CONTINUOUS_MAP_OUTPUT/'map_manifest.json'),
    'distance_basis':'unchanged frozen-global-profile Hellinger distances',
    'region_basis':'original dataset GlobalRegion',
    'status':'continuous approximate colours; not country clusters or exact nearest-neighbour evidence'}
atomic_json(final_inventory,TRANSFER_OUTPUT/'objective1_final_evidence_inventory.json')
print('World maps with dataset-area outlines saved. No model, profile ID, country region or original distance changed.')

## 31 Country similarity and fieldwork income groups

**Objective:** determine whether countries with closer income categories also have more similar mixtures of the frozen B4/E8 respondent profiles. This is a post-validation descriptive extension, not a new model-selection criterion or an independent validation experiment.

**Implementation:** reuse the saved full-precision Hellinger distances, whose values were verified against the original country profile vectors. No respondent model, feature encoding, assignment, country split or geographical grouping is changed. Within-country profile proportions retain survey weights; countries themselves receive equal influence. The analysis uses only aggregate country records.

### What income means here

Use the dataset's `CountryIncomeLevel`, which records the World Bank FY2023–24 classification at fieldwork: low, lower-middle, upper-middle and high income. These are ordered categories, not measured GNI amounts. The gap between adjacent categories is therefore a category step, not an equal monetary difference. Spearman correlation compares ranks of these category gaps with ranks of profile distances; it does not assume a linear relationship with income in dollars. Do not substitute current income classifications.

There are 17 low-, 40 lower-middle-, 37 upper-middle- and 47 high-income countries. Venezuela is unclassified and excluded from income association calculations, but not from the study or the maps. This leaves 141 classified countries (9,870 distinct pairs) in the full post-validation analysis and 112 (6,216 pairs) in the development-only analysis. The latter is a sensitivity check using a nested subset, not an independent replication.

### Main results

In the full classified set, income-gap/profile-distance Spearman correlation is **0.211 for B4** and **0.229 for E8**. Development-only values are **0.233** and **0.232**. Thus the positive association is present in both models, but is modest: income groups do not reproduce the country-distance geometry.

The equal-focal-country mean distance to same-income peers versus other-income peers is **0.182 versus 0.214 for B4**, and **0.237 versus 0.279 for E8**. Each country contributes one mean to each comparison. This prevents larger income groups from dominating purely through their greater number of pairs. The group-by-group heatmaps additionally display pooled distinct-pair means; group sizes and pair counts are saved separately. Diagonal cells compare different countries in the same income group and never include zero self-distances.

The nearest classified country belongs to the same income group for **41.8% of B4 countries** and **51.8% of E8 countries**. For comparison, a uniformly selected different classified country has the same income group with mean probability **27.0%**, calculated from the actual group sizes. This baseline is descriptive, not a sampling test; the observed nearest-neighbour relationships themselves share countries. Same-income proportions among the ten nearest peers are **37.3%** and **43.0%**. Income matching is enriched, but many closest peers have a different income category.

### Geography and dependence between pairs

Pairs are not independent: every country appears in many pairs. We therefore do **not** report ordinary correlation p-values treating 9,870 pairs as independent observations. For a transparent reference, permute income labels for whole countries 999 times, with a fixed seed, preserving income counts. Also permute within the original dataset regions, preserving each region's income composition. The latter asks whether the observed association exceeds what the regional income pattern alone can generate under within-region label exchangeability.

For B4, the within-region reference mean correlation is **0.131**, with a central 95% randomization interval **0.081–0.179**; the observed **0.211** exceeds it (upper-tail reference fraction **0.002**). Development-only B4 gives the same qualitative conclusion. For E8, the corresponding mean is **0.186**, interval **0.133–0.240**, and upper-tail fraction **0.067**; observed **0.229** does not clearly exceed that reference. Development-only E8 is similar (0.069). These intervals describe shuffled-label reference distributions, not confidence intervals for the observed correlation, and the fractions are conditional randomization summaries, not population or causal p-values. With 999 permutations, precision is limited and the smallest possible reported fraction is 0.001.

Restricting the actual pairs to countries in the same dataset region gives correlations **0.172 for B4** and **0.136 for E8**. That descriptive restriction and the shuffled-label reference answer different questions; a positive within-region coefficient does not by itself establish an independent income effect. Region-preserving permutations do not adjust for physical proximity within regions or every country characteristic. Some regions are income-homogeneous and cannot contribute label rearrangement.

### Interpretation and limitations

The defensible finding is that **country income group is associated with global profile composition**, with substantial overlap between income and geography. B4 supplies clearer evidence of association beyond the coarse regional income pattern than E8. This does not show that income causes profile differences, that countries in one income group are interchangeable, or that country clusters should replace geographical regions. Nor does it establish that these profiles are an income classification in disguise: the correlations are modest and cross-income nearest neighbours remain common.

These are complete-case country mixtures, not unrestricted national population distributions. Complete-case retention, reporting patterns, exposure, institutions and other national conditions may covary with income. The two frozen models also use different profile definitions. Compare their association patterns without treating their distance values or components as identical. If a later article analyses income as a contextual explanatory variable, it should distinguish these exploratory results from a pre-specified, appropriately controlled explanatory analysis.

**Maps:** use the companion's fieldwork-income view for category context, and B4/E8 original-distance views to inspect distances from any selected country. The distance views use saved Hellinger values, not approximate overview colours. Closest-country lists display each peer's income group and region. The four-by-four heatmaps show the relationship more directly than a map alone. Original dataset-region outlines are unchanged; small-country points are presentation aids, not new geographical assignments.

**Evidence:** `course_artifacts/week05_country_income_similarity_v1/2f1deca4408a3b91`. The manifest records input fingerprints, exclusions, weighting, randomization design and limitations. Tables and figures below are saved outputs from the aggregate extension, not reruns of the respondent clustering.


In [ ]:
# Objective: Verify and display saved aggregate income comparisons without refitting respondent profiles.
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display, Image
INCOME_OUTPUT=ROOT/'course_artifacts/week05_country_income_similarity_v1/2f1deca4408a3b91'
income_manifest=json.loads((INCOME_OUTPUT/'analysis_manifest.json').read_text())
assert not income_manifest['model_refitting']
for filename,digest in income_manifest['files'].items():
    assert hashlib.sha256((INCOME_OUTPUT/filename).read_bytes()).hexdigest()==digest,filename
for relative,digest in income_manifest['source_hashes'].items():
    assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest()==digest,relative
income_summary=pd.read_csv(INCOME_OUTPUT/'income_distance_summary.csv')
income_references=pd.read_csv(INCOME_OUTPUT/'country_label_permutation_references.csv')
display(pd.read_csv(INCOME_OUTPUT/'income_group_counts.csv'))
display(income_summary)
display(income_references)
for scope in ['development','all_countries_post_validation']:
    display(Image(filename=str(INCOME_OUTPUT/f'{scope}_income_distance_heatmaps.png')))
print('Verified aggregate income extension. No clustering refitted; income associations exclude unclassified Venezuela.')


## Shallow burden-score trees for fixed B4/E8 labels

**Objective:** quantify how much membership can be explained by trees of depths 1, 2 and 3, with depth 5 retained for comparison. Use the same original mean worry, seven-domain OR-combined harm score, fixed labels, five country-grouped folds, equal-country weights, minimum 200 respondents per leaf and seed 2023 as the preceding diagnostic. Only depth-1 and depth-2 trees were newly fitted (20 fits); fingerprint-verified depth-3/depth-5 fold results were reused.

**Interpretation:** a depth-d multiclass tree has at most 2^d leaves and can predict at most that many distinct profiles. Its balanced accuracy cannot exceed min(2^d, k)/k when all k profiles are present in evaluation. For B4, depth 1 is limited to 50%; for E8, depths 1 and 2 are limited to 25% and 50%. These are capacity bounds, not chance benchmarks or expected performance. The constant-label benchmarks are 25% and 12.5%, respectively.

**Scope:** these are post-freeze, development-only explanations. Cross-validation assesses classifiers conditional on the already-fixed global profile definitions, not independent recovery of the clustering. No held-out responses, profile redesign or new retention threshold enter this extension. Trees optimize country-weighted Gini impurity, not class-balanced loss; poor rare-profile recovery can affect balanced accuracy. Low shallow-tree scores alone do not prove that a profile is intrinsically multidimensional.

The full implementation is preserved in the K-means notebook. This section replays the same verified aggregate evidence.


In [ ]:
"""Explain fixed B/E labels with the Week 3 country-block procedures; no clustering fit."""
from pathlib import Path
import hashlib, json, platform
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits


def recover_fixed_be(root):
    """Check the original cache fingerprint and reconstruct its exact row order."""
    source = root / 'course_artifacts/week04_experiment1_complete_overall_worry_v3'
    input_path = root / 'course_artifacts/week04_search_v4/stage1_eligible_development.parquet'
    design = json.loads((source / 'design.json').read_text())
    fingerprint = hashlib.sha256(input_path.read_bytes() + json.dumps(design, sort_keys=True).encode()).hexdigest()
    domains = ['food', 'water', 'crime', 'severe_weather', 'traffic', 'mental_health']
    worry = ['worry_' + d for d in domains]
    personal = ['harm_' + d + '_personal' for d in domains + ['work']]
    known = ['harm_' + d + '_known_person' for d in domains + ['work']]
    raw = pd.read_parquet(input_path)
    eligible = raw[worry].notna().sum(axis=1).ge(4) & raw[personal].notna().sum(axis=1).ge(4)
    data = raw.loc[eligible & raw[worry + personal + known].notna().all(axis=1)].reset_index(drop=True)
    assert data.country_split.eq('train').all() and data.WPID_RANDOM.is_unique
    assert len(data) == 109488 and data.COUNTRY_ISO3.nunique() == 113
    assert data[worry].isin([0, 1, 2]).all().all()
    assert data[personal + known].isin([0, 1]).all().all()
    assert np.isfinite(data.WGT).all() and data.WGT.gt(0).all()
    specs = {'B': ('012', 'balanced', 4), 'E': ('013', 'worry_emphasized', 8)}
    labels, cache_hashes = {}, {}
    for model, (encoding, balance, k) in specs.items():
        path = source / f'reference_{encoding}_{balance}_k{k}.joblib'
        saved = joblib.load(path)
        assert saved['fingerprint'] == fingerprint and len(saved['labels']) == len(data)
        labels[model] = np.asarray(saved['labels'], dtype=int)
        assert np.array_equal(np.unique(labels[model]), np.arange(k))
        cache_hashes[model] = hashlib.sha256(path.read_bytes()).hexdigest()
    return source, data, worry, personal, known, labels, fingerprint, cache_hashes


def equal_country_weights(data):
    return (data.WGT / data.groupby('COUNTRY_ISO3').WGT.transform('sum') / data.COUNTRY_ISO3.nunique()).to_numpy()




In [ ]:
# Objective: verify source models and replay saved depth-1/2/3/5 explanatory results.
from pathlib import Path
import json
import pandas as pd
from IPython.display import display,Markdown
SHALLOW_ROOT=Path.cwd()
if not (SHALLOW_ROOT/'course_artifacts').exists():SHALLOW_ROOT=SHALLOW_ROOT.parent
SHALLOW_OUTPUT=SHALLOW_ROOT/'course_artifacts/week04_experiment1_complete_overall_worry_v3/explanatory_fidelity/shallow_8ef4c1c5cfd3b7e6'
shallow_provenance=json.loads((SHALLOW_OUTPUT/'design_and_provenance.json').read_text())
_,_,_,_,_,_,shallow_fingerprint,shallow_hashes=recover_fixed_be(SHALLOW_ROOT)
assert shallow_fingerprint==shallow_provenance['source_fingerprint']
assert shallow_hashes==shallow_provenance['cache_sha256']
shallow_summary=pd.read_csv(SHALLOW_OUTPUT/'burden_depths_1_2_3_5_summary.csv')
shallow_folds=pd.read_csv(SHALLOW_OUTPUT/'burden_depths_1_2_3_5_folds.csv')
assert len(shallow_folds)==40 and shallow_summary.n_folds.eq(5).all()
display(Markdown('### All depth comparisons: five-fold means and descriptive SD'))
display(shallow_summary.round(4))
display(Markdown('### Primary depths 1–3'))
display(shallow_summary.query('depth <= 3')[['model','depth','balanced_accuracy_mean',
    'balanced_accuracy_sd','weighted_accuracy_mean','constant_guess_balanced_accuracy',
    'leaf_capacity_balanced_accuracy_ceiling']].round(4))
print('No clustering fits or held-out reads. Verified fixed B/E source hashes.')


### Conclusions from depths 1–3

B4 mean balanced accuracy is **48.4%, 58.8% and 58.2%** at depths 1, 2 and 3. E8 reaches **22.7%, 29.7% and 30.0%**. At depth 1, both scores approach their respective leaf-capacity bounds, so lower E8 balanced accuracy cannot by itself be read as richer domain structure. Increasing depth from 2 to 3 makes little difference; B4 decreases slightly, which is possible because extra splits need not improve country-fold generalization. These are fold means, not evidence of a statistically significant depth difference.

Depth 5 yields balanced accuracy **62.1% for B4** and **30.2% for E8**. Its weighted accuracies, **62.6% and 51.5%**, are close to the previously reported in-sample burden-lookup ceilings of 62.95% and 51.93%. Weighted accuracy and balanced accuracy answer different questions; neither should replace the other silently. The lookup uses the same respondents for construction and evaluation and is not independent validation.

The separate single-feature binary checks are depth-1 membership rules, not these multiclass burden trees. Some E8 worry responses have nearly 100% recall but low precision, so being characteristic is not equivalent to defining membership. Preserve this qualification in the manuscript and course discussion.

**Decision:** retain frozen B4/E8 without relabelling or refitting. Use these results to describe what limited burden-score explanations recover; do not claim external-outcome benefit, natural classes or unique model optimality. Full fold results, depth-1/2 rules and provenance are saved in the fingerprinted shallow-tree directory.


## 32 Direct country-response distances and component reporting

**Objective:** compare countries in the original worry and personal/known-person harm response distributions, avoiding the information loss from reducing each country to only four or eight profile shares. Keep the historical Hellinger analysis available: it answers a different question about the mixture of frozen profiles.

**Status and chronology:** this is a post-validation descriptive extension, added on 14 October 2026 after inspecting counterintuitive neighbours in the profile-mixture maps. It does not modify B4/E8, their assignments, the frozen split, regions, or held-out validation. It is not an untouched confirmation. The feature-block allocations are copied from B (0.50) and E (0.75), not optimized to return familiar neighbours. Income and geography are not inputs to the distance.

### Cohort and weighting

Use the exact complete-response cohort of 137,460 respondents (109,488 development; 27,972 held-out) across 142 countries. Validate counts against the original country-profile vectors and, where available, IDs against the protected frozen assignment registry. Within each country normalize the original WGT over included respondents. Missing responses are not imputed and feature-specific denominators are not substituted. These are complete-case survey-weighted summaries, not unrestricted national population estimates.

For country $c$, eligible respondent $i$, feature $j$ and category $r$,
\[
w_{ci}=\frac{\mathrm{WGT}_{ci}}{\sum_{i\in\mathcal I_c}\mathrm{WGT}_{ci}},\qquad
q_{cj}(r)=\sum_{i\in\mathcal I_c}w_{ci}\mathbf1\{X_{cij}=r\}.
\]
Six worry features have categories 0/1/2; seven personal and seven known-person harm indicators are binary. The country reference contains 46 category probabilities (18 worry, 14 personal-harm, 14 known-person-harm), not respondent records.

### Distances and components

\[
T_j(c,d)=\tfrac12\sum_r|q_{cj}(r)-q_{dj}(r)|,
\quad D_W=\tfrac16\sum_{j\in W}T_j,
\quad D_S=\tfrac17\sum_{j\in H_S}T_j,
\quad D_O=\tfrac17\sum_{j\in H_O}T_j.
\]
For binary harm, $T_j$ is the absolute difference in prevalence. Then
\[
D_\alpha=\alpha D_W+\tfrac{1-\alpha}{2}D_S+\tfrac{1-\alpha}{2}D_O.
\]
B-associated weights are 50/25/25; E-associated weights are 75/12.5/12.5. All distances are bounded by 0 and 1. No square roots are needed because total variation is linear. This preserves nominal block allocation, not K-means' centred Euclidean geometry or the middle-category spacing in 0/1/3 coding.

### What the numbers mean

Report the raw overall distance, worry-category mismatch ×100, personal/known-person mean absolute prevalence gaps in percentage points, a descriptive nearest-country rank and the overall distance percentile. The percentile is the upper empirical CDF over distinct unordered country pairs in the chosen scope: 10,011 pairs for all 142 countries; 6,328 for 113 development countries. Exact ties are included. It is not a p-value, confidence measure or probability that countries are equivalent. Country pairs are dependent because they share countries.

Only map colours are rescaled from zero (the reference country itself) to its maximum distance, in seven equal-width bands. Compute the overall distance from raw components **before** this rescaling. Reference-specific normalization is generally asymmetric and cannot replace a distance for country clustering or cross-country comparisons. Never normalize each component by its own maximum and then recombine: that would introduce country-dependent effective allocations.

### Main illustrative findings

Under direct-response B allocation, Canada's USA distance is 0.0635 (rank 1, pair percentile 1.83), compared with Iran at 0.0837 (rank 19, percentile 8.65) and Cambodia at 0.1665 (rank 113, percentile 65.63). In the historical B4 mixture comparison Iran was fifth and Cambodia second. The direct comparison exposes differences hidden by coarse profile shares; it is not validated merely because Canada becomes nearest.

Finland's nearest country is Iceland under both allocations. Finland–Sweden has personal-harm distance 0.00865 but worry distance 0.11602. Its overall distance rises from 0.06848 under B allocation to 0.09225 under E allocation. The components explain this change without claiming that one neighbour list is the natural or correct ordering.

The direct-response income-gap rank correlations are 0.3185 (B allocation) and 0.2618 (E allocation), with development-only values 0.3192 and 0.2600. These are not the historical Hellinger correlations (0.211 and 0.229), and the earlier Hellinger permutation references do not apply. The new income summaries are descriptive only.

### Limitations and decision

This is a metric on the collection of marginal category distributions, but only a pseudometric on the full joint response law: equal marginals can conceal different combinations within people. No new joint-distribution analysis or resampling uncertainty has been computed for these distances. Exact ranks, especially when scores are close, remain descriptive. Survey-design limitations, translation, reporting, known-person networks and complete-case selection still apply.

Adopt direct original-response comparison as the main country-distance display, while retaining explicitly labelled profile-mixture Hellinger views and all historical results. Do not reinterpret country similarity as societal equivalence or change the frozen respondent profile models.

The following implementation reconstructs the aggregates without fitting any model; the reporting cell verifies saved results by hash. Public notebook outputs are cleared separately, but only aggregate country results appear in this new section.


In [ ]:
"""Post-validation country-response distances; never fit or modify respondent models."""
from pathlib import Path
import argparse
import base64
import hashlib
import json
import platform
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

DOMAINS = ['food', 'water', 'crime', 'severe_weather', 'traffic', 'mental_health']
WORRY_RAW = dict(zip(DOMAINS, ['WP20720','WP20721','WP20722','WP20723','WP22213','WP20726']))
HARM_RAW = dict(zip(DOMAINS + ['work'], ['WP22442','WP22443','WP22444','WP22445','WP22446','WP22447','WP22448']))
WORRY = ['worry_' + d for d in DOMAINS]
SELF = ['harm_' + d + '_personal' for d in HARM_RAW]
KNOWN = ['harm_' + d + '_known_person' for d in HARM_RAW]
FEATURES = WORRY + SELF + KNOWN
ALPHAS = {'B': .5, 'E': .75}
SCOPES = ['development', 'all_countries_post_validation']
DESIGN = {'version': 1, 'features': FEATURES, 'category_counts': [3]*6+[2]*14,
          'eligibility': 'same 20-feature complete-response cohort as frozen B4/E8',
          'weights': 'original WGT normalized within each eligible country',
          'distance': 'mean feature total variation; six worry, seven personal harm, seven known-person harm',
          'alphas': ALPHAS, 'harm_allocation': 'half to personal and half to known-person harm',
          'percentile': '100*upper empirical CDF over distinct unordered pairs in displayed scope; exact ties included',
          'rank_ties': 'distance then ISO3; ranks descriptive, not statistical certainty',
          'display': 'seven equal-width bands from reference zero to its maximum; normalize only after combining raw components',
          'limits': 'marginal summaries, not joint distributions; no new resampling uncertainty or causal inference',
          'heldout_use': 'post-validation descriptive extension; not untouched validation or model selection',
          'respondent_models_refitted': False}

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def response_features(frame):
    result = frame.copy()
    for domain, source in WORRY_RAW.items():
        assert result[source].dropna().isin([1,2,3,98,99]).all(), source
        result['worry_'+domain] = result[source].map({1:2.,2:1.,3:0.})
    for domain, source in HARM_RAW.items():
        raw = result[source]
        assert raw.dropna().isin([1,2,3,4,98,99]).all(), source
        valid = raw.isin([1,2,3,4])
        result['harm_'+domain+'_personal'] = np.where(valid,raw.isin([1,3]).astype(float),np.nan)
        result['harm_'+domain+'_known_person'] = np.where(valid,raw.isin([2,3]).astype(float),np.nan)
    return result

def response_matrices(probabilities):
    blocks, position = [], 0
    for count in DESIGN['category_counts']:
        q = probabilities[:,position:position+count]
        assert np.allclose(q.sum(axis=1),1) and np.isfinite(q).all() and (q >= 0).all()
        blocks.append(np.abs(q[:,None,:]-q[None,:,:]).sum(axis=2)/2)
        position += count
    assert position == 46
    matrices = {'worry': np.mean(blocks[:6],axis=0), 'self': np.mean(blocks[6:13],axis=0),
                'known': np.mean(blocks[13:],axis=0)}
    for model, alpha in ALPHAS.items():
        matrices[model] = alpha*matrices['worry']+(1-alpha)*(matrices['self']+matrices['known'])/2
    for matrix in matrices.values():
        assert np.allclose(matrix,matrix.T) and np.allclose(np.diag(matrix),0)
        assert np.isfinite(matrix).all() and matrix.min() >= 0 and matrix.max() <= 1+1e-12
    return matrices

def pair_percentiles(matrix):
    pool = np.sort(matrix[np.triu_indices(len(matrix),1)])
    result = 100*np.searchsorted(pool,matrix,side='right')/len(pool)
    np.fill_diagonal(result,np.nan)
    return result

def income_tables(matrix, metadata):
    classified = metadata.CountryIncomeLevel.between(1,4)
    md = metadata.loc[classified]
    d = matrix[np.ix_(np.flatnonzero(classified),np.flatnonzero(classified))]
    income = md.CountryIncomeLevel.to_numpy()
    region = md.GlobalRegion.to_numpy()
    ia,ib = np.triu_indices(len(md),1)
    values,gap = d[ia,ib],np.abs(income[ia]-income[ib])
    same_region = region[ia] == region[ib]
    cells = []
    for a in range(1,5):
        row = []
        left = np.flatnonzero(income == a)
        for b in range(1,5):
            right = np.flatnonzero(income == b)
            sub = d[np.ix_(left,right)]
            values_ab = sub[np.triu_indices(len(left),1)] if a == b else sub.ravel()
            row.append({'mean_distance':float(values_ab.mean()),'distinct_pairs':len(values_ab)})
        cells.append(row)
    summary = {'classified_countries':len(md),'distinct_pairs':len(values),
               'ordinal_gap_spearman':float(spearmanr(values,gap).statistic),
               'within_region_spearman':float(spearmanr(values[same_region],gap[same_region]).statistic),
               'interpretation':'descriptive association; no pair-independent significance test or new permutation reference'}
    return summary,cells

def run_response_distances(root):
    root = Path(root).resolve()
    source = root/'course_artifacts/week05_country_similarity_and_profile_synthesis_v1/d694096841b06e54'
    inputs = [root/'course_artifacts/week03_development_data.parquet',
              root/'course_artifacts/week03_heldout_country_data.parquet',source/'country_profile_vectors.csv',
              root/'course_artifacts/week05_frozen_objective1_v1/123d248adda6115f/freeze_seal.json']
    provenance = {'design':DESIGN,'source_hashes':{str(p.relative_to(root)):file_hash(p) for p in inputs},
                  'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}
    fingerprint = hashlib.sha256(json.dumps(provenance,sort_keys=True).encode()).hexdigest()
    output = root/'course_artifacts/week05_country_response_distances_v1'/fingerprint[:16]
    output.mkdir(parents=True,exist_ok=True)
    vectors = pd.read_csv(source/'country_profile_vectors.csv')
    metadata = vectors.drop_duplicates('COUNTRY_ISO3').set_index('COUNTRY_ISO3').sort_index()
    columns = ['WPID_RANDOM','COUNTRY_ISO3','country_split','WGT']+list(WORRY_RAW.values())+list(HARM_RAW.values())
    raw = response_features(pd.concat([pd.read_parquet(p,columns=columns) for p in inputs[:2]],ignore_index=True))
    frame = raw.loc[raw[FEATURES].notna().all(axis=1)].copy()
    assert len(frame) == 137460 and frame.WPID_RANDOM.is_unique and frame.COUNTRY_ISO3.nunique() == 142
    assert frame.WGT.gt(0).all() and np.isfinite(frame.WGT).all()
    assert frame.groupby('COUNTRY_ISO3').size().equals(metadata.complete_respondents.rename(None).astype('int64'))
    handoff = root/'course_artifacts/week05_objective1_handoff_v1/123d248adda6115f/profile_assigned_respondents.parquet'
    if handoff.exists():
        ids = pd.read_parquet(handoff,columns=['WPID_RANDOM'])
        assert set(ids.WPID_RANDOM) == set(frame.WPID_RANDOM)
        provenance['eligibility_id_registry_verified'] = True
    frame['mass'] = frame.WGT/frame.groupby('COUNTRY_ISO3').WGT.transform('sum')
    assert np.allclose(frame.groupby('COUNTRY_ISO3').mass.sum(),1)
    arrays, distribution_rows = [], []
    for j,feature in enumerate(FEATURES):
        categories = range(DESIGN['category_counts'][j])
        q = frame.groupby(['COUNTRY_ISO3',feature]).mass.sum().unstack(fill_value=0).reindex(index=metadata.index,columns=categories,fill_value=0)
        arrays.append(q.to_numpy())
        for iso in metadata.index:
            for category in categories:
                distribution_rows.append({'COUNTRY_ISO3':iso,'feature':feature,'category':category,
                                          'proportion':float(q.loc[iso,category])})
    probabilities = np.concatenate(arrays,axis=1)
    matrices = response_matrices(probabilities)
    pd.DataFrame(distribution_rows).to_csv(output/'country_response_probabilities.csv',index=False)
    metadata.reset_index().to_csv(output/'country_metadata.csv',index=False)
    payload = {'version':1,'fingerprint':fingerprint,'country_order':metadata.index.tolist(),
               'category_counts':DESIGN['category_counts'],'shape':list(probabilities.shape),'byte_order':'little',
               'probabilities_f64_base64':base64.b64encode(probabilities.astype('<f8').tobytes()).decode(),
               'alphas':ALPHAS,'income':{'summaries':{'B':{},'E':{}},'matrices':{'B':{},'E':{}}}}
    pairs,neighbours,summary,examples,income_summary = [],[],[],[],[]
    for scope in SCOPES:
        selected = np.flatnonzero(metadata.country_split.eq('train')) if scope == 'development' else np.arange(len(metadata))
        md = metadata.iloc[selected]
        codes = md.index.tolist()
        selected_matrices = {key:value[np.ix_(selected,selected)] for key,value in matrices.items()}
        percentiles = {key:pair_percentiles(value) for key,value in selected_matrices.items()}
        ia,ib = np.triu_indices(len(md),1)
        for i,j in zip(ia,ib):
            row = {'scope':scope,'iso3_a':codes[i],'iso3_b':codes[j]}
            for key in selected_matrices:
                row[key] = float(selected_matrices[key][i,j]);row[key+'_percentile'] = float(percentiles[key][i,j])
            pairs.append(row)
        for metric,d in selected_matrices.items():
            pd.DataFrame(d,index=codes,columns=codes).to_csv(output/f'{metric}_{scope}_matrix.csv')
            pd.DataFrame(percentiles[metric],index=codes,columns=codes).to_csv(output/f'{metric}_{scope}_percentiles.csv')
            pool = d[ia,ib]
            summary.append({'metric':metric,'scope':scope,'countries':len(md),'pairs':len(pool),
                            'min':float(pool.min()),'median':float(np.median(pool)),
                            'p90':float(np.quantile(pool,.9)),'max':float(pool.max())})
            for i,iso in enumerate(codes):
                order = sorted((j for j in range(len(md)) if j != i),key=lambda j:(d[i,j],codes[j]))
                for rank,j in enumerate(order[:20],1):
                    row = {'scope':scope,'metric':metric,'COUNTRY_ISO3':iso,'Country':md.iloc[i].Country,
                           'rank':rank,'peer_iso3':codes[j],'peer_country':md.iloc[j].Country,
                           'distance':float(d[i,j]),'distance_percentile':float(percentiles[metric][i,j])}
                    for key in selected_matrices:
                        row[key] = float(selected_matrices[key][i,j]);row[key+'_percentile'] = float(percentiles[key][i,j])
                    neighbours.append(row)
                if scope == 'all_countries_post_validation' and iso in ['USA','FIN'] and metric in ['B','E']:
                    targets = ['CAN','IRN','KHM','GBR'] if iso == 'USA' else ['ISL','SWE','NOR','EST']
                    for peer in targets:
                        j = codes.index(peer)
                        row = {'reference':iso,'peer':peer,'metric':metric,'rank':order.index(j)+1,
                               'distance':float(d[i,j]),'percentile':float(percentiles[metric][i,j]),
                               'maximum_from_reference':float(d[i].max()),
                               'relative_to_max':float(d[i,j]/d[i].max())}
                        row.update({key:float(selected_matrices[key][i,j]) for key in ['worry','self','known']})
                        examples.append(row)
        for model in ['B','E']:
            stats,cells = income_tables(selected_matrices[model],md)
            payload['income']['summaries'][model][scope] = stats
            payload['income']['matrices'][model][scope] = cells
            income_summary.append({'model':model,'scope':scope,**stats})
    tables = {'country_pair_response_distances':pairs,'nearest_country_responses':neighbours,
              'response_distance_summary':summary,'illustrative_country_comparisons':examples,
              'response_income_summary':income_summary}
    for filename,rows in tables.items():
        pd.DataFrame(rows).to_csv(output/(filename+'.csv'),index=False)
    (output/'response_map_payload.json').write_text(json.dumps(payload,separators=(',',':'),allow_nan=False))
    provenance.update({'fingerprint':fingerprint,'eligible_respondents':len(frame),'country_count':len(metadata),
                       'implementation_sha256':file_hash(root/'.week05_build/country_response_distances.py'),
                       'files':{p.name:file_hash(p) for p in output.iterdir() if p.name != 'analysis_manifest.json'}})
    (output/'analysis_manifest.json').write_text(json.dumps(provenance,indent=2,sort_keys=True))
    assert file_hash(inputs[3]) == provenance['source_hashes'][str(inputs[3].relative_to(root))]
    return output



In [ ]:
# Objective: verify and display the saved direct-response extension; no profile fitting.
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display
RESPONSE_ROOT=Path.cwd().resolve()
if not (RESPONSE_ROOT/'course_artifacts').is_dir():
    RESPONSE_ROOT=next(p for p in RESPONSE_ROOT.parents if (p/'course_artifacts').is_dir())
RUN_COUNTRY_RESPONSE_RECOMPUTE=False  # True recomputes country summaries only; never fits respondent models.
RESPONSE_OUTPUT=RESPONSE_ROOT/'course_artifacts/week05_country_response_distances_v1/e1ecf01551bcf2da'
if RUN_COUNTRY_RESPONSE_RECOMPUTE:
    assert run_response_distances(RESPONSE_ROOT)==RESPONSE_OUTPUT
response_manifest=json.loads((RESPONSE_OUTPUT/'analysis_manifest.json').read_text())
assert response_manifest['design']['respondent_models_refitted'] is False
for filename,digest in response_manifest['files'].items():
    assert hashlib.sha256((RESPONSE_OUTPUT/filename).read_bytes()).hexdigest()==digest,filename
for relative,digest in response_manifest['source_hashes'].items():
    source=RESPONSE_ROOT/relative
    if source.exists():
        assert hashlib.sha256(source.read_bytes()).hexdigest()==digest,relative
display(pd.read_csv(RESPONSE_OUTPUT/'response_distance_summary.csv'))
response_neighbours=pd.read_csv(RESPONSE_OUTPUT/'nearest_country_responses.csv')
display(response_neighbours.query("scope=='all_countries_post_validation' and COUNTRY_ISO3 in ['USA','FIN'] and metric in ['B','E'] and rank<=10")
    [['metric','Country','rank','peer_country','distance','distance_percentile','worry','self','known']])
display(pd.read_csv(RESPONSE_OUTPUT/'illustrative_country_comparisons.csv'))
display(pd.read_csv(RESPONSE_OUTPUT/'response_income_summary.csv'))
print('Verified saved direct-response distances. No clustering refitted; historical profile-mixture results retained.')
